In this notebook I will create a training table equivalent to ml_model_run_details

# Define Library

In [1]:
# %% [markdown]
# # Jupyter Notebook Loading Header
#
# This is a custom loading header for Jupyter Notebooks in Visual Studio Code.
# It includes common imports and settings to get you started quickly.
# %% [markdown]
## Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from google.cloud import storage
import os
import tempfile
import time
from datetime import datetime
import uuid
import joblib
import uuid

import gcsfs
import duckdb as dd
import pickle
import joblib
from typing import Union
import io
path = r'C:\Users\Dwaipayan\AppData\Roaming\gcloud\legacy_credentials\dchakroborti@tonikbank.com\adc.json'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = path
client = bigquery.Client(project='prj-prod-dataplatform')
os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"

# %% [markdown]
## Configure Settings
# Set options or configurations as needed
pd.set_option('display.max_columns', None)
pd.set_option("Display.max_rows", 100)

### Function

#### expand_calc_features

In [2]:
import pandas as pd
import json

def expand_calc_features(df):
    """
    Expand the calcFeatures JSON column into separate columns and return the complete DataFrame.

    Parameters:
    df (pd.DataFrame): Input DataFrame with calcFeatures column containing JSON data

    Returns:
    pd.DataFrame: Expanded DataFrame with all original columns plus JSON features as separate columns
    """

    # Make a copy to avoid modifying the original DataFrame
    df_expanded = df.copy()

    # Parse the calcFeatures JSON column
    calc_features_list = []

    for idx, calc_features_str in enumerate(df['calcFeatures']):
        try:
            # Parse the JSON string
            features_dict = json.loads(calc_features_str.replace("'", '"'))  # Replace single quotes with double quotes for valid JSON
            calc_features_list.append(features_dict)
        except (json.JSONDecodeError, AttributeError) as e:
            # If parsing fails, create an empty dict and print warning
            print(f"Warning: Could not parse calcFeatures at index {idx}: {e}")
            calc_features_list.append({})

    # Create DataFrame from the parsed JSON data
    calc_features_df = pd.DataFrame(calc_features_list)

    # Add prefix to JSON-derived columns to avoid conflicts
    calc_features_df = calc_features_df.add_prefix('calc_')

    # Reset index to ensure proper alignment
    df_expanded = df_expanded.reset_index(drop=True)
    calc_features_df = calc_features_df.reset_index(drop=True)

    # Combine original DataFrame with expanded calcFeatures
    result_df = pd.concat([df_expanded, calc_features_df], axis=1)

    return result_df


#### expand_calc_features_robust

In [3]:
import pandas as pd
import json

def expand_calc_features_robust(df):
    """
    Expand the calcFeatures JSON column into separate columns with better error handling.

    Parameters:
    df (pd.DataFrame): Input DataFrame with calcFeatures column containing JSON data

    Returns:
    pd.DataFrame: Expanded DataFrame with all original columns plus JSON features as separate columns
    """

    # Make a copy to avoid modifying the original DataFrame
    df_expanded = df.copy()

    # Parse the calcFeatures JSON column
    calc_features_data = []

    for idx, row in df.iterrows():
        calc_features_str = row['calcFeatures']

        if pd.isna(calc_features_str) or calc_features_str == '':
            calc_features_data.append({})
            continue

        try:
            # Clean the string and parse JSON
            cleaned_str = calc_features_str.replace("'", '"').replace('None', 'null').replace('True', 'true').replace('False', 'false')
            features_dict = json.loads(cleaned_str)
            calc_features_data.append(features_dict)
        except Exception as e:
            print(f"Warning: Could not parse calcFeatures at index {idx}: {e}")
            print(f"Problematic string: {calc_features_str[:100]}...")  # Print first 100 chars
            calc_features_data.append({})

    # Create DataFrame from the parsed JSON data
    calc_features_df = pd.DataFrame(calc_features_data)

    # Add prefix to JSON-derived columns to avoid conflicts with existing columns
    calc_features_df = calc_features_df.add_prefix('feat_')

    # Combine DataFrames
    result_df = pd.concat([df_expanded, calc_features_df], axis=1)

    print(f"Original DataFrame shape: {df.shape}")
    print(f"Expanded DataFrame shape: {result_df.shape}")
    print(f"Added {len(calc_features_df.columns)} new columns from calcFeatures")

    return result_df

# Table Name

In [4]:
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"

## Transform data v1

In [5]:
import pandas as pd
import json
import uuid
from datetime import datetime
from typing import List

def transform_data(
    d1: pd.DataFrame, 
    feature_column: List[str], 
    a: str = 'demo_score', 
    modelDisplayName: str = 'Cash_beta_trench1_Demo_backscore', 
    tc: str = "", 
    subscription_name: str = 'sil_march 25 models'
) -> pd.DataFrame:
    """
    Transforms input data into a structured format suitable for model scoring output.

    Parameters:
    - d1 (pd.DataFrame): Input DataFrame containing raw data.
    - feature_column (List[str]): List of column names to include in the 'calcFeature' JSON.
    - a (str): Column name containing the prediction score. Default is 'demo_score'.
    - modelDisplayName (str): Name of the model used for scoring.
    - tc (str): Trench category (optional).
    - do (str): Device operating system. Default is 'android'.
    - subscription_name (str): Name of the subscription or model group.

    Returns:
    - pd.DataFrame: Transformed DataFrame with structured output.
    """

    # Make a copy of the input DataFrame to avoid modifying the original
    df = d1.copy()
    
    # Initialize an empty list to store transformed rows
    output_data = []
    
    # Iterate over each row in the DataFrame
    for _, row in df.iterrows():
        # Initialize dictionary to hold feature values
        calc_feature = {}
        
        # Loop through each feature column and extract its value from the row
        for col in feature_column:
            if col in row and pd.notna(row[col]):
                # Convert datetime values to ISO format strings
                if isinstance(row[col], pd.Timestamp):
                    calc_feature[col] = row[col].isoformat()
                else:
                    calc_feature[col] = row[col]
        
        # Get the current timestamp for start_time, end_time, and publish_time
        current_time = datetime.now().isoformat()
        
        # Construct the output row dictionary with required fields
        output_row = {
            "customerId": row['customer_id'],  # Unique customer identifier
            "digitalLoanAccountId": row['digitalLoanAccountId'],  # Loan account ID
            "crifApplicationId": str(uuid.uuid4()),  # Random UUID for application ID
            "prediction": row.get(a, 0),  # Prediction score from specified column
            "start_time": current_time,  # Timestamp when processing starts
            "end_time": current_time,    # Timestamp when processing ends
            "modelDisplayName": modelDisplayName,  # Name of the model used
            "modelVersionId": "v1",  # Static model version
            "calcFeature": json.dumps(calc_feature, default=str),  # Features as JSON string
            "subscription_name": subscription_name,  # Subscription name
            "message_id": str(uuid.uuid4()),  # Random UUID for message ID
            "publish_time": current_time,  # Timestamp when message is published
            "attributes": "{}",  # Placeholder for additional attributes
            "trenchCategory": tc,  # Optional trench category
            "deviceOs": row['osType'],
            "Data_selection": row['Data_selection'],  # Data selection
            "Application_date": row['application_date'],
        }
        
        # Append the transformed row to the output list
        output_data.append(output_row)
    
    # Convert the list of dictionaries to a DataFrame
    output_df = pd.DataFrame(output_data)
    
    # Return the transformed DataFrame
    return output_df


# transform_datav2

In [6]:
import pandas as pd
import json
import uuid
from datetime import datetime
from typing import List

def transform_datav2(
    d1: pd.DataFrame, 
    feature_column: List[str], 
    a: str = 'demo_score', 
    modelDisplayName: str = 'Cash_beta_trench1_Demo_backscore', 
    tc: str = "", 
    subscription_name: str = 'sil_march 25 models'
) -> pd.DataFrame:
    """
    Transforms input data into a structured format suitable for model scoring output.

    Parameters:
    - d1 (pd.DataFrame): Input DataFrame containing raw data.
    - feature_column (List[str]): List of column names to include in the 'calcFeature' JSON.
    - a (str): Column name containing the prediction score. Default is 'demo_score'.
    - modelDisplayName (str): Name of the model used for scoring.
    - tc (str): Trench category (optional).
    - do (str): Device operating system. Default is 'android'.
    - subscription_name (str): Name of the subscription or model group.

    Returns:
    - pd.DataFrame: Transformed DataFrame with structured output.
    """

    # Make a copy of the input DataFrame to avoid modifying the original
    df = d1.copy()
    
    # Initialize an empty list to store transformed rows
    output_data = []
    
    # Iterate over each row in the DataFrame
    for _, row in df.iterrows():
        # Initialize dictionary to hold feature values
        calc_feature = {}
        
        # Loop through each feature column and extract its value from the row
        for col in feature_column:
            if col in row and pd.notna(row[col]):
                # Convert datetime values to ISO format strings
                if isinstance(row[col], pd.Timestamp):
                    calc_feature[col] = row[col].isoformat()
                else:
                    calc_feature[col] = row[col]
        
        # Get the current timestamp for start_time, end_time, and publish_time
        current_time = datetime.now().isoformat()
        
        # Construct the output row dictionary with required fields
        output_row = {
            "customerId": row['customer_id'],  # Unique customer identifier
            "digitalLoanAccountId": row['digitalLoanAccountId'],  # Loan account ID
            "crifApplicationId": str(uuid.uuid4()),  # Random UUID for application ID
            "prediction": row.get(a, 0),  # Prediction score from specified column
            "start_time": current_time,  # Timestamp when processing starts
            "end_time": current_time,    # Timestamp when processing ends
            "modelDisplayName": modelDisplayName,  # Name of the model used
            "modelVersionId": "v2",  # Static model version
            "calcFeature": json.dumps(calc_feature, default=str),  # Features as JSON string
            "subscription_name": subscription_name,  # Subscription name
            "message_id": str(uuid.uuid4()),  # Random UUID for message ID
            "publish_time": current_time,  # Timestamp when message is published
            "attributes": "{}",  # Placeholder for additional attributes
            "trenchCategory": tc,  # Optional trench category
            "deviceOs": row['osType'],  # Device operating system
            "Data_selection": row['Data_selection'],  # Data selection
            "Application_date": row['application_date'],
        }
        
        # Append the transformed row to the output list
        output_data.append(output_row)
    
    # Convert the list of dictionaries to a DataFrame
    output_df = pd.DataFrame(output_data)
    
    # Return the transformed DataFrame
    return output_df


## Transform data v1.1

In [7]:
import pandas as pd
import json
import uuid
from datetime import datetime
from typing import List

def transform_data_v1_1(
    d1: pd.DataFrame, 
    feature_column: List[str], 
    a: str = 'demo_score', 
    modelDisplayName: str = 'Cash_beta_trench1_Demo_backscore', 
    tc: str = "", 
    subscription_name: str = 'sil_march 25 models'
) -> pd.DataFrame:
    """
    Transforms input data into a structured format suitable for model scoring output.

    Parameters:
    - d1 (pd.DataFrame): Input DataFrame containing raw data.
    - feature_column (List[str]): List of column names to include in the 'calcFeature' JSON.
    - a (str): Column name containing the prediction score. Default is 'demo_score'.
    - modelDisplayName (str): Name of the model used for scoring.
    - tc (str): Trench category (optional).
    - do (str): Device operating system. Default is 'android'.
    - subscription_name (str): Name of the subscription or model group.

    Returns:
    - pd.DataFrame: Transformed DataFrame with structured output.
    """

    # Make a copy of the input DataFrame to avoid modifying the original
    df = d1.copy()
    
    # Initialize an empty list to store transformed rows
    output_data = []
    
    # Iterate over each row in the DataFrame
    for _, row in df.iterrows():
        # Initialize dictionary to hold feature values
        calc_feature = {}
        
        # Loop through each feature column and extract its value from the row
        for col in feature_column:
            if col in row and pd.notna(row[col]):
                # Convert datetime values to ISO format strings
                if isinstance(row[col], pd.Timestamp):
                    calc_feature[col] = row[col].isoformat()
                else:
                    calc_feature[col] = row[col]
        
        # Get the current timestamp for start_time, end_time, and publish_time
        current_time = datetime.now().isoformat()
        
        # Construct the output row dictionary with required fields
        output_row = {
            "customerId": row['customer_id'],  # Unique customer identifier
            "digitalLoanAccountId": row['digitalLoanAccountId'],  # Loan account ID
            "crifApplicationId": str(uuid.uuid4()),  # Random UUID for application ID
            "prediction": row.get(a, 0),  # Prediction score from specified column
            "start_time": current_time,  # Timestamp when processing starts
            "end_time": current_time,    # Timestamp when processing ends
            "modelDisplayName": modelDisplayName,  # Name of the model used
            "modelVersionId": "v1.1",  # Static model version
            "calcFeature": json.dumps(calc_feature, default=str),  # Features as JSON string
            "subscription_name": subscription_name,  # Subscription name
            "message_id": str(uuid.uuid4()),  # Random UUID for message ID
            "publish_time": current_time,  # Timestamp when message is published
            "attributes": "{}",  # Placeholder for additional attributes
            "trenchCategory": tc,  # Optional trench category
            "deviceOs": row['osType'],
            "Data_selection": row['Data_selection'],  # Data selection
            "Application_date": row['application_date'],
        }
        
        # Append the transformed row to the output list
        output_data.append(output_row)
    
    # Convert the list of dictionaries to a DataFrame
    output_df = pd.DataFrame(output_data)
    
    # Return the transformed DataFrame
    return output_df


#### PSI Functions new

In [8]:
## Updated on 27-10-2025 - Modified for Training Period Baseline
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

def identify_feature_types(df: pd.DataFrame, feature_list: List[str]) -> Dict[str, List[str]]:
    """
    Identify categorical and numerical features from the feature list.

    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe
    feature_list : List[str]
        List of features to classify

    Returns:
    --------
    Dict with 'categorical' and 'numerical' keys containing respective feature lists
    """
    categorical_features = []
    numerical_features = []

    for feature in feature_list:
        if feature not in df.columns:
            print(f"Warning: Feature '{feature}' not found in dataframe")
            continue

        # Check if feature is numeric
        if pd.api.types.is_numeric_dtype(df[feature]):
            # If unique values are less than 15 and all integers, treat as categorical
            unique_vals = df[feature].nunique()
            if unique_vals < 15 and df[feature].dropna().apply(lambda x: x == int(x) if isinstance(x, (int, float)) else False).all():
                categorical_features.append(feature)
            else:
                numerical_features.append(feature)
        else:
            categorical_features.append(feature)

    return {
        'categorical': categorical_features,
        'numerical': numerical_features
    }


def create_bins_for_features(df: pd.DataFrame,
                             numerical_features: List[str],
                             categorical_features: List[str],
                             train_period_df: pd.DataFrame) -> Dict:
    """
    Create bins for numerical features (deciles with fallback) and categorical features (top 6 + others)
    based on the entire training period data.

    Parameters:
    -----------
    df : pd.DataFrame
        Full input dataframe
    numerical_features : List[str]
        List of numerical features
    categorical_features : List[str]
        List of categorical features
    train_period_df : pd.DataFrame
        Training period dataframe (June 2024 to March 2025)

    Returns:
    --------
    Dictionary containing binning information for each feature
    """
    binning_info = {}

    # Create bins for numerical features with fallback strategy
    for feature in numerical_features:
        valid_data = train_period_df[feature].dropna()

        if len(valid_data) == 0:
            binning_info[feature] = {'type': 'numerical', 'bins': None, 'bin_ranges': {}}
            continue

        bins = None
        bin_count = None

        # Try 10 bins (deciles)
        try:
            test_bins = np.percentile(valid_data, np.arange(0, 101, 10))
            test_bins = np.unique(test_bins)
            if len(test_bins) >= 11:  # 11 edges = 10 bins
                bins = test_bins
                bin_count = 10
        except Exception as e:
            pass

        # If 10 bins not possible, try 5 bins
        if bins is None:
            try:
                test_bins = np.percentile(valid_data, np.arange(0, 101, 20))
                test_bins = np.unique(test_bins)
                if len(test_bins) >= 6:  # 6 edges = 5 bins
                    bins = test_bins
                    bin_count = 5
            except Exception as e:
                pass

        # If 5 bins not possible, try 3 bins
        if bins is None:
            try:
                test_bins = np.percentile(valid_data, [0, 33.33, 66.67, 100])
                test_bins = np.unique(test_bins)
                if len(test_bins) >= 4:  # 4 edges = 3 bins
                    bins = test_bins
                    bin_count = 3
            except Exception as e:
                pass

        # If still no bins possible, use equal distance bins of 5
        if bins is None:
            print(f"Warning: Feature '{feature}' has insufficient variance - cannot create standard bins")
            print(f"Feature '{feature}': Using equal distance bins of 5")

            min_val = valid_data.min()
            max_val = valid_data.max()

            # Create 5 equal distance bins
            bins = np.linspace(min_val, max_val, 6)  # 6 edges = 5 bins
            bins = np.unique(bins)
            bin_count = len(bins) - 1

            # If all values are the same, add slight buffer
            if bin_count == 1:
                bins = np.array([min_val - 0.1, min_val, min_val + 0.1])
                bin_count = 2
                print(f"Feature '{feature}': Constant value ({min_val}). Created 2 equal distance bins with buffer")

        # Add infinity edges to capture all values
        bins = bins.copy()
        bins[0] = -np.inf
        bins[-1] = np.inf

        print(f"Feature '{feature}': Created {bin_count} bins")

        # Create bin ranges dictionary
        bin_ranges = {}
        for i in range(len(bins)-1):
            bin_name = f"Bin_{i+1}"
            bin_ranges[bin_name] = {
                'min': bins[i],
                'max': bins[i+1],
                'range_str': f"[{bins[i]:.2f}, {bins[i+1]:.2f}]" if not np.isinf(bins[i]) and not np.isinf(bins[i+1]) else f"({bins[i]}, {bins[i+1]})"
            }

        binning_info[feature] = {
            'type': 'numerical',
            'bins': bins,
            'bin_ranges': bin_ranges,
            'bin_count': bin_count
        }

    # Create bins for categorical features (top 6 + others) using training period
    for feature in categorical_features:
        value_counts = train_period_df[feature].value_counts()
        unique_categories = value_counts.index.tolist()
        print(f"Unique categories: {unique_categories}")

        if len(unique_categories) <= 6:
            # Treat each category as a separate bin
            top_categories = unique_categories
        else:
            # Use top 6 categories only
            top_categories = value_counts.nlargest(6).index.tolist()

        print(f"Top categories for feature '{feature}': {top_categories}")

        binning_info[feature] = {
                'type': 'categorical',
                'top_categories': top_categories,
                'bin_ranges': {}  # No ranges for categorical
            }

    return binning_info


def apply_binning(df: pd.DataFrame,
                  feature: str,
                  binning_info: Dict) -> pd.Series:
    """
    Apply binning to a feature based on binning information.

    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe
    feature : str
        Feature name
    binning_info : Dict
        Binning information for the feature

    Returns:
    --------
    pd.Series with binned values
    """
    if binning_info['type'] == 'numerical':
        if binning_info['bins'] is None:
            return pd.Series(['Missing'] * len(df), index=df.index)

        bins = binning_info['bins']
        labels = [f"Bin_{i+1}" for i in range(len(bins)-1)]

        binned = pd.cut(df[feature],
                       bins=bins,
                       labels=labels,
                       include_lowest=True,
                       duplicates='drop')

        # Handle nulls - convert to string and then replace
        binned = binned.astype(str)
        binned[df[feature].isna()] = 'Missing'

        return binned

    else:  # categorical
        top_cats = binning_info['top_categories']

        # Convert to string for consistent comparison
        if pd.api.types.is_categorical_dtype(df[feature]):
            feature_data = df[feature].astype(str)
        else:
            feature_data = df[feature].astype(str)

        # Replace NaN string representation with 'Missing'
        feature_data = feature_data.replace('nan', 'Missing')

        # Convert top_cats to strings for comparison
        top_cats_str = [str(cat) for cat in top_cats]

        # Apply binning logic: use category name if in top_cats, else 'Others' (except for Missing)
        binned = feature_data.apply(lambda x: x if x in top_cats_str else ('Others' if x != 'Missing' else 'Missing'))

        return binned


def calculate_psi(expected_pct: pd.Series,
                  actual_pct: pd.Series,
                  epsilon: float = 0.0001) -> float:
    """
    Calculate Population Stability Index with proper epsilon handling and renormalization.

    Parameters:
    -----------
    expected_pct : pd.Series
        Expected (baseline) percentages
    actual_pct : pd.Series
        Actual percentages
    epsilon : float
        Small value to avoid log(0)

    Returns:
    --------
    PSI value
    """
    # Align indices
    all_bins = expected_pct.index.union(actual_pct.index)
    expected_pct = expected_pct.reindex(all_bins, fill_value=0)
    actual_pct = actual_pct.reindex(all_bins, fill_value=0)

    # Only add epsilon where values are zero
    expected_pct = expected_pct.apply(lambda x: epsilon if x == 0 else x)
    actual_pct = actual_pct.apply(lambda x: epsilon if x == 0 else x)

    # Renormalize to ensure they sum to 1 after adding epsilon
    expected_pct = expected_pct / expected_pct.sum()
    actual_pct = actual_pct / actual_pct.sum()

    # Calculate PSI
    psi_value = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))

    return psi_value


def calculate_month_on_month_psi(df: pd.DataFrame,
                                 feature_list: List[str],
                                 segment_columns: List[str],
                                 month_col: str = 'Application_month',
                                 data_selection_col: str = 'Data_selection',
                                 account_id_col: str = 'digitalLoanAccountId') -> pd.DataFrame:
    """
    Calculate PSI for each feature comparing training period (June 2024 to March 2025)
    vs each month after March 2025, overall and by segments.

    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe
    feature_list : List[str]
        List of features to calculate PSI for
    segment_columns : List[str]
        List of segment columns
    month_col : str
        Name of month column
    data_selection_col : str
        Name of data selection column (identifies train period)
    account_id_col : str
        Name of account ID column for counting distinct accounts

    Returns:
    --------
    pd.DataFrame with PSI values with one row per feature-month-segment combination
    """
    # Create a copy to avoid modifying original
    df = df.copy()

    # Identify training and test periods
    train_df = df[df[data_selection_col] == 'Train'].copy()
    test_df = df[df[data_selection_col] != 'Train'].copy()

    if len(train_df) == 0:
        raise ValueError("No training data found. Check Data_selection column.")

    print(f"Training period: {train_df[month_col].min()} to {train_df[month_col].max()}")
    print(f"Test period: {test_df[month_col].min()} to {test_df[month_col].max()}")

    # Identify feature types
    feature_types = identify_feature_types(df, feature_list)

    # Create binning strategy based on training period
    binning_info = create_bins_for_features(
        df,
        feature_types['numerical'],
        feature_types['categorical'],
        train_df
    )

    # Get sorted test months
    test_months = sorted(test_df[month_col].unique())

    results = []

    # Calculate overall PSI
    for feature in feature_list:
        if feature not in df.columns:
            continue

        # Apply binning to entire dataset
        df[f'{feature}_binned'] = apply_binning(df, feature, binning_info[feature])
        # print(f"Feature binned {df[f'{feature}_binned']}")
        # Get training period distribution (baseline)
        train_baseline = df[df[data_selection_col] == 'Train'][f'{feature}_binned'].value_counts(normalize=True)

        # Calculate PSI for each test month
        for month in test_months:
            actual_dist = df[df[month_col] == month][f'{feature}_binned'].value_counts(normalize=True)
            psi_value = calculate_psi(train_baseline, actual_dist)

            # Calculate average percentages across all bins
            expected_avg_pct = train_baseline.mean() * 100
            actual_avg_pct = actual_dist.mean() * 100

            # # Count distinct accounts for segment
            # base_segment_count = train_segment[account_id_col].nunique()
            # actual_segment_count = actual_segment[account_id_col].nunique()


            results.append({
                'Feature': feature,
                'Feature_Type': binning_info[feature]['type'],
                'Segment_Column': 'Overall',
                'Segment_Value': 'All',
                'Month': f"{month}",
                'Base_Month': 'Train (Jun 2024 - Mar 2025)',
                'Current_Month': month,
                'Expected_Percentage': expected_avg_pct,
                'Actual_Percentage': actual_avg_pct,
                'PSI': psi_value
            })

    # Calculate PSI by segments
    for segment_col in segment_columns:
        if segment_col not in df.columns:
            continue

        segments = df[segment_col].dropna().unique()

        for segment_val in segments:
            segment_df = df[df[segment_col] == segment_val]

            for feature in feature_list:
                if feature not in df.columns:
                    continue

                # Get training period distribution for segment
                train_segment = segment_df[segment_df[data_selection_col] == 'Train']
                if len(train_segment) == 0:
                    continue

                train_baseline = train_segment[f'{feature}_binned'].value_counts(normalize=True)

                # Calculate PSI for each test month
                for month in test_months:
                    actual_segment = segment_df[segment_df[month_col] == month]
                    if len(actual_segment) == 0:
                        continue

                    actual_dist = actual_segment[f'{feature}_binned'].value_counts(normalize=True)
                    psi_value = calculate_psi(train_baseline, actual_dist)

                    # Calculate average percentages across all bins
                    expected_avg_pct = train_baseline.mean() * 100
                    actual_avg_pct = actual_dist.mean() * 100

                    # Count distinct accounts for segment
                    base_segment_count = train_segment[account_id_col].nunique()
                    actual_segment_count = actual_segment[account_id_col].nunique()

                    results.append({
                        'Feature': feature,
                        'Feature_Type': binning_info[feature]['type'],
                        'Segment_Column': segment_col,
                        'Segment_Value': segment_val,
                        'Month': f"{month}",
                        'Base_Month': 'Train (Jun 2024 - Mar 2025)',
                        'Current_Month': month,
                        'Base_Count': base_segment_count,
                        'Actual_Count': actual_segment_count,
                        'Expected_Percentage': expected_avg_pct,
                        'Actual_Percentage': actual_avg_pct,
                        'PSI': psi_value
                    })

    return pd.DataFrame(results)


def calculate_bin_level_psi(df: pd.DataFrame,
                            feature_list: List[str],
                            segment_columns: List[str],
                            month_col: str = 'Application_month',
                            data_selection_col: str = 'Data_selection',
                            account_id_col: str = 'digitalLoanAccountId') -> pd.DataFrame:
    """
    Calculate bin-level PSI for each feature comparing training period
    vs each month after March 2025, overall and by segments.

    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe
    feature_list : List[str]
        List of features to calculate PSI for
    segment_columns : List[str]
        List of segment columns
    month_col : str
        Name of month column
    data_selection_col : str
        Name of data selection column
    account_id_col : str
        Name of account ID column for counting distinct accounts

    Returns:
    --------
    pd.DataFrame with bin-level PSI details including bin ranges
    """
    # Create a copy to avoid modifying original
    df = df.copy()

    # Identify training and test periods
    train_df = df[df[data_selection_col] == 'Train'].copy()
    test_df = df[df[data_selection_col] != 'Train'].copy()

    if len(train_df) == 0:
        raise ValueError("No training data found. Check Data_selection column.")

    print(f"Training period: {train_df[month_col].min()} to {train_df[month_col].max()}")
    print(f"Test period: {test_df[month_col].min()} to {test_df[month_col].max()}")

    # Identify feature types
    feature_types = identify_feature_types(df, feature_list)

    # Create binning strategy based on training period
    binning_info = create_bins_for_features(
        df,
        feature_types['numerical'],
        feature_types['categorical'],
        train_df
    )

    # Get sorted test months
    test_months = sorted(test_df[month_col].unique())

    results = []
    epsilon = 0.0001

    # Calculate overall bin-level PSI
    for feature in feature_list:
        if feature not in df.columns:
            continue

        # Apply binning to entire dataset
        df[f'{feature}_binned'] = apply_binning(df, feature, binning_info[feature])
        # print(df[f'{feature}_binned'])

        # Get training period distribution (baseline)
        train_baseline = df[df[data_selection_col] == 'Train'][f'{feature}_binned'].value_counts(normalize=True)

        # Calculate bin-level PSI for each test month
        for month in test_months:
            month_data = df[df[month_col] == month]
            actual_dist = month_data[f'{feature}_binned'].value_counts(normalize=True)

            # Count distinct accounts
            base_count = df[df[data_selection_col] == 'Train'][account_id_col].nunique()
            actual_count = month_data[account_id_col].nunique()

            # Get all bins
            all_bins = train_baseline.index.union(actual_dist.index)

            for bin_name in all_bins:
                # Simplified epsilon logic - no redundancy
                expected_pct = train_baseline.get(bin_name, 0)
                actual_pct = actual_dist.get(bin_name, 0)

                # Add epsilon only if zero
                expected_pct = epsilon if expected_pct == 0 else expected_pct
                actual_pct = epsilon if actual_pct == 0 else actual_pct

                # Calculate bin-level PSI
                bin_psi = (actual_pct - expected_pct) * np.log(actual_pct / expected_pct)

                # Get bin range information
                bin_ranges = binning_info[feature]['bin_ranges']
                if bin_name in bin_ranges:
                    bin_min = bin_ranges[bin_name]['min']
                    bin_max = bin_ranges[bin_name]['max']
                    bin_range = bin_ranges[bin_name]['range_str']
                else:
                    # For categorical or special bins (Missing, Others)
                    bin_min = None
                    bin_max = None
                    bin_range = bin_name

                results.append({
                    'Feature': feature,
                    'Feature_Type': binning_info[feature]['type'],
                    'Segment_Column': 'Overall',
                    'Segment_Value': 'All',
                    'Month': f"{month}",
                    'Base_Month': 'Train (Jun 2024 - Mar 2025)',
                    'Current_Month': month,
                    'Base_Count': base_count,
                    'Actual_Count': actual_count,
                    'Bin': bin_name,
                    'Bin_Range': bin_range,
                    'Bin_Min': bin_min,
                    'Bin_Max': bin_max,
                    'Base_Percentage': (train_baseline.get(bin_name, 0) * 100),
                    'Actual_Percentage': (actual_dist.get(bin_name, 0) * 100),
                    'Bin_PSI': bin_psi
                })

    # Calculate bin-level PSI by segments
    for segment_col in segment_columns:
        if segment_col not in df.columns:
            continue

        segments = df[segment_col].dropna().unique()

        for segment_val in segments:
            segment_df = df[df[segment_col] == segment_val]

            for feature in feature_list:
                if feature not in df.columns:
                    continue

                # Get training period distribution for segment
                train_segment = segment_df[segment_df[data_selection_col] == 'Train']
                if len(train_segment) == 0:
                    continue

                train_baseline = train_segment[f'{feature}_binned'].value_counts(normalize=True)

                # Calculate bin-level PSI for each test month
                for month in test_months:
                    actual_segment = segment_df[segment_df[month_col] == month]
                    if len(actual_segment) == 0:
                        continue

                    actual_dist = actual_segment[f'{feature}_binned'].value_counts(normalize=True)

                    # Count distinct accounts for segment
                    base_segment_count = train_segment[account_id_col].nunique()
                    actual_segment_count = actual_segment[account_id_col].nunique()

                    # Get all bins
                    all_bins = train_baseline.index.union(actual_dist.index)

                    for bin_name in all_bins:
                        # Simplified epsilon logic - no redundancy
                        expected_pct = train_baseline.get(bin_name, 0)
                        actual_pct = actual_dist.get(bin_name, 0)

                        # Add epsilon only if zero
                        expected_pct = epsilon if expected_pct == 0 else expected_pct
                        actual_pct = epsilon if actual_pct == 0 else actual_pct

                        # Calculate bin-level PSI
                        bin_psi = (actual_pct - expected_pct) * np.log(actual_pct / expected_pct)

                        # Get bin range information
                        bin_ranges = binning_info[feature]['bin_ranges']
                        if bin_name in bin_ranges:
                            bin_min = bin_ranges[bin_name]['min']
                            bin_max = bin_ranges[bin_name]['max']
                            bin_range = bin_ranges[bin_name]['range_str']
                        else:
                            # For categorical or special bins (Missing, Others)
                            bin_min = None
                            bin_max = None
                            bin_range = bin_name

                        results.append({
                            'Feature': feature,
                            'Feature_Type': binning_info[feature]['type'],
                            'Segment_Column': segment_col,
                            'Segment_Value': segment_val,
                            'Month': f"{month}",
                            'Base_Month': 'Train (Jun 2024 - Mar 2025)',
                            'Current_Month': month,
                            'Base_Count': base_segment_count,
                            'Actual_Count': actual_segment_count,
                            'Bin': bin_name,
                            'Bin_Range': bin_range,
                            'Bin_Min': bin_min,
                            'Bin_Max': bin_max,
                            'Base_Percentage': (train_baseline.get(bin_name, 0) * 100),
                            'Actual_Percentage': (actual_dist.get(bin_name, 0) * 100),
                            'Bin_PSI': bin_psi
                        })

    return pd.DataFrame(results)

# SIL V1

## SIL

#### Query from risk_mart.sil_risk_ds_master_20230101_20250309_v2

##### 'Alpha - CIC-SIL-Model'

###### Delete the data

In [9]:
sq = """
delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='Alpha - CIC-SIL-Model'
and modelVersionId = 'v1'
;
"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=eebcd27f-442e-4e03-8ded-19719b5a33b5>

In [10]:
sq = """select distinct
    r.customerId customer_id ,
    r.digitalLoanAccountId,
    r.cic_score,
    r.cic_Personal_Loans_granted_contracts_amt_24M,
    r.cic_days_since_last_inquiry, 
    r.cic_cnt_active_contracts,
    r.cic_vel_contract_nongranted_cnt_12on24,
    r.cic_max_amt_granted_24M, 
    r.cic_zero_non_granted_ever_flag,
    r.cic_tot_active_contracts_util,
    r.cic_vel_contract_granted_amt_12on24,
    r.cic_zero_granted_ever_flag,
    case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
    when lower(loanmaster.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
    date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
    case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
         between '2024-06-01' and '2024-09-30' then 'Dev_Train'
         when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-06-01' then 'Pre_Train'
                  else 'Dev_Test' end as Data_selection 
from risk_mart.sil_risk_ds_master_20230101_20250309_v2 r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where cic_score is not null
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-03-24'
;"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 11bf90b9-d1c1-4977-9add-591f5630c339 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (204452, 15)


In [11]:
feature_column = ['cic_Personal_Loans_granted_contracts_amt_24M',
       'cic_days_since_last_inquiry', 'cic_cnt_active_contracts',
       'cic_vel_contract_nongranted_cnt_12on24', 'cic_max_amt_granted_24M',
       'cic_zero_non_granted_ever_flag', 'cic_tot_active_contracts_util',
       'cic_vel_contract_granted_amt_12on24', 'cic_zero_granted_ever_flag']

In [12]:
dfd = transform_data(data, feature_column, a='cic_score', modelDisplayName='Alpha - CIC-SIL-Model') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3115665,1fa4c68a-bd9c-46ac-a75e-f794c76c45e9,6f2e14cb-400f-4651-8339-7d85119c3bf7,0.179781,2026-03-18T09:42:08.864069,2026-03-18T09:42:08.864069,Alpha - CIC-SIL-Model,v1,"{""cic_days_since_last_inquiry"": 841.0, ""cic_ze...",sil_march 25 models,2c9e536d-7b1d-4976-9bfe-54b30043d85a,2026-03-18T09:42:08.864069,{},,android,Dev_Test,2024-12-18
1,3303912,fdbecf5e-1d74-4418-8828-9c72210d59ba,9aa750e5-ab36-4c5d-9432-a8971beac1dc,0.191742,2026-03-18T09:42:08.864069,2026-03-18T09:42:08.864069,Alpha - CIC-SIL-Model,v1,"{""cic_days_since_last_inquiry"": 33.0, ""cic_vel...",sil_march 25 models,d881a086-a213-408d-a31d-9fa60f8dd14e,2026-03-18T09:42:08.864069,{},,android,Dev_Test,2025-03-05
2,3282520,b85d938e-1e15-42d9-bcc2-df4836f31043,73535343-8b72-412a-b0e7-5923f03f4e60,0.199273,2026-03-18T09:42:08.864069,2026-03-18T09:42:08.864069,Alpha - CIC-SIL-Model,v1,"{""cic_days_since_last_inquiry"": 0.0, ""cic_vel_...",sil_march 25 models,4c96bb27-4743-4dd0-ba02-b8efe2095eb3,2026-03-18T09:42:08.864069,{},,android,Dev_Test,2025-02-23
3,3190917,0d074486-b399-446c-9972-b65d9439f466,9992ddae-2c1a-4f0c-9486-35e57048bab6,0.256916,2026-03-18T09:42:08.864575,2026-03-18T09:42:08.864575,Alpha - CIC-SIL-Model,v1,"{""cic_days_since_last_inquiry"": 335.0, ""cic_cn...",sil_march 25 models,cda007b1-7607-46d7-aaca-b5d4de510d56,2026-03-18T09:42:08.864575,{},,android,Dev_Test,2025-01-15
4,2676267,5af0e5b0-2488-4f8e-8561-b6747d13f898,5fa00721-cc77-4b41-8edd-7cc39662b162,0.226442,2026-03-18T09:42:08.864575,2026-03-18T09:42:08.864575,Alpha - CIC-SIL-Model,v1,"{""cic_days_since_last_inquiry"": 0.0, ""cic_vel_...",sil_march 25 models,8d278862-8a5f-4ae7-a1fc-78ecfb2748e9,2026-03-18T09:42:08.864575,{},,android,Dev_Train,2024-07-20


In [13]:

result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,105730,2024-10-01,2025-03-08
1,Dev_Train,62128,2024-06-01,2024-09-30
2,Pre_Train,36594,2023-01-10,2024-05-31


In [14]:
# Upload to BigQuery
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=6fbd39f0-7233-4df2-9fd2-a23363ff12c2>

##### Alpha Sil Stack Model 

In [15]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='Alpha - StackingModel'
and modelVersionId = 'v1'
;"""

client.query(sq)    

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=e5d2c3e9-7059-4018-b4ee-258b89cba361>

In [16]:
sq = """ 
select distinct 
r.customerId customer_id ,
r.digitalLoanAccountId,
r.alpha_stack_score,
r.beta_demo_score,
r.cic_score,
r.apps_score,
r.credo_gen_score,
    case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
    when lower(loanmaster.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2024-06-01' and '2024-09-30' then 'Dev_Train'
        when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-06-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
from `risk_mart.sil_risk_ds_master_20230101_20250309_v2` r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where alpha_stack_score is not null
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-03-24'
;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID c994e100-487f-464d-9d45-803d5692c4e5 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (204452, 10)


In [17]:
data.columns
data.rename(columns={'beta_demo_score':'sb_demo_score', 'cic_score':'s_cic_score', 
                    'apps_score':'s_apps_score', 'credo_gen_score':'s_credo_score'}, inplace = True)
data.columns

Index(['customer_id', 'digitalLoanAccountId', 'alpha_stack_score',
       'sb_demo_score', 's_cic_score', 's_apps_score', 's_credo_score',
       'osType', 'application_date', 'Data_selection'],
      dtype='object')

In [18]:
feature_column = ['sb_demo_score',
       's_cic_score', 's_apps_score',
       's_credo_score']

dfd = transform_data(data, feature_column, a='alpha_stack_score', modelDisplayName='Alpha - StackingModel') 
dfd.head()


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3282520,b85d938e-1e15-42d9-bcc2-df4836f31043,9c4dd079-2d40-4350-a80c-a4174dd6b5b7,0.231882,2026-03-18T09:47:01.129461,2026-03-18T09:47:01.129461,Alpha - StackingModel,v1,"{""sb_demo_score"": 0.091355425025708, ""s_cic_sc...",sil_march 25 models,2c2a4f63-2cfd-44a5-a41d-4c55c5eb020e,2026-03-18T09:47:01.129461,{},,android,Dev_Test,2025-02-23
1,3190917,0d074486-b399-446c-9972-b65d9439f466,bbb50988-423d-4db1-a25b-95a57283b54c,0.570641,2026-03-18T09:47:01.129461,2026-03-18T09:47:01.129461,Alpha - StackingModel,v1,"{""sb_demo_score"": 0.0864386368508605, ""s_cic_s...",sil_march 25 models,4712a396-5855-4f1d-9fd7-6824afe8d915,2026-03-18T09:47:01.129461,{},,android,Dev_Test,2025-01-15
2,3142249,ffc408b0-ac1f-4ea8-ad3c-1575120d74c4,29bb8ae2-e87b-41fb-931b-c9407403eabd,0.110482,2026-03-18T09:47:01.129461,2026-03-18T09:47:01.129461,Alpha - StackingModel,v1,"{""sb_demo_score"": 0.18219962372724177, ""s_cic_...",sil_march 25 models,a0f663af-0c51-4d8a-9695-585dd3d5a7ed,2026-03-18T09:47:01.129461,{},,android,Dev_Test,2024-12-26
3,2928844,a10e2336-40e0-4a22-839c-e72c05052775,ba9e92f9-2e0b-4435-910a-f77b84a4c18c,0.179695,2026-03-18T09:47:01.129461,2026-03-18T09:47:01.129461,Alpha - StackingModel,v1,"{""sb_demo_score"": 0.05788462052362685, ""s_cic_...",sil_march 25 models,b437671e-4de7-44c4-be42-266244fb11f6,2026-03-18T09:47:01.129461,{},,android,Dev_Test,2024-10-10
4,3133181,bd2976e9-f995-42ca-acc4-b15ac9a7d993,d0f037bc-9c32-435f-8a79-e9df9b3f17eb,0.268717,2026-03-18T09:47:01.129461,2026-03-18T09:47:01.129461,Alpha - StackingModel,v1,"{""sb_demo_score"": 0.1008855110172025, ""s_cic_s...",sil_march 25 models,65c9e040-93f6-49b9-9b11-5f59a9398d6f,2026-03-18T09:47:01.129461,{},,android,Dev_Test,2024-12-23


In [19]:

result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,105730,2024-10-01,2025-03-08
1,Dev_Train,62128,2024-06-01,2024-09-30
2,Pre_Train,36594,2023-01-10,2024-05-31


In [20]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=5fff5019-8b47-46eb-9bf0-cfc4347e199b>

##### Beta Sil App Score

In [21]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='apps_score_model_sil'
and modelVersionId = 'v1'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=a7f01381-ec1e-48e7-87c8-1c19ab176391>

In [22]:
sq = """ 
select distinct
r.customerId customer_id ,
r.digitalLoanAccountId,
r.apps_score,
r.app_cnt_absence_tag_30d,
r.app_cnt_absence_tag_90d ,
r.app_cnt_business_ever ,
r.app_cnt_competitors_30d ,
r.app_cnt_competitors_90d ,
r.app_cnt_education_ever ,
r.app_cnt_finance_7d ,
r.app_cnt_finance_90d ,
r.app_cnt_music_and_audio_ever ,
r.app_cnt_payday_90d ,
r.app_cnt_rated_for_3plus_ever ,
r.app_cnt_travel_and_local_ever ,
r.app_first_competitors_install_to_apply_days ,
r.app_first_payday_install_to_apply_days ,
r.app_median_time_bw_installed_mins_30d ,
r.app_vel_finance_30_over_365 ,
    case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
    when lower(loanmaster.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
    date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
    case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
         between '2023-12-01' and '2024-06-30' then 'Dev_Train'
         when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2023-12-01' then 'Pre_Train'
                  else 'Dev_Test' end as Data_selection 
from `risk_mart.sil_risk_ds_master_20230101_20250309_v2` r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where apps_score is not null
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-03-20'
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 8247c67c-852f-4954-adcf-e6641ce6d151 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (317384, 22)


Sil App score V1 data taken from `risk_mart.sil_risk_ds_master_20230101_20250309_v2` which Bala created and we are missing the dl_score and cb_score in this. When connected with Oleh he says it is too old a data to recall. **So no change**

In [ ]:
feature_column = ['app_cnt_rated_for_3plus_ever',
       'app_cnt_education_ever', 'app_cnt_business_ever',
       'app_cnt_music_and_audio_ever',
       'app_cnt_travel_and_local_ever', 'app_cnt_finance_7d',
       'app_cnt_competitors_30d', 'app_cnt_absence_tag_30d',
        'app_cnt_absence_tag_90d',
       'app_cnt_finance_90d', 'app_cnt_competitors_90d',
       'app_cnt_payday_90d',
       'app_median_time_bw_installed_mins_30d',
       'app_first_competitors_install_to_apply_days',
       'app_first_payday_install_to_apply_days',
       'app_vel_finance_30_over_365']

dfd = transform_data(data, feature_column, a='apps_score', modelDisplayName='apps_score_model_sil') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3087954,4aad9fae-c843-49b2-b9ea-a0edcec2e39e,56c452b7-6c48-4670-b16f-07ae48be1762,0.403023,2026-03-18T09:50:05.123208,2026-03-18T09:50:05.123208,apps_score_model_sil,v1,"{""app_cnt_rated_for_3plus_ever"": 37.0, ""app_cn...",sil_march 25 models,f317f0c3-6934-4de9-96fb-3495291f7b44,2026-03-18T09:50:05.123208,{},,android,Dev_Test,2024-12-08
1,3070445,913c0881-7bb1-46b6-b5b5-f63ea29b2138,5eaba488-d65b-4afc-a29b-eb37730ddf89,0.596790,2026-03-18T09:50:05.123208,2026-03-18T09:50:05.123208,apps_score_model_sil,v1,"{""app_cnt_rated_for_3plus_ever"": 52.0, ""app_cn...",sil_march 25 models,32e1cfb1-808f-481d-8afc-11b61d1cfaac,2026-03-18T09:50:05.123208,{},,android,Dev_Test,2024-12-01
2,2482442,bcedccfd-5a7b-4828-9179-a28a11a6cd70,3a771f18-b6af-4924-b8d1-5668922cd275,0.880760,2026-03-18T09:50:05.123208,2026-03-18T09:50:05.123208,apps_score_model_sil,v1,"{""app_cnt_rated_for_3plus_ever"": 47.0, ""app_cn...",sil_march 25 models,3f202da0-4161-4b33-afb0-6ac253a4d1fd,2026-03-18T09:50:05.123208,{},,android,Dev_Train,2024-04-26
3,3161253,dd170581-b01e-4ed7-a906-f51529ffa3f1,b2c3e3da-b2f2-4362-bf5d-d3a0e1bd9d36,0.481553,2026-03-18T09:50:05.123208,2026-03-18T09:50:05.123208,apps_score_model_sil,v1,"{""app_cnt_rated_for_3plus_ever"": 21.0, ""app_cn...",sil_march 25 models,d7b0064e-105d-4744-b084-2dbbe24ba4a8,2026-03-18T09:50:05.123208,{},,android,Dev_Test,2025-01-02
4,3202997,d11927eb-cb7e-4d4b-b43f-16a26ac25791,8757b653-db1b-4e28-90da-680d5aebf9b1,0.695988,2026-03-18T09:50:05.124207,2026-03-18T09:50:05.124207,apps_score_model_sil,v1,"{""app_cnt_rated_for_3plus_ever"": 18.0, ""app_cn...",sil_march 25 models,0b2f1688-0cc0-4de7-bb3d-69a5e2c19de0,2026-03-18T09:50:05.124207,{},,android,Dev_Test,2025-01-20


In [24]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,200327,2024-07-01,2025-03-08
1,Dev_Train,86480,2023-12-01,2024-06-30
2,Pre_Train,30577,2023-01-02,2023-11-30


In [25]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=3b0be8e6-e6a9-40c7-beff-234f325b2494>

##### Beta SIL Demo Score

In [26]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='Beta - DemoScoreModel'
and modelVersionId = 'v1'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=5c877ffa-f65c-44c1-ad7d-8f655811d574>

In [27]:
sq = """
select distinct
r.customerId customer_id ,
r.digitalLoanAccountId,
r.beta_demo_score,
r.beta_de_ln_vas_opted_flag ,
r.beta_de_ln_doc_type_rolled ,
r.beta_de_ln_marital_status ,
r.beta_de_ln_age_bin ,
r.beta_de_ln_province_bin ,
r.beta_de_ln_ref2_type ,
r.beta_de_ln_education_level ,
r.beta_de_ln_ref1_type ,
r.beta_de_ln_industry_new_bin ,
r.beta_de_ln_appln_day_of_week ,
r.beta_de_onb_name_email_match_score ,
r.beta_de_ln_employment_type_new_bin ,
r.beta_de_ln_telconame ,
r.beta_de_time_bw_onb_loan_appln_mins ,
r.beta_de_ln_source_of_funds_new_bin ,
r.beta_de_ln_brand_bin ,
r.beta_de_ln_email_primary_domain ,
    case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
    when lower(loanmaster.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2023-07-01' and '2024-06-30' then 'Dev_Train'
        when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2023-07-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
from `risk_mart.sil_risk_ds_master_20230101_20250309_v2` r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where beta_demo_score is not null
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-03-20'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID bb41929e-fd32-4902-999f-0b6d47f14d7e successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (349206, 23)


In [28]:
feature_column = ['beta_de_ln_vas_opted_flag',
       'beta_de_ln_doc_type_rolled', 'beta_de_ln_marital_status',
       'beta_de_ln_age_bin', 'beta_de_ln_province_bin',
       'beta_de_ln_ref2_type', 'beta_de_ln_education_level',
       'beta_de_ln_ref1_type', 'beta_de_ln_industry_new_bin',
       'beta_de_ln_appln_day_of_week',
       'beta_de_onb_name_email_match_score',
       'beta_de_ln_employment_type_new_bin', 'beta_de_ln_telconame',
       'beta_de_time_bw_onb_loan_appln_mins',
       'beta_de_ln_source_of_funds_new_bin', 'beta_de_ln_brand_bin',
       'beta_de_ln_email_primary_domain']

dfd = transform_data(data, feature_column, a='beta_demo_score', modelDisplayName='Beta - DemoScoreModel') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

The shape of the transformed dataframe is:	 (349206, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3101815,e9f44ae5-b912-405d-a8af-e0a09ed3eeab,97fbc281-e5a5-409f-bc33-7b06f8a4f1c0,0.120397,2026-03-18T09:54:05.160392,2026-03-18T09:54:05.160392,Beta - DemoScoreModel,v1,"{""beta_de_ln_vas_opted_flag"": ""0"", ""beta_de_ln...",sil_march 25 models,b950c986-043a-47b1-b3ba-1718d80d5dba,2026-03-18T09:54:05.160392,{},,android,Dev_Test,2024-12-13
1,2452397,08728116-dcb2-4737-91d0-3bbaeeec1f99,938eefb7-884e-4252-a9ac-2fcab9bd641b,0.047886,2026-03-18T09:54:05.160392,2026-03-18T09:54:05.160392,Beta - DemoScoreModel,v1,"{""beta_de_ln_vas_opted_flag"": ""1"", ""beta_de_ln...",sil_march 25 models,c5977e06-08c5-4c69-93c8-86f6e0fcd5a7,2026-03-18T09:54:05.160392,{},,android,Dev_Train,2024-04-05
2,2669719,419a9cc7-52ff-4412-9b50-dc16752ffd0d,b6f91149-7585-4baa-8edb-5fda38ce3e38,0.100074,2026-03-18T09:54:05.160392,2026-03-18T09:54:05.160392,Beta - DemoScoreModel,v1,"{""beta_de_ln_vas_opted_flag"": ""1"", ""beta_de_ln...",sil_march 25 models,50a78176-bc97-42f0-b9df-43c99fbd7b3d,2026-03-18T09:54:05.160392,{},,android,Dev_Test,2024-07-18
3,3135091,3179c776-ce9c-45b5-939f-043c2d859569,1b1b65c2-2d8d-4d37-b076-9cdd326d768e,0.220649,2026-03-18T09:54:05.161389,2026-03-18T09:54:05.161389,Beta - DemoScoreModel,v1,"{""beta_de_ln_vas_opted_flag"": ""1"", ""beta_de_ln...",sil_march 25 models,fda4a711-8794-45a8-a5eb-3151da9bc95a,2026-03-18T09:54:05.161389,{},,android,Dev_Test,2024-12-24
4,2300971,865bdaf1-07b3-496c-83ee-766c803e30f1,5d491259-8111-40bf-9351-db66c24075c7,0.109321,2026-03-18T09:54:05.161389,2026-03-18T09:54:05.161389,Beta - DemoScoreModel,v1,"{""beta_de_ln_vas_opted_flag"": ""0"", ""beta_de_ln...",sil_march 25 models,1d5e2b21-6ed0-45d1-ab09-4516a27a36b0,2026-03-18T09:54:05.161389,{},,android,Dev_Train,2023-11-11


In [31]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,219881,2024-07-01,2025-03-08
1,Dev_Train,121661,2023-07-01,2024-06-30
2,Pre_Train,7664,2023-01-02,2023-06-30


In [32]:
# Upload to BigQuery
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=4ae14c07-7a96-487c-9f64-c420fad07222>

##### Beta SIL STACK Score Model

In [33]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='Beta - StackScoreModel'
and modelVersionId = 'v1'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=45b4de64-1870-4945-8074-940030a598b0>

In [34]:
sq = """ 
select  distinct
r.customerId customer_id ,
r.digitalLoanAccountId,
r.beta_stack_score,
r.apps_score,
r.credo_gen_score,
r.beta_demo_score,
    case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
    when lower(loanmaster.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2023-07-01' and '2024-06-30' then 'Dev_Train'
        when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2023-07-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
from `risk_mart.sil_risk_ds_master_20230101_20250309_v2` r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where beta_stack_score is not null
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-03-20'
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID b0db9926-230c-450b-b715-09550dd0983c successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (349206, 9)


In [35]:
feature_column = ['apps_score', 'credo_gen_score', 'beta_demo_score']
dfd = transform_data(data, feature_column, a='beta_stack_score', modelDisplayName='Beta - StackScoreModel') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

The shape of the transformed dataframe is:	 (349206, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3101815,e9f44ae5-b912-405d-a8af-e0a09ed3eeab,549c8759-8304-40fe-8bc2-b015ffa96690,0.142555,2026-03-18T09:59:05.714653,2026-03-18T09:59:05.714653,Beta - StackScoreModel,v1,"{""apps_score"": 0.5466209099472384, ""credo_gen_...",sil_march 25 models,4e4e010b-77fb-4bdf-acf2-3cb8ca435152,2026-03-18T09:59:05.714653,{},,android,Dev_Test,2024-12-13
1,3303912,fdbecf5e-1d74-4418-8828-9c72210d59ba,207314f8-bc23-4839-bcca-3c1ad92ebb9f,0.117759,2026-03-18T09:59:05.714653,2026-03-18T09:59:05.714653,Beta - StackScoreModel,v1,"{""apps_score"": 0.571720603830244, ""credo_gen_s...",sil_march 25 models,1aef0999-bc61-4023-9f98-e18707430a26,2026-03-18T09:59:05.714653,{},,android,Dev_Test,2025-03-05
2,3233200,720ba4da-a4ae-45cb-a5fb-55975118d958,02fbdaf9-5043-44ec-99b9-a6bacdef1521,0.018404,2026-03-18T09:59:05.714653,2026-03-18T09:59:05.714653,Beta - StackScoreModel,v1,"{""apps_score"": 0.3819525698328896, ""credo_gen_...",sil_march 25 models,82d458d4-29f2-4c39-91ed-4c2032adaa5a,2026-03-18T09:59:05.714653,{},,android,Dev_Test,2025-02-02
3,3072026,d31f6d62-48f8-4c15-8275-35d7b7492060,b797a4fe-b244-406a-9412-6bbe289438f6,0.216488,2026-03-18T09:59:05.714653,2026-03-18T09:59:05.714653,Beta - StackScoreModel,v1,"{""apps_score"": 0.6625739574951893, ""credo_gen_...",sil_march 25 models,604a1fbe-5119-4635-b303-4d5553e63d7b,2026-03-18T09:59:05.714653,{},,android,Dev_Test,2024-12-02
4,2444010,eabbb17e-2529-4130-aaec-89444885db33,c8309f75-ec00-456d-8e59-746d4f750aa8,0.223510,2026-03-18T09:59:05.714653,2026-03-18T09:59:05.714653,Beta - StackScoreModel,v1,"{""apps_score"": 0.6097532016762911, ""credo_gen_...",sil_march 25 models,a8b0938d-41f6-42f4-b927-df5002a506de,2026-03-18T09:59:05.714653,{},,android,Dev_Test,2024-11-13


In [36]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,219881,2024-07-01,2025-03-08
1,Dev_Train,121661,2023-07-01,2024-06-30
2,Pre_Train,7664,2023-01-02,2023-06-30


In [37]:
# Upload to BigQuery
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=07516780-c477-44cf-a12c-042dbb5474d2>

##### Alpha  - IncomeEstimationModel

In [30]:
# sq = """  
# Select 
# distinct
# r.customerId customer_id ,
# r.digitalLoanAccountId,
# r.alpha_estimated_income,
# r.inc_alpha_cic_credit_avg_credit_limit,
# r.inc_alpha_cic_max_active_contracts_amt,
# r.inc_alpha_ln_company_name,
# r.inc_alpha_ln_age,
# r.inc_alpha_doc_type_rolled,
# r.inc_alpha_ln_brand,
# r.inc_alpha_ln_city,
# r.inc_alpha_ln_cnt_dependents,
# r.inc_alpha_ln_education_level,
# r.inc_alpha_ln_employment_type_new,
# r.inc_alpha_ln_gender,
# r.inc_alpha_ln_industry_new,
# r.inc_alpha_ln_loan_prod_type,
# r.inc_alpha_ln_marital_status_new,
# r.inc_alpha_ln_nature_of_work_new,
# r.inc_alpha_ln_osversion_bin,
# r.inc_alpha_ln_purpose,
# r.inc_alpha_ln_source_of_funds_new,
# r.inc_alpha_loan_monthly_income,
# r.inc_alpha_encoded_company_name_grouped,
#     case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
#     when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
#     when lower(loanmaster.deviceType) like '%andro%' then 'android'
#     else 'ios' end osType,
# from `risk_mart.sil_risk_ds_master_20230101_20250309_v2` r
# left join risk_credit_mis.loan_master_table loanmaster
#   ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
# where r.alpha_estimated_income is not null
# ;
# """
# data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
# print(f"The shape of the dataframe is:\t {data.shape}")


In [31]:
# feature_column = ['inc_alpha_cic_credit_avg_credit_limit',
#        'inc_alpha_cic_max_active_contracts_amt', 'inc_alpha_ln_age',
#        'inc_alpha_doc_type_rolled', 'inc_alpha_ln_brand', 'inc_alpha_ln_city',
#        'inc_alpha_ln_cnt_dependents', 'inc_alpha_ln_education_level',
#        'inc_alpha_ln_employment_type_new', 'inc_alpha_ln_gender',
#        'inc_alpha_ln_industry_new', 'inc_alpha_ln_loan_prod_type',
#        'inc_alpha_ln_marital_status_new', 'inc_alpha_ln_nature_of_work_new',
#        'inc_alpha_ln_osversion_bin', 'inc_alpha_ln_purpose',
#        'inc_alpha_ln_source_of_funds_new',
#        'inc_alpha_encoded_company_name_grouped']
# dfd = transform_data(data, feature_column, a='alpha_estimated_income', modelDisplayName='Alpha  - IncomeEstimationModel') 
# dfd.head()

In [32]:
# # Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details"
# job_config = bigquery.LoadJobConfig(
#     write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
# )
# job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
# job.result() 

##### Beta - IncomeEstimationModel

In [33]:
# sq = """ 
# select
# distinct
# r.customerId customer_id ,
# r.digitalLoanAccountId,
# r.beta_estimated_income,
# r.inc_beta_ln_loan_type,
# r.inc_beta_ln_education_level,
# r.inc_beta_ln_employment_type_new,
# r.inc_beta_ln_industry_new,
# r.inc_beta_ln_age,
# r.inc_beta_ln_brand,
# r.inc_beta_ln_city,
# r.inc_beta_ln_purpose,
# r.inc_beta_ln_osversion_bin,
# r.inc_beta_ln_postal_code,
# r.inc_beta_ln_gender,
# r.inc_beta_ln_doc_type_rolled,
# r.inc_beta_ln_cnt_dependents,
# r.inc_beta_ln_source_of_funds_new,
# r.inc_beta_ln_marital_status_new,
# r.inc_beta_encoded_company_name_grouped,
#     case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
#     when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
#     when lower(loanmaster.deviceType) like '%andro%' then 'android'
#     else 'ios' end osType,
# from `risk_mart.sil_risk_ds_master_20230101_20250309_v2` r
# left join risk_credit_mis.loan_master_table loanmaster
#   ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
# where r.alpha_estimated_income is not null
# ;
# """

# # data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
# data = client.query(sq).result().to_arrow().to_pandas()
# print(f"The shape of the dataframe is:\t {data.shape}")

In [34]:
# feature_column = ['inc_beta_ln_loan_type',
#        'inc_beta_ln_education_level', 'inc_beta_ln_employment_type_new',
#        'inc_beta_ln_industry_new', 'inc_beta_ln_age', 'inc_beta_ln_brand',
#        'inc_beta_ln_city', 'inc_beta_ln_purpose', 'inc_beta_ln_osversion_bin',
#        'inc_beta_ln_postal_code', 'inc_beta_ln_gender',
#        'inc_beta_ln_doc_type_rolled', 'inc_beta_ln_cnt_dependents',
#        'inc_beta_ln_source_of_funds_new', 'inc_beta_ln_marital_status_new',
#        'inc_beta_encoded_company_name_grouped',]

# dfd = transform_data(data, feature_column, a='beta_estimated_income', modelDisplayName='Beta - IncomeEstimationModel') 
# dfd.head()

In [35]:
# # Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details"
# job_config = bigquery.LoadJobConfig(
#     write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
# )
# job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
# job.result() 

# Cash V1

## Cash

##### Alpha Cash Stack Model

###### Trench 1

In [39]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='Alpha-Cash-Stack-Model'
and modelVersionId = 'v1'
and trenchCategory = 'Trench 1'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=2c5e1ee7-0412-4bb6-a2d5-bf973ab3a53e>

In [40]:
sq = """ 
select 
  r.customer_id,
  r.digitalLoanAccountId, 
  r.stack_score,
  r.demo_score,
  r.apps_score,
  r.credo_score,
  r.cic_score,
  r.stack_score_norm,
  r.ln_os_type osType,
   date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
    case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
         between '2024-10-01' and '2025-02-28' then 'Dev_Train'
         when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-10-01' then 'Pre_Train'
                  else 'Dev_Test' end as Data_selection 
from worktable_data_analysis.cash_alpha_trench1_applied_loans_backscored_20241001_20250930 r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where r.stack_score is not null
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
< '2025-09-24'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 37ac5425-8259-4dd1-962c-091079117025 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (66557, 11)


In [41]:
feature_column = ['demo_score', 'apps_score', 'credo_score', 'cic_score', 'stack_score_norm']

dfd = transform_data(data, feature_column, a='stack_score', modelDisplayName='Alpha-Cash-Stack-Model', tc='Trench 1', subscription_name = 'Cash September 25 Models') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3617018,8520e0eb-6773-4cb4-a5b4-69c91eca0803,e2506226-0936-4636-9a74-2b3c043ba80f,0.138373,2026-03-18T11:03:05.182259,2026-03-18T11:03:05.182259,Alpha-Cash-Stack-Model,v1,"{""demo_score"": 0.40352654844653746, ""apps_scor...",Cash September 25 Models,b99d4e6f-3bb2-403b-9488-2457e993d24b,2026-03-18T11:03:05.182259,{},Trench 1,Android,Dev_Test,2025-08-12
1,3402699,ed43cdc5-1246-4c9e-b289-d7502585b819,00cc1c95-4ab0-4731-8a5c-98db13a29504,0.451364,2026-03-18T11:03:05.183257,2026-03-18T11:03:05.183257,Alpha-Cash-Stack-Model,v1,"{""demo_score"": 0.34505338162558047, ""apps_scor...",Cash September 25 Models,8d6af214-63df-4c0d-a5c9-63cc69dde36e,2026-03-18T11:03:05.183257,{},Trench 1,Android,Dev_Test,2025-04-26
2,3280854,62bb8a30-0469-4767-ab8d-cde3604c97d1,f5868fc4-1649-4693-b8c9-8e754cdf2afb,0.608720,2026-03-18T11:03:05.183257,2026-03-18T11:03:05.183257,Alpha-Cash-Stack-Model,v1,"{""demo_score"": 0.44181173754633396, ""apps_scor...",Cash September 25 Models,4042f502-17c9-4086-b6e8-7ce958d9b4ad,2026-03-18T11:03:05.183257,{},Trench 1,Android,Dev_Train,2025-02-23
3,3234846,1be5b2b6-0ddf-4332-8d0d-86cd0e066839,70a6b783-d012-4621-8b97-9327d1b11b0a,0.635680,2026-03-18T11:03:05.183257,2026-03-18T11:03:05.183257,Alpha-Cash-Stack-Model,v1,"{""demo_score"": 0.48229734711959077, ""apps_scor...",Cash September 25 Models,d1a4dbfe-d00f-4a57-91c5-c7feafbce23b,2026-03-18T11:03:05.183257,{},Trench 1,Android,Dev_Train,2025-02-14
4,3196470,35a87a9b-e049-47f7-99af-cfa9c1055b2f,9875c597-f2fb-4558-b8b7-f5f0fcbe9d88,0.118642,2026-03-18T11:03:05.183257,2026-03-18T11:03:05.183257,Alpha-Cash-Stack-Model,v1,"{""demo_score"": 0.46808809164407844, ""apps_scor...",Cash September 25 Models,50b8d7a9-5426-4399-b353-08a6a13d710f,2026-03-18T11:03:05.183257,{},Trench 1,Android,Dev_Train,2025-01-22


In [42]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,33061,2025-03-01,2025-09-23
1,Dev_Train,33496,2024-10-01,2025-02-28


In [43]:
# Upload to BigQuery
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=e1e0c4ca-c917-4e44-8ccf-fcd5ac19974a>

###### Trench 2

In [44]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='Alpha-Cash-Stack-Model'
and modelVersionId = 'v1'
and trenchCategory = 'Trench 2'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=3c155a9c-5337-450d-9626-3445e149c11a>

In [45]:
sq = """ 
select 
  r.customer_id,
  r.digitalLoanAccountId, 
  r.stack_score,
  r.demo_score,
  r.apps_score,
  r.credo_score,
  r.cic_score,
  r.stack_score_norm,
  r.ln_os_type osType,
  date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
    case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
         between '2024-10-01' and '2025-02-28' then 'Dev_Train'
         when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-10-01' then 'Pre_Train'
                  else 'Dev_Test' end as Data_selection 
from worktable_data_analysis.cash_alpha_trench2_applied_loans_backscored_20241001_20250930 r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where r.stack_score is not null
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-09-24'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 0a194d2e-e00c-4252-90fb-946060ac1ea9 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (42288, 11)


In [46]:
feature_column = ['demo_score', 'apps_score', 'credo_score', 'cic_score', 'stack_score_norm']

dfd = transform_data(data, feature_column, a='stack_score', modelDisplayName='Alpha-Cash-Stack-Model', tc='Trench 2', subscription_name = 'Cash September 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

The shape of the transformed dataframe is:	 (42288, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3122221,5a8eee75-4b41-449e-8383-c56dfad3d447,e0d46a6d-a101-4b43-9995-8f49446f6bf0,0.240961,2026-03-18T11:04:46.634719,2026-03-18T11:04:46.634719,Alpha-Cash-Stack-Model,v1,"{""demo_score"": 0.47971759321801266, ""apps_scor...",Cash September 25 Models,71a66949-942c-4f9f-88a1-633e51822f07,2026-03-18T11:04:46.634719,{},Trench 2,Android,Dev_Test,2025-05-19
1,3490839,1abdf11f-eef7-4ad9-82cc-d6be20b3e39f,3615a27b-bbb5-48fc-97cd-decc8554342b,0.127637,2026-03-18T11:04:46.635722,2026-03-18T11:04:46.635722,Alpha-Cash-Stack-Model,v1,"{""demo_score"": 0.41662078954915643, ""apps_scor...",Cash September 25 Models,4ad41dad-4053-43c2-92ef-14b34b9f5836,2026-03-18T11:04:46.635722,{},Trench 2,Android,Dev_Test,2025-09-16
2,3174411,b5c5186d-b707-4209-bbca-7c82effc6f61,2cf4d153-7429-4796-a43d-4dddf9d4cba9,0.525420,2026-03-18T11:04:46.635722,2026-03-18T11:04:46.635722,Alpha-Cash-Stack-Model,v1,"{""demo_score"": 0.48245357945334927, ""apps_scor...",Cash September 25 Models,babec447-68fd-4a85-b79d-d1d755c0673b,2026-03-18T11:04:46.635722,{},Trench 2,Android,Dev_Test,2025-06-24
3,2950119,7192a7e8-dc39-47b9-9487-ebd0956ecd7d,0e03e7e7-5543-479f-8674-68a63cf14248,0.402889,2026-03-18T11:04:46.635722,2026-03-18T11:04:46.635722,Alpha-Cash-Stack-Model,v1,"{""demo_score"": 0.5060232774978061, ""apps_score...",Cash September 25 Models,2b75c81c-c0c0-4043-88fb-75c406f53c15,2026-03-18T11:04:46.635722,{},Trench 2,Android,Dev_Test,2025-07-10
4,2326557,91436a74-6d06-4244-96bb-33930b930227,3cd0beb7-9eee-42ca-98a1-02ea3d2b0645,0.599156,2026-03-18T11:04:46.635722,2026-03-18T11:04:46.635722,Alpha-Cash-Stack-Model,v1,"{""demo_score"": 0.5812363961506116, ""apps_score...",Cash September 25 Models,f2465cee-10af-4526-9e8b-e9c9727f54fb,2026-03-18T11:04:46.635722,{},Trench 2,Android,Dev_Train,2024-12-29


In [47]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,24849,2025-03-01,2025-09-23
1,Dev_Train,17439,2024-10-01,2025-02-28


In [48]:
# Upload to BigQuery
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=40de6505-1dbd-4d9e-9873-9987fd0fb0de>

###### Trench 3

In [49]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='Alpha-Cash-Stack-Model'
and modelVersionId = 'v1'
and trenchCategory = 'Trench 3'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=99ff10eb-e6b7-4bd3-82e4-f6fde8cb0093>

In [50]:
sq = """ 
select 
  r.customer_id,
  r.digitalLoanAccountId, 
  r.stack_score,
  r.demo_score,
  r.apps_score,
  r.credo_score,
  r.cic_score,
  r.stack_score_norm,
  r.ln_os_type osType,
  date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
    case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
         between '2024-10-01' and '2025-02-28' then 'Dev_Train'
         when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-10-01' then 'Pre_Train'
                  else 'Dev_Test' end as Data_selection 
from worktable_data_analysis.cash_alpha_trench3_applied_loans_backscored_20241001_20250930 r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where r.stack_score is not null
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))< '2025-09-24'
;
""" 
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID b8181c89-3e4d-40c7-a421-73af315dccbb successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (17937, 11)


In [51]:
feature_column = ['demo_score', 'apps_score', 'credo_score', 'cic_score', 'stack_score_norm']

dfd = transform_data(data, feature_column, a='stack_score', modelDisplayName='Alpha-Cash-Stack-Model', tc='Trench 3', subscription_name = 'Cash September 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

The shape of the transformed dataframe is:	 (17937, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3232459,d0913f75-aefa-4274-a679-371f65a8ab37,41748dfd-1200-49be-b827-986c3ff25602,0.645742,2026-03-18T11:06:36.583785,2026-03-18T11:06:36.583785,Alpha-Cash-Stack-Model,v1,"{""demo_score"": 0.47332194079062734, ""apps_scor...",Cash September 25 Models,663730da-4dc3-45b8-b0d5-e70f84b5e46f,2026-03-18T11:06:36.583785,{},Trench 3,Android,Dev_Test,2025-09-23
1,1946651,529510a0-6aa3-4009-81ca-9c9cee58c6c0,5a985d67-066c-4bb9-80ab-b95cf21584c3,0.553355,2026-03-18T11:06:36.583785,2026-03-18T11:06:36.583785,Alpha-Cash-Stack-Model,v1,"{""demo_score"": 0.4728667289116435, ""apps_score...",Cash September 25 Models,cd4c7b2a-179e-4608-9bd0-d351ea574460,2026-03-18T11:06:36.583785,{},Trench 3,Android,Dev_Test,2025-03-12
2,2016414,e1336272-00fe-46bc-a3c7-2c95b85dec22,de7d5215-267c-41ce-bbf3-6e81e848d02f,0.161681,2026-03-18T11:06:36.583785,2026-03-18T11:06:36.583785,Alpha-Cash-Stack-Model,v1,"{""demo_score"": 0.490716044327779, ""apps_score""...",Cash September 25 Models,9e8689dd-3eae-425d-b17b-8835b44154bb,2026-03-18T11:06:36.583785,{},Trench 3,Android,Dev_Test,2025-09-03
3,2253707,99bf1660-16b4-4b1a-96cf-4c3275b5cbf1,831a0ea1-c635-4893-928c-67f84bdf226d,0.363170,2026-03-18T11:06:36.584784,2026-03-18T11:06:36.584784,Alpha-Cash-Stack-Model,v1,"{""demo_score"": 0.5062712148390874, ""apps_score...",Cash September 25 Models,f4efc97e-c5b7-4078-8153-7a01c82b7cd7,2026-03-18T11:06:36.584784,{},Trench 3,Android,Dev_Train,2025-01-31
4,1473312,36faca39-2549-4a67-9c95-3bb9935c0508,5f0a1544-7b17-4303-87cc-78d80004a4ca,0.438229,2026-03-18T11:06:36.584784,2026-03-18T11:06:36.584784,Alpha-Cash-Stack-Model,v1,"{""demo_score"": 0.32875871137856677, ""apps_scor...",Cash September 25 Models,a2d7e9e1-e70e-4dd2-955f-36dca511a27c,2026-03-18T11:06:36.584784,{},Trench 3,Android,Dev_Test,2025-06-09


In [52]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,11563,2025-03-01,2025-09-23
1,Dev_Train,6374,2024-10-01,2025-02-28


In [53]:
# Upload to BigQuery
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=94e7c15e-b234-406a-b9b3-f0a0c7ce08a7>

##### Alpha Cash CIC Model

In [54]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='Alpha-Cash-CIC-Model'
and modelVersionId = 'v1'
and trenchCategory = 'Trench 1'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=387beb15-1cf6-4138-a9db-d078a7679399>

###### Trench 1

In [55]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,
r.ca_cic_score,
r.max_age_all_contracts_snapshot,
r.ratio_overdue_contracts_to_granted_contracts,
r.ScoreRange,
r.ln_loan_level_user_type,
r.has_ever_been_overdue,
r.latest_granted_contract_overdue_flag,
r.ratio_closed_over_new_granted_cnt_24M,
r.ratio_risky_contracts_to_granted_contracts,
r.Short_and_Term_Loans_granted_contracts_cnt_24M,
r.flg_zero_non_granted_ever,
r.Personal_Loans_granted_contracts_amt_24M,
r.CreditAvgCreditLimit,
r.flg_zero_granted_ever,
    case when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%os%' then 'ios'
    when lower(lmt.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-01-31' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection  
from worktable_data_analysis.cash_alpha_cic_all_applied_backscored_20240901_20250930 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where trench_category = 'Trench 1'
and r.ca_cic_score is not null 
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-09-24'
;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


Job ID 7a7eaad7-0bb0-4831-8fe6-5d8c8a35c1b9 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (76716, 19)


In [56]:
data.columns

Index(['customer_id', 'digitalLoanAccountId', 'ca_cic_score',
       'max_age_all_contracts_snapshot',
       'ratio_overdue_contracts_to_granted_contracts', 'ScoreRange',
       'ln_loan_level_user_type', 'has_ever_been_overdue',
       'latest_granted_contract_overdue_flag',
       'ratio_closed_over_new_granted_cnt_24M',
       'ratio_risky_contracts_to_granted_contracts',
       'Short_and_Term_Loans_granted_contracts_cnt_24M',
       'flg_zero_non_granted_ever', 'Personal_Loans_granted_contracts_amt_24M',
       'CreditAvgCreditLimit', 'flg_zero_granted_ever', 'osType',
       'application_date', 'Data_selection'],
      dtype='object')

In [57]:
feature_column = ['max_age_all_contracts_snapshot',
       'ratio_overdue_contracts_to_granted_contracts', 'ScoreRange',
       'ln_loan_level_user_type', 'has_ever_been_overdue',
       'latest_granted_contract_overdue_flag',
       'ratio_closed_over_new_granted_cnt_24M',
       'ratio_risky_contracts_to_granted_contracts',
       'Short_and_Term_Loans_granted_contracts_cnt_24M',
       'flg_zero_non_granted_ever', 'Personal_Loans_granted_contracts_amt_24M',
       'CreditAvgCreditLimit', 'flg_zero_granted_ever', 'ca_cic_score']

dfd = transform_data(data, feature_column, a='ca_cic_score', modelDisplayName='Alpha-Cash-CIC-Model', tc='Trench 1', subscription_name = 'Cash September 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

The shape of the transformed dataframe is:	 (76716, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,2986396,737f64c3-ea1c-4315-8498-1580dda8ffd5,11ca1d03-1640-401d-b573-ed3ab291e6f2,0.526884,2026-03-18T11:08:52.110023,2026-03-18T11:08:52.110023,Alpha-Cash-CIC-Model,v1,"{""max_age_all_contracts_snapshot"": 2944.0, ""ra...",Cash September 25 Models,322047a5-5878-4a94-aadf-f985398e553c,2026-03-18T11:08:52.110023,{},Trench 1,android,Dev_Train,2024-10-31
1,3364713,7935bbc9-551c-4f64-88dd-90ae030c7781,5496b90c-2594-484a-af34-599915ddddf0,0.453298,2026-03-18T11:08:52.110023,2026-03-18T11:08:52.110023,Alpha-Cash-CIC-Model,v1,"{""max_age_all_contracts_snapshot"": 2340.0, ""ra...",Cash September 25 Models,3ba7cda7-3de1-4169-8dd6-fc0dfa2afae9,2026-03-18T11:08:52.110023,{},Trench 1,ios,Dev_Test,2025-04-07
2,3047465,6d2d2a98-cd1b-4a48-b9d0-d490f43cb430,158d8fcc-acb1-4fb8-b91e-70956355c58c,0.428219,2026-03-18T11:08:52.111023,2026-03-18T11:08:52.111023,Alpha-Cash-CIC-Model,v1,"{""max_age_all_contracts_snapshot"": 1978.0, ""ra...",Cash September 25 Models,0bf6cdb9-12e0-4dc8-8908-a472a446cb09,2026-03-18T11:08:52.111023,{},Trench 1,ios,Dev_Train,2024-11-23
3,3329745,03dd5025-e675-44c4-86bf-9fa67682f067,3179b825-1ed5-4adb-a2ef-215657d197f0,0.314406,2026-03-18T11:08:52.111023,2026-03-18T11:08:52.111023,Alpha-Cash-CIC-Model,v1,"{""max_age_all_contracts_snapshot"": 3159.0, ""ra...",Cash September 25 Models,c88127f9-d02f-4564-93ef-132466b93eec,2026-03-18T11:08:52.111023,{},Trench 1,ios,Dev_Test,2025-03-20
4,3243559,a9019d7f-ee02-4431-9a5f-9f3fe98db82f,4efc203f-e3ed-402c-8b97-7d18e171ca72,0.643168,2026-03-18T11:08:52.111023,2026-03-18T11:08:52.111023,Alpha-Cash-CIC-Model,v1,"{""max_age_all_contracts_snapshot"": 5320.0, ""ra...",Cash September 25 Models,a65c2c8c-d836-4098-af06-fa2865099992,2026-03-18T11:08:52.111023,{},Trench 1,ios,Dev_Test,2025-02-07


In [58]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,38878,2025-02-01,2025-09-23
1,Dev_Train,37838,2024-09-01,2025-01-31


In [59]:
# Upload to BigQuery
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=147d7204-751e-4566-a133-e087a6d8c2a4>

###### Trench 2

In [60]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='Alpha-Cash-CIC-Model'
and modelVersionId = 'v1'
and trenchCategory = 'Trench 2'
;
"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=66131339-8d01-450c-9c63-6f3d1b8e34e4>

In [61]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,
r.ca_cic_score,
r.max_age_all_contracts_snapshot,
r.ratio_overdue_contracts_to_granted_contracts,
r.ScoreRange,
r.ln_loan_level_user_type,
r.has_ever_been_overdue,
r.latest_granted_contract_overdue_flag,
r.ratio_closed_over_new_granted_cnt_24M,
r.ratio_risky_contracts_to_granted_contracts,
r.Short_and_Term_Loans_granted_contracts_cnt_24M,
r.flg_zero_non_granted_ever,
r.Personal_Loans_granted_contracts_amt_24M,
r.CreditAvgCreditLimit,
r.flg_zero_granted_ever,
    case when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%os%' then 'ios'
    when lower(lmt.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-01-31' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection  
from worktable_data_analysis.cash_alpha_cic_all_applied_backscored_20240901_20250930 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where trench_category = 'Trench 2'
and r.ca_cic_score is not null 
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-09-24'
;
"""


data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 402609d6-7ddf-4cfb-96dd-bd26b3000490 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (47776, 19)


In [62]:
feature_column = ['max_age_all_contracts_snapshot',
       'ratio_overdue_contracts_to_granted_contracts', 'ScoreRange',
       'ln_loan_level_user_type', 'has_ever_been_overdue',
       'latest_granted_contract_overdue_flag',
       'ratio_closed_over_new_granted_cnt_24M',
       'ratio_risky_contracts_to_granted_contracts',
       'Short_and_Term_Loans_granted_contracts_cnt_24M',
       'flg_zero_non_granted_ever', 'Personal_Loans_granted_contracts_amt_24M',
       'CreditAvgCreditLimit', 'flg_zero_granted_ever', 'ca_cic_score']

dfd = transform_data(data, feature_column, a='ca_cic_score', modelDisplayName='Alpha-Cash-CIC-Model', tc='Trench 2', subscription_name = 'Cash September 25 Models') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,2341127,847b664d-56b7-4f35-a828-aa0ad6af6518,268a5c5c-afb0-4338-95dc-025766ed0303,0.428450,2026-03-18T11:10:56.256809,2026-03-18T11:10:56.256809,Alpha-Cash-CIC-Model,v1,"{""max_age_all_contracts_snapshot"": 1837.0, ""ra...",Cash September 25 Models,825a723e-2fe8-4fbe-bf6b-3f5dec45a6ae,2026-03-18T11:10:56.256809,{},Trench 2,android,Dev_Test,2025-03-04
1,2143377,9400dc77-d9f2-476f-a496-faa4be7f8c4e,5f9e0c89-ab46-42b0-ac53-4a32cace75ed,0.587711,2026-03-18T11:10:56.256809,2026-03-18T11:10:56.256809,Alpha-Cash-CIC-Model,v1,"{""max_age_all_contracts_snapshot"": 876.0, ""rat...",Cash September 25 Models,b6f4fd5d-7c11-44eb-b0e5-d1e9fdbfb7de,2026-03-18T11:10:56.256809,{},Trench 2,android,Dev_Test,2025-09-20
2,2958049,5280e7cf-3503-42c1-aa21-2e91a7909f32,aba3caf0-10b4-487a-b951-6e2e319738da,0.527674,2026-03-18T11:10:56.256809,2026-03-18T11:10:56.256809,Alpha-Cash-CIC-Model,v1,"{""max_age_all_contracts_snapshot"": 432.0, ""rat...",Cash September 25 Models,9c107a8b-4454-4202-9bd3-070016738757,2026-03-18T11:10:56.256809,{},Trench 2,ios,Dev_Test,2025-08-07
3,2234293,edfa1d5d-ba3f-4c48-b8ef-954645abb9fd,5f3d11e9-f5e6-49e1-a821-d294882f2005,0.528337,2026-03-18T11:10:56.256809,2026-03-18T11:10:56.256809,Alpha-Cash-CIC-Model,v1,"{""max_age_all_contracts_snapshot"": 682.0, ""rat...",Cash September 25 Models,3e044ee0-71fe-4f2d-b767-74c2fcb71a42,2026-03-18T11:10:56.256809,{},Trench 2,ios,Dev_Train,2024-11-04
4,3033767,5d454dba-b166-4b04-93db-4fe0ce8283cc,3e9e2bbf-3888-4c23-ae63-d5ee3a84be14,0.684411,2026-03-18T11:10:56.256809,2026-03-18T11:10:56.256809,Alpha-Cash-CIC-Model,v1,"{""max_age_all_contracts_snapshot"": 627.0, ""rat...",Cash September 25 Models,5c31c95c-9166-4fe6-a271-b6a4a11fa958,2026-03-18T11:10:56.256809,{},Trench 2,ios,Dev_Test,2025-06-28


In [63]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,29036,2025-02-01,2025-09-23
1,Dev_Train,18740,2024-09-01,2025-01-31


In [64]:
# Upload to BigQuery
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=c1368b5a-4e9a-4870-9581-1ed51ec7922c>

###### Trench 3

In [65]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='Alpha-Cash-CIC-Model'
and modelVersionId = 'v1'
and trenchCategory = 'Trench 3'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=23e62451-f752-4975-9c9b-00f10e4e7623>

In [66]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,
r.ca_cic_score,
r.max_age_all_contracts_snapshot,
r.ratio_overdue_contracts_to_granted_contracts,
r.ScoreRange,
r.ln_loan_level_user_type,
r.has_ever_been_overdue,
r.latest_granted_contract_overdue_flag,
r.ratio_closed_over_new_granted_cnt_24M,
r.ratio_risky_contracts_to_granted_contracts,
r.Short_and_Term_Loans_granted_contracts_cnt_24M,
r.flg_zero_non_granted_ever,
r.Personal_Loans_granted_contracts_amt_24M,
r.CreditAvgCreditLimit,
r.flg_zero_granted_ever,
    case when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%os%' then 'ios'
    when lower(lmt.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-01-31' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
from worktable_data_analysis.cash_alpha_cic_all_applied_backscored_20240901_20250930 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where trench_category = 'Trench 3'
and r.ca_cic_score is not null 
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-09-24'
;
"""


data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 43e8f1c3-c3f5-4f64-a261-6b504654a257 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (18967, 19)


In [67]:
feature_column = ['max_age_all_contracts_snapshot',
       'ratio_overdue_contracts_to_granted_contracts', 'ScoreRange',
       'ln_loan_level_user_type', 'has_ever_been_overdue',
       'latest_granted_contract_overdue_flag',
       'ratio_closed_over_new_granted_cnt_24M',
       'ratio_risky_contracts_to_granted_contracts',
       'Short_and_Term_Loans_granted_contracts_cnt_24M',
       'flg_zero_non_granted_ever', 'Personal_Loans_granted_contracts_amt_24M',
       'CreditAvgCreditLimit', 'flg_zero_granted_ever', 'ca_cic_score']

dfd = transform_data(data, feature_column, a='ca_cic_score', modelDisplayName='Alpha-Cash-CIC-Model', tc='Trench 3', subscription_name = 'Cash September 25 Models') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,2234122,d4378f76-5abf-4487-afa9-e4d73d6e6159,f18fe6b4-3231-4bd9-b613-c415bb9475a6,0.337786,2026-03-18T11:13:58.604190,2026-03-18T11:13:58.604190,Alpha-Cash-CIC-Model,v1,"{""max_age_all_contracts_snapshot"": 2238.0, ""ra...",Cash September 25 Models,f3984a43-944d-4f16-abd3-d6a192fdbd7e,2026-03-18T11:13:58.604190,{},Trench 3,ios,Dev_Train,2025-01-16
1,2394064,695fc84a-3f73-4f1a-9384-6379f5ff0dc3,3ecb6522-700f-48e0-b236-b9109d099fe6,0.478663,2026-03-18T11:13:58.604721,2026-03-18T11:13:58.604721,Alpha-Cash-CIC-Model,v1,"{""max_age_all_contracts_snapshot"": 913.0, ""rat...",Cash September 25 Models,52a1d827-a621-47f9-a4d1-6066610a0b41,2026-03-18T11:13:58.604721,{},Trench 3,android,Dev_Test,2025-08-29
2,2185953,8dedee56-da36-4801-b260-a10b2ea07c53,9c3df741-a40a-42ee-b070-472b391da691,0.416669,2026-03-18T11:13:58.604721,2026-03-18T11:13:58.604721,Alpha-Cash-CIC-Model,v1,"{""max_age_all_contracts_snapshot"": 718.0, ""rat...",Cash September 25 Models,93139886-2243-4a29-8235-e6aec671fa19,2026-03-18T11:13:58.604721,{},Trench 3,ios,Dev_Test,2025-09-02
3,3485245,9e78d106-6970-4e5b-8ce0-1fd7e3c7c59e,26cfee65-8af7-49d6-977c-8639368bc55b,0.400831,2026-03-18T11:13:58.604721,2026-03-18T11:13:58.604721,Alpha-Cash-CIC-Model,v1,"{""max_age_all_contracts_snapshot"": 73.0, ""rati...",Cash September 25 Models,6b277514-aa5e-43d1-bff7-89d0fb779b33,2026-03-18T11:13:58.604721,{},Trench 3,android,Dev_Test,2025-07-04
4,2433781,5017af7f-652e-4118-b1d3-c366ecc7ce05,4c299043-25fb-41b8-91b6-1aea2a9a8db3,0.448235,2026-03-18T11:13:58.605257,2026-03-18T11:13:58.605257,Alpha-Cash-CIC-Model,v1,"{""max_age_all_contracts_snapshot"": 2210.0, ""ra...",Cash September 25 Models,6dd8169f-f099-4e30-be7d-312ba911f736,2026-03-18T11:13:58.605257,{},Trench 3,ios,Dev_Train,2025-01-05


In [68]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,12909,2025-02-01,2025-09-23
1,Dev_Train,6058,2024-09-01,2025-01-31


In [69]:
# Upload to BigQuery
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=56c55edc-9d1c-4bdb-ae36-807f18601a26>

##### Beta-Cash-Demo-Model

In [70]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='Beta-Cash-Demo-Model'
and modelVersionId = 'v1'
and trenchCategory = 'Trench 1'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=61eb8966-1e72-4d1f-83fd-1d9c92ad6cee>

###### Trench 1

In [71]:
sq = """ 
select 
r.customerId customer_id,
r.digitalLoanAccountId,
r.c_demo_score Beta_Cash_Demo_Score,
r.ln_vas_opted_flag, 
r.ln_self_dec_income, 
r.ln_age,
r.ln_source_funds_new_bin, 
r.ln_loan_level_user_type,
r.ln_industry_new_cat_bin, 
r.ln_marital_status,
r.ln_doc_type_rolled, 
r.ln_education_level,
r.ln_ref2_type, 
r.ln_email_primary_domain, 
r.ln_province_bin,
    case when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%os%' then 'ios'
    when lower(lmt.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-10-01' and '2025-01-31' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-10-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
from worktable_data_analysis.cash_beta_demo_all_applied_backscored_20241001_20250930 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where trench_category = 'Trench 1'
and r.c_demo_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-09-24'
;

"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 09a0484b-4279-4143-bdf3-c4afc1e1eceb successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (318201, 18)


In [72]:
feature_column = ['ln_vas_opted_flag',
       'ln_self_dec_income', 'ln_age',
       'ln_source_funds_new_bin', 'ln_loan_level_user_type',
       'ln_industry_new_cat_bin',
       'ln_marital_status',
       'ln_doc_type_rolled',
       'ln_education_level',
       'ln_ref2_type', 'ln_email_primary_domain',
       'ln_province_bin','Beta_Cash_Demo_Score']

dfd = transform_data(data, feature_column, a='Beta_Cash_Demo_Score', modelDisplayName='Beta-Cash-Demo-Model', tc='Trench 1', subscription_name = 'Cash September 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

The shape of the transformed dataframe is:	 (318201, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3508502,52d50dc9-e007-44c7-9db9-a21716886416,8a8466e2-535b-433b-a398-709a8674a9a4,0.381712,2026-03-18T11:17:52.414934,2026-03-18T11:17:52.414934,Beta-Cash-Demo-Model,v1,"{""ln_vas_opted_flag"": ""0"", ""ln_self_dec_income...",Cash September 25 Models,20012f6a-52e5-42a4-b61f-41114fa2e40a,2026-03-18T11:17:52.414934,{},Trench 1,android,Dev_Test,2025-06-20
1,2912213,035edc0e-318b-486d-89f7-6b5d5b53bbe3,5b5fc759-7405-4edc-b82c-aab8ca7c29fa,0.612205,2026-03-18T11:17:52.414934,2026-03-18T11:17:52.414934,Beta-Cash-Demo-Model,v1,"{""ln_vas_opted_flag"": ""1"", ""ln_self_dec_income...",Cash September 25 Models,eee7c194-db10-40fd-a7c2-ac1cb32c1c74,2026-03-18T11:17:52.414934,{},Trench 1,ios,Dev_Train,2024-10-06
2,3468466,8495b960-c3b4-454b-becf-03e84b522a2f,36e61ee8-4790-4168-9868-bcd6d8d16b0f,0.440595,2026-03-18T11:17:52.414934,2026-03-18T11:17:52.414934,Beta-Cash-Demo-Model,v1,"{""ln_vas_opted_flag"": ""0"", ""ln_self_dec_income...",Cash September 25 Models,efb3df2b-a7e6-4e4f-952a-8a61c0929561,2026-03-18T11:17:52.414934,{},Trench 1,android,Dev_Test,2025-05-30
3,3060477,eb240e33-eb29-4bf2-867b-5f70975a785f,722d7326-7494-4ac6-bc72-0dcebf896edd,0.549232,2026-03-18T11:17:52.414934,2026-03-18T11:17:52.414934,Beta-Cash-Demo-Model,v1,"{""ln_vas_opted_flag"": ""1"", ""ln_self_dec_income...",Cash September 25 Models,b12d34dd-390b-47c7-8e64-1a55122a1ba1,2026-03-18T11:17:52.414934,{},Trench 1,android,Dev_Train,2024-11-28
4,3575791,e8a89144-835b-4afa-ae36-590f09f4fc61,05bb9743-160e-494a-a3ad-4f8c88740b69,0.578488,2026-03-18T11:17:52.415937,2026-03-18T11:17:52.415937,Beta-Cash-Demo-Model,v1,"{""ln_vas_opted_flag"": ""1"", ""ln_self_dec_income...",Cash September 25 Models,a974e3ab-1d0e-4276-bfa2-acc55dcb5d2c,2026-03-18T11:17:52.415937,{},Trench 1,android,Dev_Test,2025-07-26


In [73]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,191471,2025-02-01,2025-09-23
1,Dev_Train,126730,2024-10-01,2025-01-31


In [74]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=b18294c0-2573-4799-bf77-2facc771f3c0>

###### Trench 2

In [75]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='Beta-Cash-Demo-Model'
and modelVersionId = 'v1'
and trenchCategory = 'Trench 2'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=09b779c2-d3a5-4310-b01e-5c7680c8abb1>

In [76]:
sq = """ 
select 
r.customerId customer_id,
r.digitalLoanAccountId,
r.c_demo_score Beta_Cash_Demo_Score,
r.ln_vas_opted_flag, 
r.ln_self_dec_income, 
r.ln_age,
r.ln_source_funds_new_bin, 
r.ln_loan_level_user_type,
r.ln_industry_new_cat_bin, 
r.ln_marital_status,
r.ln_doc_type_rolled, 
r.ln_education_level,
r.ln_ref2_type, 
r.ln_email_primary_domain, 
r.ln_province_bin,
    case when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%os%' then 'ios'
    when lower(lmt.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-10-01' and '2025-01-31' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-10-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
from worktable_data_analysis.cash_beta_demo_all_applied_backscored_20241001_20250930 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where trench_category = 'Trench 2'
and r.c_demo_score is not null 
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-09-24'
;

"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 076a4366-b275-4e3b-bda4-7bca35929ee2 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (122498, 18)


In [77]:
feature_column = ['ln_vas_opted_flag',
       'ln_self_dec_income', 'ln_age',
       'ln_source_funds_new_bin', 'ln_loan_level_user_type',
       'ln_industry_new_cat_bin',
       'ln_marital_status',
       'ln_doc_type_rolled',
       'ln_education_level',
       'ln_ref2_type', 'ln_email_primary_domain',
       'ln_province_bin','Beta_Cash_Demo_Score']

dfd = transform_data(data, feature_column, a='Beta_Cash_Demo_Score', modelDisplayName='Beta-Cash-Demo-Model', tc='Trench 2', subscription_name = 'Cash September 25 Models') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,1788163,6ec1952b-1124-49ef-9f49-ad8a4a82c2bf,05bff0b5-ffcd-48d5-9993-15622cf31009,0.459938,2026-03-18T11:30:43.719651,2026-03-18T11:30:43.719651,Beta-Cash-Demo-Model,v1,"{""ln_vas_opted_flag"": ""0"", ""ln_self_dec_income...",Cash September 25 Models,0073fa5b-39d4-4519-aef2-cd28966298ab,2026-03-18T11:30:43.719651,{},Trench 2,ios,Dev_Test,2025-03-01
1,3297258,e596156d-5285-4b58-8cbd-5369c348c307,a2d9de3c-f6bb-4afb-ad51-86b7f730ce84,0.486459,2026-03-18T11:30:43.720182,2026-03-18T11:30:43.720182,Beta-Cash-Demo-Model,v1,"{""ln_vas_opted_flag"": ""1"", ""ln_self_dec_income...",Cash September 25 Models,8c0959ff-35fa-46a9-ab35-de4289dab665,2026-03-18T11:30:43.720182,{},Trench 2,ios,Dev_Test,2025-06-05
2,2821792,6a0fd9bb-ae09-4f52-8248-940f8e2932f5,0d5a93b4-32de-4f72-8eae-40907ef765c3,0.458312,2026-03-18T11:30:43.720182,2026-03-18T11:30:43.720182,Beta-Cash-Demo-Model,v1,"{""ln_vas_opted_flag"": ""1"", ""ln_self_dec_income...",Cash September 25 Models,5daad47c-fb47-46c9-93cb-5f42b85c32f5,2026-03-18T11:30:43.720182,{},Trench 2,ios,Dev_Train,2024-11-06
3,2880912,c609a574-de41-4679-9a9c-e4c0d2920eaf,39f1a41f-2634-4b14-a443-0820fe5537c5,0.463824,2026-03-18T11:30:43.720182,2026-03-18T11:30:43.720182,Beta-Cash-Demo-Model,v1,"{""ln_vas_opted_flag"": ""1"", ""ln_self_dec_income...",Cash September 25 Models,40a51712-6ce3-43a5-8d97-792cf07c8ae1,2026-03-18T11:30:43.720182,{},Trench 2,ios,Dev_Test,2025-03-12
4,2452791,40dbf1f6-3582-40ea-9665-6796b4c08253,5df4a6f5-f0c0-41db-9ade-d7d55bc4c0cb,0.532592,2026-03-18T11:30:43.720182,2026-03-18T11:30:43.720182,Beta-Cash-Demo-Model,v1,"{""ln_vas_opted_flag"": ""1"", ""ln_self_dec_income...",Cash September 25 Models,a47a0c44-0df9-49c2-9bcd-1ae65f61ca35,2026-03-18T11:30:43.720182,{},Trench 2,android,Dev_Train,2024-10-17


In [78]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,84722,2025-02-01,2025-09-23
1,Dev_Train,37776,2024-10-01,2025-01-31


In [79]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=06097096-ff49-4fba-a71f-0107ca9e2aae>

###### Trench 3

In [80]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='Beta-Cash-Demo-Model'
and modelVersionId = 'v1'
and trenchCategory = 'Trench 3'
;
"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=a37010fa-9fa7-4394-ae15-933ce66afa2d>

In [81]:
sq = """ 
select 
r.customerId customer_id,
r.digitalLoanAccountId,
r.c_demo_score Beta_Cash_Demo_Score,
r.ln_vas_opted_flag, 
r.ln_self_dec_income, 
r.ln_age,
r.ln_source_funds_new_bin, 
r.ln_loan_level_user_type,
r.ln_industry_new_cat_bin, 
r.ln_marital_status,
r.ln_doc_type_rolled, 
r.ln_education_level,
r.ln_ref2_type, 
r.ln_email_primary_domain, 
r.ln_province_bin,
    case when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%os%' then 'ios'
    when lower(lmt.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-10-01' and '2025-01-31' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-10-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
from worktable_data_analysis.cash_beta_demo_all_applied_backscored_20241001_20250930 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where trench_category = 'Trench 3'
and r.c_demo_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-09-24'
;

"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 613155d5-eecb-4e06-a06c-207a2699277d successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (41635, 18)


In [82]:
feature_column = ['ln_vas_opted_flag',
       'ln_self_dec_income', 'ln_age',
       'ln_source_funds_new_bin', 'ln_loan_level_user_type',
       'ln_industry_new_cat_bin',
       'ln_marital_status',
       'ln_doc_type_rolled',
       'ln_education_level',
       'ln_ref2_type', 'ln_email_primary_domain',
       'ln_province_bin','Beta_Cash_Demo_Score']

dfd = transform_data(data, feature_column, a='Beta_Cash_Demo_Score', modelDisplayName='Beta-Cash-Demo-Model', tc='Trench 3', subscription_name = 'Cash September 25 Models') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,2473239,9dae8ba4-987d-4659-876b-0172a119bbce,e8827325-eeb6-4c19-87dc-b6bdaf370ae9,0.362167,2026-03-18T11:31:49.007461,2026-03-18T11:31:49.007461,Beta-Cash-Demo-Model,v1,"{""ln_vas_opted_flag"": ""0"", ""ln_self_dec_income...",Cash September 25 Models,a30c76aa-6559-45ec-a6c1-8554ff2a99f3,2026-03-18T11:31:49.007461,{},Trench 3,android,Dev_Test,2025-08-09
1,2753487,97ddcdef-648b-4263-935b-e80a68416c6d,4b5a114c-eb50-480a-854e-8e3a7fef2972,0.456890,2026-03-18T11:31:49.007461,2026-03-18T11:31:49.007461,Beta-Cash-Demo-Model,v1,"{""ln_vas_opted_flag"": ""1"", ""ln_self_dec_income...",Cash September 25 Models,4897d5a1-a4dd-4faf-96d8-774c4501d801,2026-03-18T11:31:49.007461,{},Trench 3,android,Dev_Test,2025-02-07
2,2634965,bf843a56-8ef6-4883-bdf3-cd78dcd95f7b,e87ccf46-f7ec-4152-8717-fe705e604374,0.369777,2026-03-18T11:31:49.007461,2026-03-18T11:31:49.007461,Beta-Cash-Demo-Model,v1,"{""ln_vas_opted_flag"": ""0"", ""ln_self_dec_income...",Cash September 25 Models,ec3e3f19-cc54-4239-889d-bb581af57a1e,2026-03-18T11:31:49.007461,{},Trench 3,android,Dev_Test,2025-09-13
3,2574620,0cd4383f-8637-4ab8-a184-41659f6c180b,169cba1d-b5ed-4c2d-9eea-43008245c387,0.457273,2026-03-18T11:31:49.007461,2026-03-18T11:31:49.007461,Beta-Cash-Demo-Model,v1,"{""ln_vas_opted_flag"": ""1"", ""ln_self_dec_income...",Cash September 25 Models,9725ebcc-48da-4c3c-87af-ce971f0338ad,2026-03-18T11:31:49.007461,{},Trench 3,android,Dev_Test,2025-04-05
4,3173464,9d1f2056-e5ba-449c-a3c6-712427b971ef,dd52cc71-6f40-4777-981e-335ef25c8e77,0.657089,2026-03-18T11:31:49.007461,2026-03-18T11:31:49.007461,Beta-Cash-Demo-Model,v1,"{""ln_vas_opted_flag"": ""1"", ""ln_self_dec_income...",Cash September 25 Models,26553a94-f98b-4d81-84ee-23a1898b525a,2026-03-18T11:31:49.007461,{},Trench 3,android,Dev_Test,2025-06-30


In [83]:

result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,30895,2025-02-01,2025-09-23
1,Dev_Train,10740,2024-10-01,2025-01-31


In [84]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=8a7363b0-7810-44bf-bc64-8b9637e1e6b7>

##### Beta-Cash-Stack-Model

In [85]:
sq = """select digitalLoanAccountId, modelDisplayName, modelVersionId from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='Beta-Cash-Stack-Model'
and modelVersionId = 'v1'
and trenchCategory = 'Trench 1'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=c8e26ff1-39d1-4d1c-941b-464fe9f4c79a>

In [86]:
sq = """ 
select r.customer_id,
r.digitalLoanAccountId,
r.demo_score,
r.apps_score,
r.credo_score,
r.stack_score Beta_cash_stack_score,
r.stack_score_norm,
    case when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%os%' then 'ios'
    when lower(lmt.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-10-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-10-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
 from worktable_data_analysis.cash_beta_trench1_applied_loans_backscored_20241001_20250930 r
 left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
 where r.stack_score is not null 
 and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-09-24'
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 6a9e6804-9773-448c-a15e-a0a96b23117b successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (318108, 10)


In [87]:
feature_column = ['demo_score',
       'apps_score', 'credo_score',
       'stack_score', 'stack_score_norm']

dfd = transform_data(data, feature_column, a='Beta_cash_stack_score', modelDisplayName='Beta-Cash-Stack-Model', tc='Trench 1', subscription_name = 'Cash September 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

The shape of the transformed dataframe is:	 (318108, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,2924472,a7c9887a-4ad7-40a1-b5da-0274defdf1c2,33bd4626-2f06-43d3-9c9c-34badf492fda,0.593867,2026-03-18T11:34:19.943809,2026-03-18T11:34:19.943809,Beta-Cash-Stack-Model,v1,"{""demo_score"": 0.5305738053152241, ""apps_score...",Cash September 25 Models,063b07e1-784d-45d3-8258-f6215f1024d3,2026-03-18T11:34:19.943809,{},Trench 1,android,Dev_Train,2024-10-09
1,3597789,744ee17d-1d39-46c7-a317-d9791a67e43f,cd4d41e6-3abf-41ed-ac06-78f3ca53b471,0.475068,2026-03-18T11:34:19.944809,2026-03-18T11:34:19.944809,Beta-Cash-Stack-Model,v1,"{""demo_score"": 0.4638063885143461, ""apps_score...",Cash September 25 Models,f87f65a9-130a-4de1-a291-d6ed74e32fa5,2026-03-18T11:34:19.944809,{},Trench 1,android,Dev_Test,2025-08-04
2,3072816,6478827f-c4c6-42a8-b405-52e27f67a72d,c1e7b03c-53f5-4d4e-9131-b6fdfd1502d0,0.551120,2026-03-18T11:34:19.944809,2026-03-18T11:34:19.944809,Beta-Cash-Stack-Model,v1,"{""demo_score"": 0.4205301817149657, ""apps_score...",Cash September 25 Models,55655c1b-67b4-4942-90b4-a768e79c29e5,2026-03-18T11:34:19.944809,{},Trench 1,android,Dev_Train,2024-12-02
3,3214924,3221f7a1-b8a7-4536-a543-940e54b4a51e,df89becc-9b71-4799-978d-3b2f3982c8d5,0.967717,2026-03-18T11:34:19.944809,2026-03-18T11:34:19.944809,Beta-Cash-Stack-Model,v1,"{""demo_score"": 0.5653547816283172, ""apps_score...",Cash September 25 Models,6b101f63-4a96-447a-9a5d-d29ca2fda157,2026-03-18T11:34:19.944809,{},Trench 1,android,Dev_Train,2025-01-25
4,3606043,dd17bac5-6b84-407c-a986-4881237a1f32,575be3dc-e4d4-46c1-a947-4f81aa9bbea9,0.259660,2026-03-18T11:34:19.944809,2026-03-18T11:34:19.944809,Beta-Cash-Stack-Model,v1,"{""demo_score"": 0.4979835042473126, ""apps_score...",Cash September 25 Models,9809606a-553b-4283-b2a2-84c19292c3c5,2026-03-18T11:34:19.944809,{},Trench 1,android,Dev_Test,2025-08-07


In [88]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,162675,2025-03-01,2025-09-23
1,Dev_Train,155433,2024-10-01,2025-02-28


In [89]:
# Upload to BigQuery
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=72f11453-f56d-45ef-92b0-ba46e6d971b3>

###### Trench 2

In [90]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='Beta-Cash-Stack-Model'
and modelVersionId = 'v1'
and trenchCategory = 'Trench 2'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=081a0aeb-d065-4f5e-922d-c44e508eea58>

In [91]:
sq = """ 
select r.customer_id,
r.digitalLoanAccountId,
r.demo_score,
r.apps_score,
r.credo_score,
r.stack_score Beta_cash_stack_score,
r.stack_score_norm,
    case when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%os%' then 'ios'
    when lower(lmt.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-10-01' and '2025-02-28' then 'Dev_Train'
        when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-10-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
 from worktable_data_analysis.cash_beta_trench2_applied_loans_backscored_20241001_20250930 r
 left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
 where r.stack_score is not null
 and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-09-24'
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID a7d4e5d5-1052-4333-8670-cf53eab24364 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (119665, 10)


In [92]:
feature_column = ['demo_score',
       'apps_score', 'credo_score',
       'stack_score', 'stack_score_norm']

dfd = transform_data(data, feature_column, a='Beta_cash_stack_score', modelDisplayName='Beta-Cash-Stack-Model', tc='Trench 2', subscription_name = 'Cash September 25 Models') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,2712364,5034c719-03f5-4406-a317-278a1219e060,f15a932d-30c0-4bf6-a310-e8b8f468bcda,0.593036,2026-03-18T11:39:48.472678,2026-03-18T11:39:48.472678,Beta-Cash-Stack-Model,v1,"{""demo_score"": 0.5625537096786807, ""apps_score...",Cash September 25 Models,da94f762-df91-4340-b550-f2048508fb41,2026-03-18T11:39:48.472678,{},Trench 2,android,Dev_Test,2025-06-28
1,2972925,07b71f05-a801-42be-9525-10f558571919,56fb0a81-82bc-4dd7-8820-d9976a2f4e4c,0.792588,2026-03-18T11:39:48.473236,2026-03-18T11:39:48.473236,Beta-Cash-Stack-Model,v1,"{""demo_score"": 0.5346467326626658, ""apps_score...",Cash September 25 Models,d0f9c04b-70f1-4c10-bb27-f6b8fb780e3c,2026-03-18T11:39:48.473236,{},Trench 2,android,Dev_Test,2025-08-03
2,2465436,8e0b13e7-600e-4cf4-94d8-04cd65d0373f,881a60cd-4466-4558-bff1-1dd63a2e9116,0.896601,2026-03-18T11:39:48.473236,2026-03-18T11:39:48.473236,Beta-Cash-Stack-Model,v1,"{""demo_score"": 0.5822205408045505, ""apps_score...",Cash September 25 Models,7176d14b-f1d4-43b4-97d1-2c8f767aba6f,2026-03-18T11:39:48.473236,{},Trench 2,android,Dev_Train,2025-02-28
3,1001008,e9f455f9-6a40-46d8-9f90-2c330c1072de,791b1e82-8f1e-4386-8f8f-dcad0a69ea7c,0.315956,2026-03-18T11:39:48.473236,2026-03-18T11:39:48.473236,Beta-Cash-Stack-Model,v1,"{""demo_score"": 0.4125929031032222, ""apps_score...",Cash September 25 Models,10c2cee1-e971-4c77-9987-b3d49daa1cc3,2026-03-18T11:39:48.473236,{},Trench 2,android,Dev_Train,2024-12-15
4,2023014,e1f7ef5e-4a11-4b44-9b30-610c6be30ade,308da34c-8276-4bab-b7af-32cc7e8059fb,0.575568,2026-03-18T11:39:48.473776,2026-03-18T11:39:48.473776,Beta-Cash-Stack-Model,v1,"{""demo_score"": 0.5796784046869733, ""apps_score...",Cash September 25 Models,32cfa185-1948-4155-a241-a9a45a2dbc24,2026-03-18T11:39:48.473776,{},Trench 2,android,Dev_Test,2025-05-06


In [93]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,72337,2025-03-01,2025-09-23
1,Dev_Train,47328,2024-10-01,2025-02-28


In [94]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=6084e354-3c6e-41d4-8075-4ed78971430d>

###### Trench 3

In [95]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='Beta-Cash-Stack-Model'
and modelVersionId = 'v1'
and trenchCategory = 'Trench 3'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=e251c104-5b90-45b0-98ce-1e95f3450357>

In [96]:
sq = """ 
select r.customer_id,
r.digitalLoanAccountId,
r.demo_score,
r.apps_score,
r.credo_score,
r.stack_score Beta_cash_stack_score,
r.stack_score_norm,
    case when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%os%' then 'ios'
    when lower(lmt.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-10-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-10-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
 from worktable_data_analysis.cash_beta_trench3_applied_loans_backscored_20241001_20250930 r
 left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
 where r.stack_score is not null 
 and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-09-24'

"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


Job ID 35a58cd0-cf47-4fdf-8793-29fc64f33a37 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (41609, 10)


In [97]:
feature_column = ['demo_score',
       'apps_score', 'credo_score',
       'stack_score', 'stack_score_norm']

dfd = transform_data(data, feature_column, a='Beta_cash_stack_score', modelDisplayName='Beta-Cash-Stack-Model', tc='Trench 3', subscription_name = 'Cash September 25 Models') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,2896633,65b5a55d-bbb0-47f7-8004-309c7c1a4b32,9e9a0392-3128-44cb-9bab-4c813ff68f54,0.578827,2026-03-18T12:56:31.791466,2026-03-18T12:56:31.791466,Beta-Cash-Stack-Model,v1,"{""demo_score"": 0.3937891139097688, ""apps_score...",Cash September 25 Models,9e4d50cd-08cb-49db-ab9f-41f3eb7c6b67,2026-03-18T12:56:31.791466,{},Trench 3,android,Dev_Test,2025-08-26
1,2180586,6c0cabb9-2b5f-4969-a2c1-8b5dc5dcb9ee,644e7994-9efd-4513-97fe-e1b62f40fa94,0.292360,2026-03-18T12:56:31.791466,2026-03-18T12:56:31.791466,Beta-Cash-Stack-Model,v1,"{""demo_score"": 0.35974015796720676, ""apps_scor...",Cash September 25 Models,eaa867ec-c85a-4e59-9365-ab4d13bbddd8,2026-03-18T12:56:31.791466,{},Trench 3,android,Dev_Test,2025-06-20
2,3284192,3863da07-b08c-4fc8-baf4-3de96e3e6b12,d7f90fe8-366f-45cb-8808-6b08132986d6,0.284916,2026-03-18T12:56:31.791466,2026-03-18T12:56:31.791466,Beta-Cash-Stack-Model,v1,"{""demo_score"": 0.36442355147834665, ""apps_scor...",Cash September 25 Models,4b958b32-67a6-4523-8475-fc3b01e9b33d,2026-03-18T12:56:31.791466,{},Trench 3,android,Dev_Test,2025-09-07
3,2947101,8b52aa37-bf78-40b7-b6cd-b570f76290fc,fc46eace-80bd-4999-abd4-e78f5c9e4626,0.272992,2026-03-18T12:56:31.792465,2026-03-18T12:56:31.792465,Beta-Cash-Stack-Model,v1,"{""demo_score"": 0.3739967373920038, ""apps_score...",Cash September 25 Models,a29e097c-2bf2-40fc-8f04-ef3b147efeba,2026-03-18T12:56:31.792465,{},Trench 3,android,Dev_Test,2025-07-14
4,1393335,58f7d455-2a26-42cc-9676-29c1e1accb66,e642cdd8-5ca7-4c6b-a7f9-5a16d48658cf,0.663791,2026-03-18T12:56:31.792465,2026-03-18T12:56:31.792465,Beta-Cash-Stack-Model,v1,"{""demo_score"": 0.5162181959233343, ""apps_score...",Cash September 25 Models,f09a7a38-94a8-4f76-8b6d-ae4751484ddf,2026-03-18T12:56:31.792465,{},Trench 3,android,Dev_Test,2025-09-04


In [98]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,27473,2025-03-01,2025-09-23
1,Dev_Train,14136,2024-10-01,2025-02-28


In [99]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=6d93a636-ea49-4a70-93ad-c5e0a33ec1b2>

#####  Beta Cash AppScore Model

###### Trench 1

In [85]:
## query to add the trench category trench 1 and trench 2

# create or replace table risk_mart.applied_quick_loan_new_applicants_20230101_20250930_app_scored_copy as 
# select * from risk_mart.applied_quick_loan_new_applicants_20230101_20250930_app_scored;



# ALTER TABLE risk_mart.applied_quick_loan_new_applicants_20230101_20250930_app_scored_copy
# ADD COLUMN trench_category STRING;

# MERGE risk_mart.applied_quick_loan_new_applicants_20230101_20250930_app_scored_copy AS target
# USING (
#   SELECT 
#     a.digitalLoanAccountId,
#     CASE 
#       WHEN b.ln_loan_level_user_type = '2_New Applicant'
#            AND DATE_DIFF(DATE(b.ln_appln_submit_datetime), DATE(b.onb_tsa_onboarding_datetime), DAY) <= 30 THEN 'Trench 1'
#       WHEN b.ln_loan_level_user_type = '2_New Applicant'
#            AND DATE_DIFF(DATE(b.ln_appln_submit_datetime), DATE(b.onb_tsa_onboarding_datetime), DAY) > 30 THEN 'Trench 2'
#       WHEN b.ln_loan_level_user_type = '1_Repeat Applicant' THEN 'Trench 3'
#       ELSE NULL
#     END AS trench_category,
#     ROW_NUMBER() OVER (PARTITION BY a.digitalLoanAccountId ORDER BY b.ln_appln_submit_datetime DESC) AS row_num
#   FROM risk_mart.applied_quick_loan_new_applicants_20230101_20250930_app_scored_copy a
#   LEFT JOIN prj-prod-dataplatform.risk_mart.applied_loans_20210701_20250930_trans b
#     ON a.digitalLoanAccountId = b.digitalLoanAccountId
# ) AS source
# ON target.digitalLoanAccountId = source.digitalLoanAccountId
# WHEN MATCHED AND source.row_num = 1 THEN
#   UPDATE SET target.trench_category = source.trench_category;

In [100]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='Beta-Cash-AppScore-Model'
and modelVersionId = 'v1'
and trenchCategory = 'Trench 1'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=8eb35dd9-fad0-4a08-a824-d7811308bf44>

In [101]:
sq = """ 
select 
  r.customerId customer_id,
  r.digitalLoanAccountId,
  r.apps_score beta_cash_app_score,  
  r.cb_score ml_score,
  r.dl_score, 
  r.app_cnt_health_and_fitness_ever app_cnt_health_and_fitness_ever, 
  r.app_cnt_shopping_ever app_cnt_shopping_ever,
  r.app_cnt_crypto_ever app_cnt_crypto_ever, 
  r.app_cnt_driver_ever app_cnt_driver_ever,
  r.app_cnt_payday_180d app_cnt_payday_180d, 
  r.app_cnt_gambling_180d app_cnt_gambling_180d,
  r.app_avg_time_bw_installed_mins_3d app_avg_time_bw_installed_mins_3d,
  r.app_median_time_bw_installed_mins_ever app_median_time_bw_installed_mins_3d,
    case when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%os%' then 'ios'
    when lower(lmt.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
 date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-08-13' and '2025-01-31' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-08-13' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
from risk_mart.applied_quick_loan_new_applicants_20230101_20250930_app_scored_copy r 
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 1'
and r.apps_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-09-24'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 1dab6d83-d5d5-4e25-b545-9cd95f3fd237 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (382443, 16)


For cash app score v1 we have the cb_score and dl_score which we can use risk_mart.applied_quick_loan_new_applicants_20230101_20250930_app_scored_copy. I need to delete the data from the training dataset and then run it again.

In [102]:
data.columns

Index(['customer_id', 'digitalLoanAccountId', 'beta_cash_app_score',
       'ml_score', 'dl_score', 'app_cnt_health_and_fitness_ever',
       'app_cnt_shopping_ever', 'app_cnt_crypto_ever', 'app_cnt_driver_ever',
       'app_cnt_payday_180d', 'app_cnt_gambling_180d',
       'app_avg_time_bw_installed_mins_3d',
       'app_median_time_bw_installed_mins_3d', 'osType', 'application_date',
       'Data_selection'],
      dtype='object')

In [103]:
feature_column = ['beta_cash_app_score', 'ml_score', 'dl_score',
       'app_cnt_health_and_fitness_ever', 'app_cnt_shopping_ever',
       'app_cnt_crypto_ever', 'app_cnt_driver_ever', 'app_cnt_payday_180d',
       'app_cnt_gambling_180d', 'app_avg_time_bw_installed_mins_3d',
       'app_median_time_bw_installed_mins_3d']

dfd = transform_data(data, feature_column, a='beta_cash_app_score', modelDisplayName='Beta-Cash-AppScore-Model', tc='Trench 1', subscription_name = 'Cash September 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.info()

The shape of the transformed dataframe is:	 (382443, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 382443 entries, 0 to 382442
Data columns (total 17 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   customerId            382443 non-null  int64  
 1   digitalLoanAccountId  382443 non-null  object 
 2   crifApplicationId     382443 non-null  object 
 3   prediction            382443 non-null  float64
 4   start_time            382443 non-null  object 
 5   end_time              382443 non-null  object 
 6   modelDisplayName      382443 non-null  object 
 7   modelVersionId        382443 non-null  object 
 8   calcFeature           382443 non-null  object 
 9   subscription_name     382443 non-null  object 
 10  message_id            382443 non-null  object 
 11  publish_time          382443 non-null  object 
 12  attributes            382443 non-null  object 
 13  trenchCategory        382443 non-null  object 


In [104]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,125423,2025-02-01,2025-09-23
1,Dev_Train,141774,2024-08-13,2025-01-31
2,Pre_Train,115246,2024-01-01,2024-08-12


In [105]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=37c207f9-9bc5-4eec-a825-1bae0ba6799c>

###### Trench 2

In [106]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='Beta-Cash-AppScore-Model'
and modelVersionId = 'v1'
and trenchCategory = 'Trench 2'
;"""
client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=28e0e3a5-f31b-424f-8fc7-5f1079eaabb9>

In [107]:
sq = """ 
select 
  r.customerId customer_id,
  r.digitalLoanAccountId,
  r.apps_score beta_cash_app_score,  
  r.cb_score ml_score,
  r.dl_score, 
  r.app_cnt_health_and_fitness_ever app_cnt_health_and_fitness_ever, 
  r.app_cnt_shopping_ever app_cnt_shopping_ever,
  r.app_cnt_crypto_ever app_cnt_crypto_ever, 
  r.app_cnt_driver_ever app_cnt_driver_ever,
  r.app_cnt_payday_180d app_cnt_payday_180d, 
  r.app_cnt_gambling_180d app_cnt_gambling_180d,
  r.app_avg_time_bw_installed_mins_3d app_avg_time_bw_installed_mins_3d,
  r.app_median_time_bw_installed_mins_ever app_median_time_bw_installed_mins_3d,
    case when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%os%' then 'ios'
    when lower(lmt.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
 date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
 case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-08-13' and '2025-01-31' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-08-13' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
from risk_mart.applied_quick_loan_new_applicants_20230101_20250930_app_scored_copy r 
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 2'
and r.apps_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-09-24'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 5224eb9b-3047-4ea0-bbd7-9262adae6a98 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (121596, 16)


In [108]:
feature_column = ['beta_cash_app_score', 'ml_score', 'dl_score',
       'app_cnt_health_and_fitness_ever', 'app_cnt_shopping_ever',
       'app_cnt_crypto_ever', 'app_cnt_driver_ever', 'app_cnt_payday_180d',
       'app_cnt_gambling_180d', 'app_avg_time_bw_installed_mins_3d',
       'app_median_time_bw_installed_mins_3d']

dfd = transform_data(data, feature_column, a='beta_cash_app_score', modelDisplayName='Beta-Cash-AppScore-Model', tc='Trench 2', subscription_name = 'Cash September 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.info()

The shape of the transformed dataframe is:	 (121596, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121596 entries, 0 to 121595
Data columns (total 17 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   customerId            121596 non-null  int64  
 1   digitalLoanAccountId  121596 non-null  object 
 2   crifApplicationId     121596 non-null  object 
 3   prediction            121596 non-null  float64
 4   start_time            121596 non-null  object 
 5   end_time              121596 non-null  object 
 6   modelDisplayName      121596 non-null  object 
 7   modelVersionId        121596 non-null  object 
 8   calcFeature           121596 non-null  object 
 9   subscription_name     121596 non-null  object 
 10  message_id            121596 non-null  object 
 11  publish_time          121596 non-null  object 
 12  attributes            121596 non-null  object 
 13  trenchCategory        121596 non-null  object 


In [109]:
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,2268329,c1a53360-1c0c-41b9-97ac-a73f8141fe9a,ced84585-160c-4326-a38d-f6c62aa10595,0.461094,2026-03-18T13:06:16.278673,2026-03-18T13:06:16.278673,Beta-Cash-AppScore-Model,v1,"{""beta_cash_app_score"": 0.4610943408992045, ""m...",Cash September 25 Models,951d2d31-3e84-4477-ad40-9dd97e8f5ae3,2026-03-18T13:06:16.278673,{},Trench 2,android,Dev_Train,2024-11-28
1,1525900,73f166c0-f676-4b39-93a6-8da0069f1bf8,d03084bc-6668-4807-9e85-899741b12fa1,0.287290,2026-03-18T13:06:16.278673,2026-03-18T13:06:16.278673,Beta-Cash-AppScore-Model,v1,"{""beta_cash_app_score"": 0.28729003972778117, ""...",Cash September 25 Models,5e56646a-6aa7-4cdd-9e61-bc18f2c2cf99,2026-03-18T13:06:16.278673,{},Trench 2,android,Dev_Train,2024-09-21
2,1714930,29f2c53f-ebd1-49c7-8226-0d668546aa1b,1b6bb9be-f310-41b9-a032-25dd53bb6d8d,0.352218,2026-03-18T13:06:16.278673,2026-03-18T13:06:16.278673,Beta-Cash-AppScore-Model,v1,"{""beta_cash_app_score"": 0.35221791426433646, ""...",Cash September 25 Models,d5ab00ff-9060-4ff3-b150-710889981284,2026-03-18T13:06:16.278673,{},Trench 2,android,Dev_Test,2025-03-20
3,2569290,1a38cd66-9f04-4e46-af80-c2d06059e602,c0676e81-aa42-4770-ac62-10966bfb1888,0.383925,2026-03-18T13:06:16.278673,2026-03-18T13:06:16.278673,Beta-Cash-AppScore-Model,v1,"{""beta_cash_app_score"": 0.38392486963033223, ""...",Cash September 25 Models,82e00959-6543-41b5-8bae-f0f8a5f624de,2026-03-18T13:06:16.278673,{},Trench 2,android,Dev_Train,2024-09-17
4,3327140,f9b23b7e-427f-44f2-84c6-bee8fd5ca2c9,2f534b73-d0bb-45b2-998b-f9220d143f25,0.314893,2026-03-18T13:06:16.278673,2026-03-18T13:06:16.278673,Beta-Cash-AppScore-Model,v1,"{""beta_cash_app_score"": 0.31489311360770134, ""...",Cash September 25 Models,d8684362-d9aa-429a-aebb-32762cb873ab,2026-03-18T13:06:16.278673,{},Trench 2,android,Dev_Test,2025-06-26


In [110]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,54667,2025-02-01,2025-09-23
1,Dev_Train,36978,2024-08-13,2025-01-31
2,Pre_Train,29951,2024-01-01,2024-08-12


In [111]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=597b5b53-7217-49a1-b24a-a763b3aa6d10>

###### Trench 3

In [112]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='Beta-Cash-AppScore-Model'
and modelVersionId = 'v1'
and trenchCategory = 'Trench 3'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=e5104cda-9c5e-4269-8fbd-49376eb17a11>

In [113]:
sq = """   
 select 
  r.customerId customer_id, 
  r.digitalLoanAccountId,
  r.apps_score beta_cash_app_score,
  r.cb_score ml_score,
  r.dl_score,
  r.app_cnt_health_and_fitness_ever_binned app_cnt_health_and_fitness_ever,
  r.app_cnt_productivity_ever_binned app_cnt_productivity_ever, 
  r.app_cnt_rated_for_18plus_ever_binned app_cnt_rated_for_18plus_ever,
  r.app_cnt_books_and_reference_ever_binned app_cnt_books_and_reference_ever, 
  r.app_cnt_gaming_180d_binned app_cnt_gaming_180d,
  r.app_cnt_absence_tag_365d_binned app_cnt_absence_tag_365d,
  r.app_last_payday_install_to_apply_days_binned app_last_payday_install_to_apply_days,
  r.app_cnt_absence_tag_365d_binned,
  r.app_cnt_gaming_180d_binned,
  r.app_cnt_productivity_ever_binned,
  r.app_cnt_rated_for_18plus_ever_binned,
  r.app_cnt_health_and_fitness_ever_binned,
  r.app_cnt_books_and_reference_ever_binned,
  r.app_last_payday_install_to_apply_days_binned,
  r.ln_user_type,
  case when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%os%' then 'ios'
    when lower(lmt.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
  date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
 case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-08-13' and '2025-01-31' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-08-13' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
from risk_mart.b_score_model_applied_loans_cash_20240101_20250930_app_scored r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.apps_score is not null 
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-09-24'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 287b9524-4f81-44a6-a9c0-00f40b6bf5c2 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (65874, 23)


In [114]:
feature_column = ['beta_cash_app_score', 'ml_score', 'dl_score',
       'app_cnt_health_and_fitness_ever', 'app_cnt_productivity_ever',
       'app_cnt_rated_for_18plus_ever', 'app_cnt_books_and_reference_ever',
       'app_cnt_gaming_180d', 'app_cnt_absence_tag_365d',
       'app_last_payday_install_to_apply_days',
       'app_cnt_absence_tag_365d_binned', 'app_cnt_gaming_180d_binned',
       'app_cnt_productivity_ever_binned',
       'app_cnt_rated_for_18plus_ever_binned',
       'app_cnt_health_and_fitness_ever_binned',
       'app_cnt_books_and_reference_ever_binned',
       'app_last_payday_install_to_apply_days_binned', 'ln_user_type']

dfd = transform_data(data, feature_column, a='beta_cash_app_score', modelDisplayName='Beta-Cash-AppScore-Model', tc='Trench 3', subscription_name = 'Cash September 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

The shape of the transformed dataframe is:	 (65874, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,2717596,cd4689d8-9bc1-45c7-a1e7-815c843e7fc6,a7083f86-3b00-4634-a0e0-cce691d5a8c2,0.546645,2026-03-18T13:10:37.491465,2026-03-18T13:10:37.491465,Beta-Cash-AppScore-Model,v1,"{""beta_cash_app_score"": 0.5466446185735238, ""m...",Cash September 25 Models,e7363dbf-d9fa-4aaa-8f1d-7f5069cd69ed,2026-03-18T13:10:37.491465,{},Trench 3,android,Dev_Train,2024-12-05
1,1634133,723652a6-8dd0-49b4-8318-2644e9a482ff,3661d128-4db6-47fe-8172-9e5c9d0af88b,0.428434,2026-03-18T13:10:37.491465,2026-03-18T13:10:37.491465,Beta-Cash-AppScore-Model,v1,"{""beta_cash_app_score"": 0.4284338043684377, ""m...",Cash September 25 Models,0ea8a5b7-1198-4a49-ae47-d59948706145,2026-03-18T13:10:37.491465,{},Trench 3,android,Pre_Train,2023-07-08
2,1294761,71ea2284-0443-4cea-b85a-92cfa5371482,8101c297-7b63-4484-8e45-d1efa819e558,0.448839,2026-03-18T13:10:37.492021,2026-03-18T13:10:37.492021,Beta-Cash-AppScore-Model,v1,"{""beta_cash_app_score"": 0.4488393469897295, ""m...",Cash September 25 Models,e5b2f25a-ae7f-4dea-9e19-8af78567c70d,2026-03-18T13:10:37.492021,{},Trench 3,android,Pre_Train,2024-01-23
3,2147734,1d355d65-e603-40dc-9e5a-b9ac8f7c2753,6b7c0bd9-d58a-4855-8e51-764f2ca1e15c,0.516770,2026-03-18T13:10:37.492021,2026-03-18T13:10:37.492021,Beta-Cash-AppScore-Model,v1,"{""beta_cash_app_score"": 0.5167703599502085, ""m...",Cash September 25 Models,0775e07a-f8c2-4f23-ab88-279c02668db5,2026-03-18T13:10:37.492021,{},Trench 3,android,Pre_Train,2023-11-30
4,2705022,a0a81a9b-78f9-4d6c-8d56-ea4114e40862,70223021-a662-4470-9375-10f0768e3208,0.549510,2026-03-18T13:10:37.492021,2026-03-18T13:10:37.492021,Beta-Cash-AppScore-Model,v1,"{""beta_cash_app_score"": 0.5495098814808624, ""m...",Cash September 25 Models,7ac89136-f836-4b2c-908b-4def0423676b,2026-03-18T13:10:37.492021,{},Trench 3,android,Dev_Test,2025-03-20


In [115]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,43321,2025-02-01,2025-09-23
1,Dev_Train,12362,2024-08-13,2025-01-31
2,Pre_Train,10191,2023-01-01,2024-08-12


In [116]:
dfd.info()
dfd['customerId'] = pd.to_numeric(dfd['customerId'], errors='coerce')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65874 entries, 0 to 65873
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   customerId            65874 non-null  object 
 1   digitalLoanAccountId  65874 non-null  object 
 2   crifApplicationId     65874 non-null  object 
 3   prediction            65874 non-null  float64
 4   start_time            65874 non-null  object 
 5   end_time              65874 non-null  object 
 6   modelDisplayName      65874 non-null  object 
 7   modelVersionId        65874 non-null  object 
 8   calcFeature           65874 non-null  object 
 9   subscription_name     65874 non-null  object 
 10  message_id            65874 non-null  object 
 11  publish_time          65874 non-null  object 
 12  attributes            65874 non-null  object 
 13  trenchCategory        65874 non-null  object 
 14  deviceOs              65874 non-null  object 
 15  Data_selection     

In [117]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=9473c41b-2708-4f68-ba50-08a7792ba38b>

# SIL V2

*   Sil-Alpha-CIC-SIL-Model: worktable_data_analysis.sil_alpha_cic_all_applied_backscored_20240901_20250930
*   Sil-Alpha-StackingModel: worktable_data_analysis.sil_alpha_applied_loans_backscored_20240901_20251013_option3
*   Sil-Beta-AppsScoreModel:
*   New Applicants (T1, T2): risk_mart.applied_sil_trench1_trench2_loan_jan2024_30sep2025_app_scored, risk_mart.applied_sil_new_applicants_loan_oct0125_oct1325_app_scored
*   Repeat Applicants (T3): risk_mart.b_score_model_applied_loans_sil_20240101_20250930_app_scored, risk_mart.applied_sil_repeat_applicants_loan_oct0125_oct1325_app_scored
*   Sil-Beta-DemoScoreModel: worktable_data_analysis.sil_beta_demo_all_applied_backscored_20240801_20251015
*   Sil-Beta-StackScoreModel: worktable_data_analysis.sil_beta_applied_loans_backscored_20240801_20251013_option3m

##### 'Alpha - CIC-SIL-Model'

##### Trench 1

In [142]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='cic_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 1'
;
"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=6e64543a-604d-4ce9-87df-6345d4e252ed>

In [143]:
sq = """ 
select 
r.customerId customer_id,
r.digitalLoanAccountId, 
r.c_cic_score ,
r.ScoreRange,
       ln_loan_level_user_type, flg_zero_non_granted_ever,
       flg_zero_granted_ever,
       Personal_Loans_granted_contracts_amt_24M,
       granted_contracts_cnt_6M, total_overdue_granted_contracts,
       has_ever_been_overdue, cnt_nongranted_contracts_3M,
       cnt_active_contracts, max_amt_granted_24M,
       tot_active_contracts_util, days_since_last_closed,
       vel_contract_nongranted_cnt_6on12,
       vel_contract_granted_amt_6on12,
       vel_contract_closed_amt_3on12,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
 date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
 case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-01-31' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
from worktable_data_analysis.sil_alpha_cic_all_applied_backscored_20240901_20250930 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 1'
and r.c_cic_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID f1337adc-6822-445f-8e75-344f7e973335 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (224718, 22)


In [144]:
data.columns

Index(['customer_id', 'digitalLoanAccountId', 'c_cic_score', 'ScoreRange',
       'ln_loan_level_user_type', 'flg_zero_non_granted_ever',
       'flg_zero_granted_ever', 'Personal_Loans_granted_contracts_amt_24M',
       'granted_contracts_cnt_6M', 'total_overdue_granted_contracts',
       'has_ever_been_overdue', 'cnt_nongranted_contracts_3M',
       'cnt_active_contracts', 'max_amt_granted_24M',
       'tot_active_contracts_util', 'days_since_last_closed',
       'vel_contract_nongranted_cnt_6on12', 'vel_contract_granted_amt_6on12',
       'vel_contract_closed_amt_3on12', 'osType', 'application_date',
       'Data_selection'],
      dtype='object')

In [145]:
data.head()

,customer_id,digitalLoanAccountId,c_cic_score,ScoreRange,ln_loan_level_user_type,flg_zero_non_granted_ever,flg_zero_granted_ever,Personal_Loans_granted_contracts_amt_24M,granted_contracts_cnt_6M,total_overdue_granted_contracts,has_ever_been_overdue,cnt_nongranted_contracts_3M,cnt_active_contracts,max_amt_granted_24M,tot_active_contracts_util,days_since_last_closed,vel_contract_nongranted_cnt_6on12,vel_contract_granted_amt_6on12,vel_contract_closed_amt_3on12,osType,application_date,Data_selection
0,3689452,445aa824-27bd-43c6-8d7b-12d60c12b034,0.249927,Missing,2_New Applicant,1,0,67817.0,NaN,10.0,1.0,NaN,3.0,20154.0,0.394789,26.0,NaN,NaN,3.336709,ios,2025-09-17,Dev_Test
1,3165246,f826da25-a473-4de2-bfdd-98f38135df68,0.461102,Missing,2_New Applicant,0,0,12500.0,NaN,6.0,1.0,NaN,NaN,7551.0,NaN,232.0,NaN,NaN,NaN,ios,2025-01-04,Dev_Train
2,3418828,85f1ade8-959d-49ef-85c6-e088ae79827a,0.259071,Di,2_New Applicant,0,0,NaN,NaN,13.0,1.0,NaN,2.0,12596.0,0.000000,65.0,2.012048,NaN,0.885086,android,2025-05-04,Dev_Test
3,3139257,c98bccaa-7989-4eb0-9ac4-5d916e14bdd4,0.313481,NH_Hi,2_New Applicant,0,0,661885.0,1.0,8.0,1.0,NaN,3.0,661885.0,1.134676,102.0,0.668000,4.47224,NaN,ios,2024-12-25,Dev_Train
4,3000827,3b15775d-ba18-4d1d-a3a0-6932c1a26078,0.335649,Missing,2_New Applicant,1,0,755199.0,NaN,6.0,1.0,NaN,11.0,370000.0,1.279565,49.0,NaN,NaN,0.036681,android,2024-11-05,Dev_Train


In [146]:
feature_column = ['ScoreRange',
       'ln_loan_level_user_type', 'flg_zero_non_granted_ever',
       'flg_zero_granted_ever', 'Personal_Loans_granted_contracts_amt_24M',
       'granted_contracts_cnt_6M', 'total_overdue_granted_contracts',
       'has_ever_been_overdue', 'cnt_nongranted_contracts_3M',
       'cnt_active_contracts', 'max_amt_granted_24M',
       'tot_active_contracts_util', 'days_since_last_closed',
       'vel_contract_nongranted_cnt_6on12', 'vel_contract_granted_amt_6on12',
       'vel_contract_closed_amt_3on12',]

dfd = transform_datav2(data, feature_column, a='c_cic_score', modelDisplayName='cic_model_sil', tc='Trench 1', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

The shape of the transformed dataframe is:	 (224718, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3689452,445aa824-27bd-43c6-8d7b-12d60c12b034,ded013e8-52e9-4ec5-a107-97497979ea8c,0.249927,2026-03-18T13:18:30.653503,2026-03-18T13:18:30.653503,cic_model_sil,v2,"{""ScoreRange"": ""Missing"", ""ln_loan_level_user_...",Cash November 25 Models,57318932-0745-44d1-ac0c-939f3afc036e,2026-03-18T13:18:30.653503,{},Trench 1,ios,Dev_Test,2025-09-17
1,3165246,f826da25-a473-4de2-bfdd-98f38135df68,fa0745a2-5e49-4ad6-bcd5-fd18d055cd98,0.461102,2026-03-18T13:18:30.653503,2026-03-18T13:18:30.653503,cic_model_sil,v2,"{""ScoreRange"": ""Missing"", ""ln_loan_level_user_...",Cash November 25 Models,38f9914e-9227-48f6-a54f-c39dad43ae22,2026-03-18T13:18:30.653503,{},Trench 1,ios,Dev_Train,2025-01-04
2,3418828,85f1ade8-959d-49ef-85c6-e088ae79827a,a339e12a-7ea2-4f5a-816f-ac4d2f4b4489,0.259071,2026-03-18T13:18:30.654503,2026-03-18T13:18:30.654503,cic_model_sil,v2,"{""ScoreRange"": ""Di"", ""ln_loan_level_user_type""...",Cash November 25 Models,b187b049-5d1b-434c-9942-597adf2e67d2,2026-03-18T13:18:30.654503,{},Trench 1,android,Dev_Test,2025-05-04
3,3139257,c98bccaa-7989-4eb0-9ac4-5d916e14bdd4,15df8859-60cc-4841-8a36-c9a0beb773ce,0.313481,2026-03-18T13:18:30.654503,2026-03-18T13:18:30.654503,cic_model_sil,v2,"{""ScoreRange"": ""NH_Hi"", ""ln_loan_level_user_ty...",Cash November 25 Models,18db038c-414d-4f8a-8f94-a3fab204a067,2026-03-18T13:18:30.654503,{},Trench 1,ios,Dev_Train,2024-12-25
4,3000827,3b15775d-ba18-4d1d-a3a0-6932c1a26078,ade004c5-93c6-4526-980d-e71866455881,0.335649,2026-03-18T13:18:30.654503,2026-03-18T13:18:30.654503,cic_model_sil,v2,"{""ScoreRange"": ""Missing"", ""ln_loan_level_user_...",Cash November 25 Models,495e7735-04d2-4ade-a5ee-85d7d90b0aab,2026-03-18T13:18:30.654503,{},Trench 1,android,Dev_Train,2024-11-05


In [147]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,130029,2025-02-01,2025-11-15
1,Dev_Train,94689,2024-09-01,2025-01-31


In [148]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=0d1b89a5-c2e4-4a17-9309-a4358ca3abeb>

##### Trench 2

In [149]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='cic_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 2'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=3d15fa82-2d35-4226-b388-50cf36f559ee>

In [150]:
sq = """ 
select 
r.customerId customer_id,
r.digitalLoanAccountId, 
r.c_cic_score ,
r.ScoreRange,
       ln_loan_level_user_type, flg_zero_non_granted_ever,
       flg_zero_granted_ever,
       Personal_Loans_granted_contracts_amt_24M,
       granted_contracts_cnt_6M, total_overdue_granted_contracts,
       has_ever_been_overdue, cnt_nongranted_contracts_3M,
       cnt_active_contracts, max_amt_granted_24M,
       tot_active_contracts_util, days_since_last_closed,
       vel_contract_nongranted_cnt_6on12,
       vel_contract_granted_amt_6on12,
       vel_contract_closed_amt_3on12,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
 case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-01-31' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
from worktable_data_analysis.sil_alpha_cic_all_applied_backscored_20240901_20250930 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 2'
and r.c_cic_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
 ;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 0785c379-46e2-49c5-8a32-da3ac20f290a successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (10175, 22)


In [151]:
data.columns

Index(['customer_id', 'digitalLoanAccountId', 'c_cic_score', 'ScoreRange',
       'ln_loan_level_user_type', 'flg_zero_non_granted_ever',
       'flg_zero_granted_ever', 'Personal_Loans_granted_contracts_amt_24M',
       'granted_contracts_cnt_6M', 'total_overdue_granted_contracts',
       'has_ever_been_overdue', 'cnt_nongranted_contracts_3M',
       'cnt_active_contracts', 'max_amt_granted_24M',
       'tot_active_contracts_util', 'days_since_last_closed',
       'vel_contract_nongranted_cnt_6on12', 'vel_contract_granted_amt_6on12',
       'vel_contract_closed_amt_3on12', 'osType', 'application_date',
       'Data_selection'],
      dtype='object')

In [152]:
data.head()

,customer_id,digitalLoanAccountId,c_cic_score,ScoreRange,ln_loan_level_user_type,flg_zero_non_granted_ever,flg_zero_granted_ever,Personal_Loans_granted_contracts_amt_24M,granted_contracts_cnt_6M,total_overdue_granted_contracts,has_ever_been_overdue,cnt_nongranted_contracts_3M,cnt_active_contracts,max_amt_granted_24M,tot_active_contracts_util,days_since_last_closed,vel_contract_nongranted_cnt_6on12,vel_contract_granted_amt_6on12,vel_contract_closed_amt_3on12,osType,application_date,Data_selection
0,1923580,421d394d-0ade-4ab7-bd73-1de794144de9,0.509831,Missing,2_New Applicant,0,0,NaN,2.0,9.0,1.0,NaN,5.0,91000.0,1.282551,57.0,1.994012,0.829817,1.494748,android,2024-10-11,Dev_Train
1,1759650,9f0b50dd-c825-4a79-a284-f554f63e60ff,0.313782,Missing,2_New Applicant,0,0,24246.0,NaN,19.0,1.0,NaN,1.0,5500.0,0.000000,63.0,NaN,NaN,1.000000,android,2025-02-12,Dev_Test
2,2206509,f2ed9ac6-7277-4f64-bb4e-7d5c15c02b33,0.325080,Missing,2_New Applicant,0,0,23453.0,NaN,9.0,1.0,1.0,24.0,255000.0,0.176194,22.0,2.012048,NaN,0.095238,ios,2025-06-25,Dev_Test
3,3165017,058c8292-30fc-4570-bdc6-94acf48f613e,0.390597,Ai,2_New Applicant,0,0,15000.0,NaN,9.0,1.0,NaN,1.0,15000.0,0.000000,394.0,1.994012,NaN,NaN,ios,2025-06-20,Dev_Test
4,3071143,c45e0bfb-3f76-423a-a680-fc5e9c5ffa9e,0.202156,Di,2_New Applicant,0,0,343950.0,3.0,10.0,1.0,1.0,7.0,225000.0,1.092431,19.0,0.798561,0.484150,2.366013,android,2025-04-30,Dev_Test


In [153]:
feature_column = ['ScoreRange',
       'ln_loan_level_user_type', 'flg_zero_non_granted_ever',
       'flg_zero_granted_ever', 'Personal_Loans_granted_contracts_amt_24M',
       'granted_contracts_cnt_6M', 'total_overdue_granted_contracts',
       'has_ever_been_overdue', 'cnt_nongranted_contracts_3M',
       'cnt_active_contracts', 'max_amt_granted_24M',
       'tot_active_contracts_util', 'days_since_last_closed',
       'vel_contract_nongranted_cnt_6on12', 'vel_contract_granted_amt_6on12',
       'vel_contract_closed_amt_3on12',]

dfd = transform_datav2(data, feature_column, a='c_cic_score', modelDisplayName='cic_model_sil', tc='Trench 2', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

The shape of the transformed dataframe is:	 (10175, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,1923580,421d394d-0ade-4ab7-bd73-1de794144de9,99ea09bb-80f2-4f12-81b1-9b8c9939aef5,0.509831,2026-03-18T13:20:52.392876,2026-03-18T13:20:52.392876,cic_model_sil,v2,"{""ScoreRange"": ""Missing"", ""ln_loan_level_user_...",Cash November 25 Models,91b1be56-4f0a-4137-8fa4-1c4c145724ac,2026-03-18T13:20:52.392876,{},Trench 2,android,Dev_Train,2024-10-11
1,1759650,9f0b50dd-c825-4a79-a284-f554f63e60ff,59acd5a3-e1cd-498c-849d-d83c29a04823,0.313782,2026-03-18T13:20:52.392876,2026-03-18T13:20:52.392876,cic_model_sil,v2,"{""ScoreRange"": ""Missing"", ""ln_loan_level_user_...",Cash November 25 Models,9a28e7b9-dfe6-40a6-8048-dec05eb62322,2026-03-18T13:20:52.392876,{},Trench 2,android,Dev_Test,2025-02-12
2,2206509,f2ed9ac6-7277-4f64-bb4e-7d5c15c02b33,fb7d264d-e0c4-496b-bd0d-4f2ed3020fd5,0.325080,2026-03-18T13:20:52.393876,2026-03-18T13:20:52.393876,cic_model_sil,v2,"{""ScoreRange"": ""Missing"", ""ln_loan_level_user_...",Cash November 25 Models,63aff0c8-89d3-49af-a559-279d34e914e0,2026-03-18T13:20:52.393876,{},Trench 2,ios,Dev_Test,2025-06-25
3,3165017,058c8292-30fc-4570-bdc6-94acf48f613e,6fbcdda5-5f16-4926-aa3f-9dbc6b57060c,0.390597,2026-03-18T13:20:52.393876,2026-03-18T13:20:52.393876,cic_model_sil,v2,"{""ScoreRange"": ""Ai"", ""ln_loan_level_user_type""...",Cash November 25 Models,9adb0efc-8327-45b7-a425-7bfd02c43e5c,2026-03-18T13:20:52.393876,{},Trench 2,ios,Dev_Test,2025-06-20
4,3071143,c45e0bfb-3f76-423a-a680-fc5e9c5ffa9e,a2e9289a-0e5c-4d89-9f8f-9029cedd16ad,0.202156,2026-03-18T13:20:52.393876,2026-03-18T13:20:52.393876,cic_model_sil,v2,"{""ScoreRange"": ""Di"", ""ln_loan_level_user_type""...",Cash November 25 Models,f88e35ef-e4dc-44f0-a4a7-bc77d022bc66,2026-03-18T13:20:52.393876,{},Trench 2,android,Dev_Test,2025-04-30


In [154]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,6943,2025-02-01,2025-09-30
1,Dev_Train,3232,2024-09-01,2025-01-31


In [155]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=097a8e7b-72b1-4ec1-b5b6-49efb730129a>

##### Trench 3

In [156]:
sq = """
delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='cic_model_sil'
and modelVersionId = 'v1'
and trenchCategory = 'Trench 3'
;
"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=54a60488-d495-4b19-8e3b-3437ab827938>

In [157]:
sq = """ 
select 
r.customerId customer_id,
r.digitalLoanAccountId, 
r.c_cic_score ,
r.ScoreRange,
       ln_loan_level_user_type, flg_zero_non_granted_ever,
       flg_zero_granted_ever,
       Personal_Loans_granted_contracts_amt_24M,
       granted_contracts_cnt_6M, total_overdue_granted_contracts,
       has_ever_been_overdue, cnt_nongranted_contracts_3M,
       cnt_active_contracts, max_amt_granted_24M,
       tot_active_contracts_util, days_since_last_closed,
       vel_contract_nongranted_cnt_6on12,
       vel_contract_granted_amt_6on12,
       vel_contract_closed_amt_3on12,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
 case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-01-31' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
from worktable_data_analysis.sil_alpha_cic_all_applied_backscored_20240901_20250930 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 3'
and r.c_cic_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID bb5963b4-1f40-4a19-a0d8-87ea60200b2d successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (11515, 22)


In [158]:
data.columns

Index(['customer_id', 'digitalLoanAccountId', 'c_cic_score', 'ScoreRange',
       'ln_loan_level_user_type', 'flg_zero_non_granted_ever',
       'flg_zero_granted_ever', 'Personal_Loans_granted_contracts_amt_24M',
       'granted_contracts_cnt_6M', 'total_overdue_granted_contracts',
       'has_ever_been_overdue', 'cnt_nongranted_contracts_3M',
       'cnt_active_contracts', 'max_amt_granted_24M',
       'tot_active_contracts_util', 'days_since_last_closed',
       'vel_contract_nongranted_cnt_6on12', 'vel_contract_granted_amt_6on12',
       'vel_contract_closed_amt_3on12', 'osType', 'application_date',
       'Data_selection'],
      dtype='object')

In [159]:
data.head()

,customer_id,digitalLoanAccountId,c_cic_score,ScoreRange,ln_loan_level_user_type,flg_zero_non_granted_ever,flg_zero_granted_ever,Personal_Loans_granted_contracts_amt_24M,granted_contracts_cnt_6M,total_overdue_granted_contracts,has_ever_been_overdue,cnt_nongranted_contracts_3M,cnt_active_contracts,max_amt_granted_24M,tot_active_contracts_util,days_since_last_closed,vel_contract_nongranted_cnt_6on12,vel_contract_granted_amt_6on12,vel_contract_closed_amt_3on12,osType,application_date,Data_selection
0,3448163,64667579-354d-4f2f-bc86-ac9c51a9944a,0.079853,Ci,1_Repeat Applicant,0,0,NaN,NaN,0.0,0.0,NaN,1.0,NaN,0.000000,NaN,2.012048,NaN,NaN,android,2025-08-30,Dev_Test
1,3109225,59328792-ff17-4c3c-9b69-7c5daca07ec0,0.140160,Ai,1_Repeat Applicant,1,0,NaN,NaN,0.0,0.0,NaN,NaN,8390.0,NaN,20.0,NaN,NaN,1.000000,android,2025-09-14,Dev_Test
2,3123116,13f13f70-b2dc-4448-aac1-bcf2ace33700,0.077279,NH_Hi,1_Repeat Applicant,0,0,129463.0,3.0,0.0,0.0,1.0,4.0,51594.0,0.596586,10.0,1.994012,1.137605,0.800388,android,2025-08-31,Dev_Test
3,2702009,3857b22e-2c89-4898-a56a-2a742bb30ea4,0.132599,Ai,1_Repeat Applicant,1,0,NaN,NaN,0.0,0.0,NaN,1.0,5799.0,0.390067,NaN,NaN,NaN,NaN,android,2025-01-31,Dev_Train
4,2705443,814e7699-9b5d-4a87-95c9-6fe58343c2df,0.059123,Fi,1_Repeat Applicant,1,0,825769.0,2.0,0.0,0.0,NaN,3.0,810000.0,0.802743,65.0,NaN,1.484259,1.031923,android,2025-04-08,Dev_Test


In [160]:
feature_column = ['ScoreRange',
       'ln_loan_level_user_type', 'flg_zero_non_granted_ever',
       'flg_zero_granted_ever', 'Personal_Loans_granted_contracts_amt_24M',
       'granted_contracts_cnt_6M', 'total_overdue_granted_contracts',
       'has_ever_been_overdue', 'cnt_nongranted_contracts_3M',
       'cnt_active_contracts', 'max_amt_granted_24M',
       'tot_active_contracts_util', 'days_since_last_closed',
       'vel_contract_nongranted_cnt_6on12', 'vel_contract_granted_amt_6on12',
       'vel_contract_closed_amt_3on12',]

dfd = transform_datav2(data, feature_column, a='c_cic_score', modelDisplayName='cic_model_sil', tc='Trench 3', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

The shape of the transformed dataframe is:	 (11515, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3448163,64667579-354d-4f2f-bc86-ac9c51a9944a,0fd40ed9-5e6a-4660-b49f-c8d2f7385ac6,0.079853,2026-03-18T13:23:50.648451,2026-03-18T13:23:50.648451,cic_model_sil,v2,"{""ScoreRange"": ""Ci"", ""ln_loan_level_user_type""...",Cash November 25 Models,b160ca9c-2f44-41a2-95da-0caad2bb4953,2026-03-18T13:23:50.648451,{},Trench 3,android,Dev_Test,2025-08-30
1,3109225,59328792-ff17-4c3c-9b69-7c5daca07ec0,1ec98212-3abe-4ace-a0e4-34b8053b2327,0.140160,2026-03-18T13:23:50.648451,2026-03-18T13:23:50.648451,cic_model_sil,v2,"{""ScoreRange"": ""Ai"", ""ln_loan_level_user_type""...",Cash November 25 Models,56f06471-a5d9-439a-8d9f-a1f1c054f295,2026-03-18T13:23:50.648451,{},Trench 3,android,Dev_Test,2025-09-14
2,3123116,13f13f70-b2dc-4448-aac1-bcf2ace33700,c08cda49-1e5f-4148-ab2c-55bd0a0a01b5,0.077279,2026-03-18T13:23:50.649460,2026-03-18T13:23:50.649460,cic_model_sil,v2,"{""ScoreRange"": ""NH_Hi"", ""ln_loan_level_user_ty...",Cash November 25 Models,94db6230-29a4-4bf4-9bed-a712cdd8896d,2026-03-18T13:23:50.649460,{},Trench 3,android,Dev_Test,2025-08-31
3,2702009,3857b22e-2c89-4898-a56a-2a742bb30ea4,2da63459-b578-43ef-9c67-72bfad2ddfa2,0.132599,2026-03-18T13:23:50.649460,2026-03-18T13:23:50.649460,cic_model_sil,v2,"{""ScoreRange"": ""Ai"", ""ln_loan_level_user_type""...",Cash November 25 Models,9de17c97-1b38-4c64-a6f3-ea1bdfaff881,2026-03-18T13:23:50.649460,{},Trench 3,android,Dev_Train,2025-01-31
4,2705443,814e7699-9b5d-4a87-95c9-6fe58343c2df,ce1e0895-bb31-4d79-a7d6-638460ad0198,0.059123,2026-03-18T13:23:50.649460,2026-03-18T13:23:50.649460,cic_model_sil,v2,"{""ScoreRange"": ""Fi"", ""ln_loan_level_user_type""...",Cash November 25 Models,79c60f95-6ed1-42de-be7e-757c5685830e,2026-03-18T13:23:50.649460,{},Trench 3,android,Dev_Test,2025-04-08


In [161]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,7246,2025-02-01,2025-09-30
1,Dev_Train,4269,2024-09-01,2025-01-31


In [162]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=020ffb9c-9ba6-45ef-8381-f4ab80fb2665>

##### Alpha Sil Stack Model 

##### Trench 1

In [163]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='alpha_stack_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 1'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=f0c07116-bf75-4315-b5ac-ae4a3702fba4>

In [164]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,  
r.sa_stack_score ,
r.sb_demo_score,
r.apps_score,
r.s_credo_score,
r.sa_cic_score,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
ln_loan_type,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from worktable_data_analysis.sil_alpha_applied_loans_backscored_20240901_20251013_option3 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 1'
and sa_stack_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


Job ID 2e3a9ede-7fe3-4dee-b8a7-aeee581cd3bc successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (232107, 11)


In [165]:
data.columns

Index(['customer_id', 'digitalLoanAccountId', 'sa_stack_score',
       'sb_demo_score', 'apps_score', 's_credo_score', 'sa_cic_score',
       'osType', 'ln_loan_type', 'application_date', 'Data_selection'],
      dtype='object')

In [166]:
data.head()

,customer_id,digitalLoanAccountId,sa_stack_score,sb_demo_score,apps_score,s_credo_score,sa_cic_score,osType,ln_loan_type,application_date,Data_selection
0,3711379,e0dd7934-a35b-44ee-843c-34e7dc92727f,0.672188,0.500623,0.493070,0.206244,0.442793,android,SIL ZERO,2025-09-28,Dev_Test
1,3324998,0adab6b0-0a06-496b-bf3e-eb13c0c4b9de,0.063752,0.258695,0.143028,0.074653,0.118948,android,SIL ZERO,2025-03-17,Dev_Test
2,3590345,03972320-362c-44c4-8e23-1a1ad726bc3c,0.381231,0.317476,NaN,0.091057,0.363428,ios,SIL ZERO,2025-07-31,Dev_Test
3,3621685,77e561bd-05eb-4a28-9138-99ab7c04800d,0.528169,0.521979,0.359197,0.247165,0.344859,android,SIL ZERO,2025-09-05,Dev_Test
4,3522856,28ad140d-fdfa-4ab5-a3ee-086c9682311d,0.289346,0.318908,0.429465,0.030021,0.344859,android,SIL ZERO,2025-06-27,Dev_Test


In [167]:
feature_column = ['sb_demo_score', 'apps_score', 's_credo_score', 'sa_cic_score',]

dfd = transform_datav2(data, feature_column, a='sa_stack_score', modelDisplayName='alpha_stack_model_sil', tc='Trench 1', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

The shape of the transformed dataframe is:	 (232107, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3711379,e0dd7934-a35b-44ee-843c-34e7dc92727f,79109593-7a0b-48aa-8679-43fd976e20f1,0.672188,2026-03-18T13:27:09.869631,2026-03-18T13:27:09.869631,alpha_stack_model_sil,v2,"{""sb_demo_score"": 0.5006232009314898, ""apps_sc...",Cash November 25 Models,e3e5513d-936c-4b6a-9bd2-fe0980fcd289,2026-03-18T13:27:09.869631,{},Trench 1,android,Dev_Test,2025-09-28
1,3324998,0adab6b0-0a06-496b-bf3e-eb13c0c4b9de,f1bdd45e-8cc5-4e64-8d6c-531ac6ec55b9,0.063752,2026-03-18T13:27:09.869631,2026-03-18T13:27:09.869631,alpha_stack_model_sil,v2,"{""sb_demo_score"": 0.25869535892928314, ""apps_s...",Cash November 25 Models,c300ddc0-1a8d-4923-b900-8658831be6a2,2026-03-18T13:27:09.869631,{},Trench 1,android,Dev_Test,2025-03-17
2,3590345,03972320-362c-44c4-8e23-1a1ad726bc3c,d2e290a7-d006-4ece-b8a1-6d5b834405f7,0.381231,2026-03-18T13:27:09.869631,2026-03-18T13:27:09.869631,alpha_stack_model_sil,v2,"{""sb_demo_score"": 0.31747606239363607, ""s_cred...",Cash November 25 Models,f455c537-889b-4e19-951d-1533b8f0d81f,2026-03-18T13:27:09.869631,{},Trench 1,ios,Dev_Test,2025-07-31
3,3621685,77e561bd-05eb-4a28-9138-99ab7c04800d,97a9b39f-b6db-4892-a801-3926fb7095e9,0.528169,2026-03-18T13:27:09.869631,2026-03-18T13:27:09.869631,alpha_stack_model_sil,v2,"{""sb_demo_score"": 0.5219786171009214, ""apps_sc...",Cash November 25 Models,aa1f4b6a-1f82-4623-ab43-d3832d14946c,2026-03-18T13:27:09.869631,{},Trench 1,android,Dev_Test,2025-09-05
4,3522856,28ad140d-fdfa-4ab5-a3ee-086c9682311d,6d413f10-d74b-4043-8c9b-ad488c05f3ba,0.289346,2026-03-18T13:27:09.870652,2026-03-18T13:27:09.870652,alpha_stack_model_sil,v2,"{""sb_demo_score"": 0.3189080225804797, ""apps_sc...",Cash November 25 Models,c39c6de6-301d-48f2-ad44-783a82a87f05,2026-03-18T13:27:09.870652,{},Trench 1,android,Dev_Test,2025-06-27


In [168]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,124744,2025-03-01,2025-11-15
1,Dev_Train,107363,2024-09-01,2025-02-28


In [169]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=b7bd0660-a7dd-41a5-92d6-400ed8a91854>

##### Trench 2

In [170]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='alpha_stack_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 2'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=9ee5668f-a958-4017-94ce-1c85dff79634>

In [171]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,  
r.sa_stack_score ,
r.sb_demo_score,
r.apps_score,
r.s_credo_score,
r.sa_cic_score,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
ln_loan_type,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from worktable_data_analysis.sil_alpha_applied_loans_backscored_20240901_20251013_option3 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 2'
and sa_stack_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


Job ID 931efb98-1f32-4485-9083-df20bc25cfcc successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (10592, 11)


In [172]:
data.columns

Index(['customer_id', 'digitalLoanAccountId', 'sa_stack_score',
       'sb_demo_score', 'apps_score', 's_credo_score', 'sa_cic_score',
       'osType', 'ln_loan_type', 'application_date', 'Data_selection'],
      dtype='object')

In [173]:
data.head()

,customer_id,digitalLoanAccountId,sa_stack_score,sb_demo_score,apps_score,s_credo_score,sa_cic_score,osType,ln_loan_type,application_date,Data_selection
0,3262042,2201b3a2-b0b4-4835-876a-db46193804bf,0.479728,0.405268,0.447964,0.086345,0.392868,android,SIL Competitor,2025-08-03,Dev_Test
1,2387742,129cd5c3-223f-4758-aa10-3b49bdef10b6,0.437951,0.479125,NaN,0.049538,0.288705,ios,SIL Competitor,2025-05-05,Dev_Test
2,3369429,8d0fd20c-80d8-42b8-b46d-3bb7825f6203,0.656393,0.447675,0.623634,0.153387,0.345233,android,SIL Competitor,2025-07-10,Dev_Test
3,2578405,49ca271e-14f7-4963-ae35-4e5945d3f95e,0.535884,0.435029,0.665279,0.133979,0.193938,android,SIL Competitor,2025-07-04,Dev_Test
4,3180844,5bd3d932-1dce-41bd-8d08-dd6e1a2c5176,0.480241,0.357332,NaN,0.185610,0.318677,ios,SIL Competitor,2025-07-14,Dev_Test


In [174]:
feature_column = ['sb_demo_score', 'apps_score', 's_credo_score', 'sa_cic_score',]

dfd = transform_datav2(data, feature_column, a='sa_stack_score', modelDisplayName='alpha_stack_model_sil', tc='Trench 2', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

The shape of the transformed dataframe is:	 (10592, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3262042,2201b3a2-b0b4-4835-876a-db46193804bf,b3c4f6f7-e958-41e0-9310-e015a4046758,0.479728,2026-03-18T13:40:05.824923,2026-03-18T13:40:05.824923,alpha_stack_model_sil,v2,"{""sb_demo_score"": 0.4052679214594425, ""apps_sc...",Cash November 25 Models,19b44c46-8229-4704-9c15-97e5d0dd7edb,2026-03-18T13:40:05.824923,{},Trench 2,android,Dev_Test,2025-08-03
1,2387742,129cd5c3-223f-4758-aa10-3b49bdef10b6,0f841e94-2518-446d-9ac8-4dfe099b9a55,0.437951,2026-03-18T13:40:05.825469,2026-03-18T13:40:05.825469,alpha_stack_model_sil,v2,"{""sb_demo_score"": 0.47912512350372377, ""s_cred...",Cash November 25 Models,62eb74f0-bebf-4cf1-93b4-5c979eaaf427,2026-03-18T13:40:05.825469,{},Trench 2,ios,Dev_Test,2025-05-05
2,3369429,8d0fd20c-80d8-42b8-b46d-3bb7825f6203,3be794b9-040c-4700-aa36-b6dc61659558,0.656393,2026-03-18T13:40:05.825469,2026-03-18T13:40:05.825469,alpha_stack_model_sil,v2,"{""sb_demo_score"": 0.4476747521058565, ""apps_sc...",Cash November 25 Models,72975838-be95-4e06-810b-a56feb19f044,2026-03-18T13:40:05.825469,{},Trench 2,android,Dev_Test,2025-07-10
3,2578405,49ca271e-14f7-4963-ae35-4e5945d3f95e,62982da5-fa1a-4cb1-b916-a60310e641ad,0.535884,2026-03-18T13:40:05.825992,2026-03-18T13:40:05.825992,alpha_stack_model_sil,v2,"{""sb_demo_score"": 0.4350287261963506, ""apps_sc...",Cash November 25 Models,9df05203-dcb3-4279-999f-25ca78dbd457,2026-03-18T13:40:05.825992,{},Trench 2,android,Dev_Test,2025-07-04
4,3180844,5bd3d932-1dce-41bd-8d08-dd6e1a2c5176,f3885d6f-c2e7-4fda-8e12-2c4dc249ab1f,0.480241,2026-03-18T13:40:05.825992,2026-03-18T13:40:05.825992,alpha_stack_model_sil,v2,"{""sb_demo_score"": 0.35733203867724783, ""s_cred...",Cash November 25 Models,4f79a2de-94c8-4ea0-bc75-147c9cd27b2c,2026-03-18T13:40:05.825992,{},Trench 2,ios,Dev_Test,2025-07-14


In [175]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,6825,2025-03-01,2025-10-13
1,Dev_Train,3767,2024-09-01,2025-02-28


In [176]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=f70bc267-aaa0-413c-92b7-b75921a97a0f>

##### Trench 3

In [177]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='alpha_stack_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 3'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=5d9f2263-7feb-4d2e-b803-46d6651adca3>

In [178]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,  
r.sa_stack_score ,
r.sb_demo_score,
r.apps_score,
r.s_credo_score,
r.sa_cic_score,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
ln_loan_type,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from worktable_data_analysis.sil_alpha_applied_loans_backscored_20240901_20251013_option3 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 3'
and sa_stack_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


Job ID 77bb225d-56f3-425c-85d1-44857c0ae4a4 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (11874, 11)


In [179]:
data.columns

Index(['customer_id', 'digitalLoanAccountId', 'sa_stack_score',
       'sb_demo_score', 'apps_score', 's_credo_score', 'sa_cic_score',
       'osType', 'ln_loan_type', 'application_date', 'Data_selection'],
      dtype='object')

In [180]:
data.head()

,customer_id,digitalLoanAccountId,sa_stack_score,sb_demo_score,apps_score,s_credo_score,sa_cic_score,osType,ln_loan_type,application_date,Data_selection
0,2987225,2355eed7-f6d6-4b6f-83d5-8bd36b95374d,0.544774,0.377749,0.584459,0.298956,0.138251,android,SIL Competitor,2025-03-30,Dev_Test
1,2219515,895d01d4-c7d1-4afc-a414-328af93e14f5,0.274702,0.292695,0.526561,0.103902,0.103358,android,SIL Competitor,2025-05-19,Dev_Test
2,3124094,aae714e8-4c2a-400e-a09d-35b732c503d4,0.211122,0.252091,NaN,0.074150,0.106337,ios,SIL ZERO,2025-08-06,Dev_Test
3,3083049,2cdaa5fe-f8e9-4396-9b49-49d64d0cab65,0.280124,0.131825,0.654840,0.150196,0.083645,android,SIL Competitor,2025-09-19,Dev_Test
4,2928485,0a0331ef-dbde-4c29-8e18-29c08d561fa9,0.151194,0.096254,0.382579,0.152447,0.140574,android,SIL Competitor,2025-10-10,Dev_Test


In [181]:
feature_column = ['sb_demo_score', 'apps_score', 's_credo_score', 'sa_cic_score',]

dfd = transform_datav2(data, feature_column, a='sa_stack_score', modelDisplayName='alpha_stack_model_sil', tc='Trench 3', subscription_name = 'Cash November 25 Models')
print(f"The shape of the transformed dataframe is:\t {dfd.shape}") 
dfd.head()

The shape of the transformed dataframe is:	 (11874, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,2987225,2355eed7-f6d6-4b6f-83d5-8bd36b95374d,ab7ff269-1823-4a6e-8635-d4249e6500c4,0.544774,2026-03-18T13:42:17.412572,2026-03-18T13:42:17.412572,alpha_stack_model_sil,v2,"{""sb_demo_score"": 0.3777490097270223, ""apps_sc...",Cash November 25 Models,e48602c5-833f-4c7a-a885-020c1f2e5632,2026-03-18T13:42:17.412572,{},Trench 3,android,Dev_Test,2025-03-30
1,2219515,895d01d4-c7d1-4afc-a414-328af93e14f5,c602d867-18d6-4188-a86e-fb091aa129bf,0.274702,2026-03-18T13:42:17.412572,2026-03-18T13:42:17.412572,alpha_stack_model_sil,v2,"{""sb_demo_score"": 0.29269512529001973, ""apps_s...",Cash November 25 Models,b6e72b06-a735-4395-85b8-663a3a7c25d3,2026-03-18T13:42:17.412572,{},Trench 3,android,Dev_Test,2025-05-19
2,3124094,aae714e8-4c2a-400e-a09d-35b732c503d4,82c331f7-efbe-4bde-bd24-55950d2a7396,0.211122,2026-03-18T13:42:17.412572,2026-03-18T13:42:17.412572,alpha_stack_model_sil,v2,"{""sb_demo_score"": 0.25209106978826856, ""s_cred...",Cash November 25 Models,dd73a89d-45a5-431f-a20c-94250c317b61,2026-03-18T13:42:17.412572,{},Trench 3,ios,Dev_Test,2025-08-06
3,3083049,2cdaa5fe-f8e9-4396-9b49-49d64d0cab65,20fcbb56-de51-4dee-bb01-209a68b94a7b,0.280124,2026-03-18T13:42:17.412572,2026-03-18T13:42:17.412572,alpha_stack_model_sil,v2,"{""sb_demo_score"": 0.13182451879618157, ""apps_s...",Cash November 25 Models,103e8695-f2ce-4c98-8e50-ee5539d919d4,2026-03-18T13:42:17.412572,{},Trench 3,android,Dev_Test,2025-09-19
4,2928485,0a0331ef-dbde-4c29-8e18-29c08d561fa9,a9146525-5f53-464e-8e63-7a2a34586109,0.151194,2026-03-18T13:42:17.413572,2026-03-18T13:42:17.413572,alpha_stack_model_sil,v2,"{""sb_demo_score"": 0.09625371182021789, ""apps_s...",Cash November 25 Models,9ac187c3-1227-431d-8787-99171ad8a0df,2026-03-18T13:42:17.413572,{},Trench 3,android,Dev_Test,2025-10-10


In [182]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,6833,2025-03-01,2025-10-13
1,Dev_Train,5041,2024-09-01,2025-02-28


In [183]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=6055d4f7-ca9e-46d3-9b98-31b1275cd2d5>

##### Beta SIL STACK Score Model

##### Trench 1

In [184]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='beta_stack_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 1'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=9e063ffd-dd64-4bfd-9cc8-d7b1e20aa755>

In [185]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,  
r.sb_stack_score ,
r.sb_demo_score,
r.apps_score,
r.s_credo_score,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
ln_loan_type,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from prj-prod-dataplatform.worktable_data_analysis.sil_beta_applied_loans_backscored_20240801_20251013_option3 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 1'
and sb_stack_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


Job ID c7a48a4b-a506-4621-b160-6041d1f038fb successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (343968, 10)


In [186]:
data.columns

Index(['customer_id', 'digitalLoanAccountId', 'sb_stack_score',
       'sb_demo_score', 'apps_score', 's_credo_score', 'osType',
       'ln_loan_type', 'application_date', 'Data_selection'],
      dtype='object')

In [187]:
data.head()

,customer_id,digitalLoanAccountId,sb_stack_score,sb_demo_score,apps_score,s_credo_score,osType,ln_loan_type,application_date,Data_selection
0,3355005,b606e676-90e5-4f6c-a658-ce4d7f552325,0.352212,0.361102,NaN,0.092014,ios,SIL Competitor,2025-04-02,Dev_Test
1,3552257,dfe20a12-1235-4637-8d65-0b7060642054,0.482973,0.575337,0.458589,0.067895,android,SIL Competitor,2025-07-12,Dev_Test
2,3545718,911b656c-574f-4221-9610-6ab49a0d0109,0.491041,0.608599,0.413824,0.092557,android,SIL-Instore,2025-07-08,Dev_Test
3,3567544,ed002636-6e43-4876-835f-9342974ea475,0.333845,0.263899,NaN,0.159638,ios,SIL-Instore,2025-07-20,Dev_Test
4,3611679,5f09a7df-20e3-4fbd-be57-216da2c9da28,0.547915,0.464424,0.560874,0.140714,android,SIL-Instore,2025-08-10,Dev_Test


In [188]:
feature_column = ['sb_demo_score', 'apps_score', 's_credo_score']

dfd = transform_datav2(data, feature_column, a='sb_stack_score', modelDisplayName='beta_stack_model_sil', tc='Trench 1', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

The shape of the transformed dataframe is:	 (343968, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3355005,b606e676-90e5-4f6c-a658-ce4d7f552325,86479ab3-4bf7-4787-aa45-0a031a508bf9,0.352212,2026-03-18T13:44:17.698015,2026-03-18T13:44:17.698015,beta_stack_model_sil,v2,"{""sb_demo_score"": 0.36110233725638813, ""s_cred...",Cash November 25 Models,f0e6fb03-eb31-4ebc-900e-4513949e1a7f,2026-03-18T13:44:17.698015,{},Trench 1,ios,Dev_Test,2025-04-02
1,3552257,dfe20a12-1235-4637-8d65-0b7060642054,4deb1cb7-12ef-45be-a5cc-62ef6280e8e5,0.482973,2026-03-18T13:44:17.698015,2026-03-18T13:44:17.698015,beta_stack_model_sil,v2,"{""sb_demo_score"": 0.575336879256294, ""apps_sco...",Cash November 25 Models,0b755984-75f3-498f-b52a-8fa819a95d43,2026-03-18T13:44:17.698015,{},Trench 1,android,Dev_Test,2025-07-12
2,3545718,911b656c-574f-4221-9610-6ab49a0d0109,7fccf08a-5ac4-4404-ba08-1ca647b81d61,0.491041,2026-03-18T13:44:17.698015,2026-03-18T13:44:17.698015,beta_stack_model_sil,v2,"{""sb_demo_score"": 0.6085993263503627, ""apps_sc...",Cash November 25 Models,25463b05-f772-4152-982a-00fcfab8733c,2026-03-18T13:44:17.698015,{},Trench 1,android,Dev_Test,2025-07-08
3,3567544,ed002636-6e43-4876-835f-9342974ea475,5db84336-747b-4247-97f8-d66b49ea91c0,0.333845,2026-03-18T13:44:17.698015,2026-03-18T13:44:17.698015,beta_stack_model_sil,v2,"{""sb_demo_score"": 0.2638985629418433, ""s_credo...",Cash November 25 Models,94b63586-389f-4904-853d-a00c1ee47fbb,2026-03-18T13:44:17.698015,{},Trench 1,ios,Dev_Test,2025-07-20
4,3611679,5f09a7df-20e3-4fbd-be57-216da2c9da28,2861a465-0b02-4739-8aa0-c609ab4f9301,0.547915,2026-03-18T13:44:17.698015,2026-03-18T13:44:17.698015,beta_stack_model_sil,v2,"{""sb_demo_score"": 0.46442441233290965, ""apps_s...",Cash November 25 Models,4fd09f41-c3c8-46d9-83a2-a17dad29e1d2,2026-03-18T13:44:17.698015,{},Trench 1,android,Dev_Test,2025-08-10


In [189]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,165423,2025-03-01,2025-11-15
1,Dev_Train,178545,2024-08-01,2025-02-28


In [190]:
# Upload to BigQuery
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=ecc62c4b-85cf-4a4a-a7c6-2885d80fff9a>

##### Trench 2

In [191]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='beta_stack_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 2'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=2a236814-7335-4add-a9e1-94762ac81bf9>

In [192]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,  
r.sb_stack_score ,
r.sb_demo_score,
r.apps_score,
r.s_credo_score,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
ln_loan_type,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from prj-prod-dataplatform.worktable_data_analysis.sil_beta_applied_loans_backscored_20240801_20251013_option3 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 2'
and sb_stack_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


Job ID db701812-5922-43a6-b704-a419345b3714 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (13099, 10)


In [193]:
data.columns

Index(['customer_id', 'digitalLoanAccountId', 'sb_stack_score',
       'sb_demo_score', 'apps_score', 's_credo_score', 'osType',
       'ln_loan_type', 'application_date', 'Data_selection'],
      dtype='object')

In [194]:
data.head()

,customer_id,digitalLoanAccountId,sb_stack_score,sb_demo_score,apps_score,s_credo_score,osType,ln_loan_type,application_date,Data_selection
0,3305188,db423863-3336-4f28-9721-79acde2182bc,0.365681,0.394234,NaN,0.075432,ios,SIL ZERO,2025-04-12,Dev_Test
1,3211288,6e4b4ad2-26da-40cc-b6c6-6519ad2c4bac,0.339600,0.414519,0.366754,0.139482,android,SIL ZERO,2025-04-01,Dev_Test
2,3520602,c275eae9-799e-4c69-bad1-eaa3645ffca7,0.740406,0.571154,0.587088,0.257950,android,SIL ZERO,2025-10-04,Dev_Test
3,2758550,f553b8a6-13b0-475f-b119-fb192e1a45db,0.517966,0.535973,NaN,0.122216,ios,SIL Competitor,2025-04-10,Dev_Test
4,1132167,bdccf23c-c514-466f-acc7-00c56f744d6a,0.409094,0.407775,0.498938,0.094215,android,SIL Competitor,2025-04-24,Dev_Test


In [195]:
feature_column = ['sb_demo_score', 'apps_score', 's_credo_score']

dfd = transform_datav2(data, feature_column, a='sb_stack_score', modelDisplayName='beta_stack_model_sil', tc='Trench 2', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

The shape of the transformed dataframe is:	 (13099, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3305188,db423863-3336-4f28-9721-79acde2182bc,59001fbd-2707-4c26-8cbf-885db1259f46,0.365681,2026-03-18T13:52:46.203365,2026-03-18T13:52:46.203365,beta_stack_model_sil,v2,"{""sb_demo_score"": 0.3942341334470709, ""s_credo...",Cash November 25 Models,958f2b80-eaf4-4432-b5c4-bb19c8c10230,2026-03-18T13:52:46.203365,{},Trench 2,ios,Dev_Test,2025-04-12
1,3211288,6e4b4ad2-26da-40cc-b6c6-6519ad2c4bac,a1b0ac5e-f5c5-44b0-ab9a-0a49cb963827,0.339600,2026-03-18T13:52:46.203365,2026-03-18T13:52:46.203365,beta_stack_model_sil,v2,"{""sb_demo_score"": 0.41451907035371716, ""apps_s...",Cash November 25 Models,360abb5d-5fe1-4f09-ad76-93945660fd3c,2026-03-18T13:52:46.203365,{},Trench 2,android,Dev_Test,2025-04-01
2,3520602,c275eae9-799e-4c69-bad1-eaa3645ffca7,ab5c897e-47da-4f84-bca0-01f217e63155,0.740406,2026-03-18T13:52:46.203365,2026-03-18T13:52:46.203365,beta_stack_model_sil,v2,"{""sb_demo_score"": 0.5711544141747951, ""apps_sc...",Cash November 25 Models,8a4e1d1d-1fc4-45be-9384-e3243bd7e567,2026-03-18T13:52:46.203365,{},Trench 2,android,Dev_Test,2025-10-04
3,2758550,f553b8a6-13b0-475f-b119-fb192e1a45db,29c73e2f-80cf-48ae-843e-100face1f7e6,0.517966,2026-03-18T13:52:46.203365,2026-03-18T13:52:46.203365,beta_stack_model_sil,v2,"{""sb_demo_score"": 0.5359731968258878, ""s_credo...",Cash November 25 Models,ccaf880b-f99d-428b-bed3-8e1de6852307,2026-03-18T13:52:46.203365,{},Trench 2,ios,Dev_Test,2025-04-10
4,1132167,bdccf23c-c514-466f-acc7-00c56f744d6a,00995c1e-8f08-4590-856e-0a63cc696505,0.409094,2026-03-18T13:52:46.204366,2026-03-18T13:52:46.204366,beta_stack_model_sil,v2,"{""sb_demo_score"": 0.4077753352313462, ""apps_sc...",Cash November 25 Models,ada7e05d-5d52-4ddb-9126-7e8a0807b002,2026-03-18T13:52:46.204366,{},Trench 2,android,Dev_Test,2025-04-24


In [196]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,7599,2025-03-01,2025-10-13
1,Dev_Train,5500,2024-08-01,2025-02-28


In [197]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=7d2b660b-09c5-4ed8-8bde-613c175075d8>

##### Trench 3

In [199]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='beta_stack_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 3'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=b5d03a29-81fa-4999-b08a-34e4ac98b396>

In [200]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,  
r.sb_stack_score ,
r.sb_demo_score,
r.apps_score,
r.s_credo_score,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
ln_loan_type,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from prj-prod-dataplatform.worktable_data_analysis.sil_beta_applied_loans_backscored_20240801_20251013_option3 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 3'
and sb_stack_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


Job ID 9bc17f22-fbe5-4f59-bc7d-bc45e04ac702 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (13802, 10)


In [201]:
data.columns

Index(['customer_id', 'digitalLoanAccountId', 'sb_stack_score',
       'sb_demo_score', 'apps_score', 's_credo_score', 'osType',
       'ln_loan_type', 'application_date', 'Data_selection'],
      dtype='object')

In [202]:
data.head()

,customer_id,digitalLoanAccountId,sb_stack_score,sb_demo_score,apps_score,s_credo_score,osType,ln_loan_type,application_date,Data_selection
0,3031736,1baffa6c-8aef-4717-9935-4eb80d56cf0c,0.184079,0.265795,0.354866,0.051139,android,SIL Competitor,2025-06-29,Dev_Test
1,3009495,dc3a437f-b33d-401d-b021-3aebead9256c,0.116324,0.139818,0.314420,0.055836,android,SIL Competitor,2025-04-12,Dev_Test
2,2998886,05eaef6f-6f0d-4495-bae5-b31d99ba700b,0.225952,0.114125,0.533593,0.081924,android,SIL Competitor,2025-07-25,Dev_Test
3,3175607,c7632cd5-e256-48ef-891b-039766e07afe,0.331529,0.235677,0.494777,0.161249,android,SIL-Instore,2025-09-30,Dev_Test
4,2879224,2515cf7c-6508-4fed-927d-ff20b07302eb,0.413739,0.297374,0.555051,0.142537,android,SIL-Instore,2025-07-26,Dev_Test


In [203]:
feature_column = ['sb_demo_score', 'apps_score', 's_credo_score']

dfd = transform_datav2(data, feature_column, a='sb_stack_score', modelDisplayName='beta_stack_model_sil', tc='Trench 3', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

The shape of the transformed dataframe is:	 (13802, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3031736,1baffa6c-8aef-4717-9935-4eb80d56cf0c,98870fab-ff90-4256-b07f-ed499e9ae931,0.184079,2026-03-18T15:17:01.187505,2026-03-18T15:17:01.187505,beta_stack_model_sil,v2,"{""sb_demo_score"": 0.2657949959014428, ""apps_sc...",Cash November 25 Models,4563c1de-f1bb-4151-b501-deed87800f18,2026-03-18T15:17:01.187505,{},Trench 3,android,Dev_Test,2025-06-29
1,3009495,dc3a437f-b33d-401d-b021-3aebead9256c,2bf0bb34-6747-4291-b094-1e444938b602,0.116324,2026-03-18T15:17:01.187505,2026-03-18T15:17:01.187505,beta_stack_model_sil,v2,"{""sb_demo_score"": 0.13981812598791601, ""apps_s...",Cash November 25 Models,be0a4ab7-3ac5-4e00-ba09-47248d82fbe5,2026-03-18T15:17:01.187505,{},Trench 3,android,Dev_Test,2025-04-12
2,2998886,05eaef6f-6f0d-4495-bae5-b31d99ba700b,7a311fb6-9064-46cf-afd1-52dbf1a31f68,0.225952,2026-03-18T15:17:01.187505,2026-03-18T15:17:01.187505,beta_stack_model_sil,v2,"{""sb_demo_score"": 0.11412513176590218, ""apps_s...",Cash November 25 Models,15088271-ab39-4a79-90bb-8a6af5e4f381,2026-03-18T15:17:01.187505,{},Trench 3,android,Dev_Test,2025-07-25
3,3175607,c7632cd5-e256-48ef-891b-039766e07afe,e84348f7-311f-4796-a2b2-386b215ea734,0.331529,2026-03-18T15:17:01.187505,2026-03-18T15:17:01.187505,beta_stack_model_sil,v2,"{""sb_demo_score"": 0.23567735950090052, ""apps_s...",Cash November 25 Models,8b31831e-a893-407b-8a74-45f270a3c682,2026-03-18T15:17:01.187505,{},Trench 3,android,Dev_Test,2025-09-30
4,2879224,2515cf7c-6508-4fed-927d-ff20b07302eb,0d900709-462e-41fa-820e-3f5376b5a19c,0.413739,2026-03-18T15:17:01.187505,2026-03-18T15:17:01.187505,beta_stack_model_sil,v2,"{""sb_demo_score"": 0.2973739866118426, ""apps_sc...",Cash November 25 Models,57cafaf8-3367-4eb3-a75f-ca530c3d814d,2026-03-18T15:17:01.187505,{},Trench 3,android,Dev_Test,2025-07-26


In [204]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,7383,2025-03-01,2025-10-13
1,Dev_Train,6419,2024-08-01,2025-02-28


In [205]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=b74944cb-fb1c-406d-8eb2-0fca8cad1fe7>

##### Beta Sil App Score

##### Trench 1

In [206]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='apps_score_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 1'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=f942c118-9786-4501-b590-2e4aed8293f9>

In [207]:
sq = """ 
select 
distinct 
loanmaster.customerId customer_id,
r.digitalLoanAccountId,
r.apps_score,
r.lr_score ml_score,
r.dl_score dl_score,
app_cnt_payday_ever,
app_cnt_finance_ever,
app_cnt_competitors_sil_ever,
app_cnt_competitors_ever,
app_cnt_finance_365d,
app_cnt_absence_tag_365d,
app_cnt_competitors_sil_365d,
app_cnt_finance_7d,
app_cnt_rated_for_3plus_ever,
app_cnt_payday_7d,
    case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
    when lower(loanmaster.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-01-31' then 'Dev_Train'
        when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from  risk_mart.applied_sil_new_applicants_loan_20241001_20251124_app_scored r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where apps_score is not null
and trench_category = 'Trench 1'
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


Job ID 29332497-d934-4ae2-88fe-87a65cfb3704 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (289957, 18)


In [208]:
data.columns

Index(['customer_id', 'digitalLoanAccountId', 'apps_score', 'ml_score',
       'dl_score', 'app_cnt_payday_ever', 'app_cnt_finance_ever',
       'app_cnt_competitors_sil_ever', 'app_cnt_competitors_ever',
       'app_cnt_finance_365d', 'app_cnt_absence_tag_365d',
       'app_cnt_competitors_sil_365d', 'app_cnt_finance_7d',
       'app_cnt_rated_for_3plus_ever', 'app_cnt_payday_7d', 'osType',
       'application_date', 'Data_selection'],
      dtype='object')

In [209]:
data.head()

,customer_id,digitalLoanAccountId,apps_score,ml_score,dl_score,app_cnt_payday_ever,app_cnt_finance_ever,app_cnt_competitors_sil_ever,app_cnt_competitors_ever,app_cnt_finance_365d,app_cnt_absence_tag_365d,app_cnt_competitors_sil_365d,app_cnt_finance_7d,app_cnt_rated_for_3plus_ever,app_cnt_payday_7d,osType,application_date,Data_selection
0,3722891,46350f5e-89b9-48f5-b7d4-bd905a45a263,0.447590,0.339078,0.536373,6,12.0,3,5,4.0,8.0,2.0,0.0,53.0,0.0,android,2025-10-04,Dev_Test
1,3729022,0c333dca-c829-4179-92a5-41954f1644cf,0.426756,0.279493,0.547243,6,13.0,4,5,4.0,3.0,1.0,0.0,38.0,0.0,android,2025-10-07,Dev_Test
2,3325103,61cc04e1-4a58-4b0d-bfd4-cf3deacd8bd3,0.597721,0.541667,0.643583,6,12.0,2,5,5.0,30.0,1.0,0.0,29.0,0.0,android,2025-03-17,Dev_Test
3,3672591,2205cb73-4ed8-4621-9211-cb2013d2e32d,0.327388,0.318969,0.334276,7,21.0,4,11,17.0,5.0,3.0,0.0,53.0,0.0,android,2025-09-08,Dev_Test
4,3605772,e21695b8-a9ac-49f0-87e7-45b09f1817f7,0.233528,0.413353,0.086399,4,12.0,1,5,12.0,9.0,1.0,0.0,56.0,0.0,android,2025-08-07,Dev_Test


In [210]:
feature_column = ['ml_score', 'dl_score', 'app_cnt_payday_ever', 'app_cnt_finance_ever',
       'app_cnt_competitors_sil_ever', 'app_cnt_competitors_ever',
       'app_cnt_finance_365d', 'app_cnt_absence_tag_365d',
       'app_cnt_competitors_sil_365d', 'app_cnt_finance_7d',
       'app_cnt_rated_for_3plus_ever', 'app_cnt_payday_7d',]

dfd = transform_datav2(data, feature_column, a='apps_score', modelDisplayName='apps_score_model_sil', tc='Trench 1', subscription_name = 'Cash November 25 Models')
print(f"the shape of the transformed dataframe is:\t {dfd.shape}") 
dfd.head()

the shape of the transformed dataframe is:	 (289957, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3722891,46350f5e-89b9-48f5-b7d4-bd905a45a263,9d54e178-09e5-4348-984f-39356a28ce21,0.447590,2026-03-18T15:21:48.293058,2026-03-18T15:21:48.293058,apps_score_model_sil,v2,"{""ml_score"": 0.33907846284938215, ""dl_score"": ...",Cash November 25 Models,c408d193-76fd-4fe6-81b8-c79d5f1e204a,2026-03-18T15:21:48.293058,{},Trench 1,android,Dev_Test,2025-10-04
1,3729022,0c333dca-c829-4179-92a5-41954f1644cf,3a51b7cb-6109-4327-8de7-ad5fd0375d02,0.426756,2026-03-18T15:21:48.293058,2026-03-18T15:21:48.293058,apps_score_model_sil,v2,"{""ml_score"": 0.2794930620963465, ""dl_score"": 0...",Cash November 25 Models,885cec17-5253-4f9e-a5be-981ba0e3f0be,2026-03-18T15:21:48.293058,{},Trench 1,android,Dev_Test,2025-10-07
2,3325103,61cc04e1-4a58-4b0d-bfd4-cf3deacd8bd3,a59f5cc0-e7ea-475d-abd7-0e5991700776,0.597721,2026-03-18T15:21:48.293058,2026-03-18T15:21:48.293058,apps_score_model_sil,v2,"{""ml_score"": 0.5416669513384693, ""dl_score"": 0...",Cash November 25 Models,8eb55c60-1292-45f0-a596-13d7ad844617,2026-03-18T15:21:48.293058,{},Trench 1,android,Dev_Test,2025-03-17
3,3672591,2205cb73-4ed8-4621-9211-cb2013d2e32d,cc0c4e98-569e-4803-b532-beb8e8529e13,0.327388,2026-03-18T15:21:48.293058,2026-03-18T15:21:48.293058,apps_score_model_sil,v2,"{""ml_score"": 0.3189690597114731, ""dl_score"": 0...",Cash November 25 Models,9e119f56-a984-49ab-a8df-0c200db69426,2026-03-18T15:21:48.293058,{},Trench 1,android,Dev_Test,2025-09-08
4,3605772,e21695b8-a9ac-49f0-87e7-45b09f1817f7,1a834797-03ad-4570-b46f-885ffafd6968,0.233528,2026-03-18T15:21:48.293058,2026-03-18T15:21:48.293058,apps_score_model_sil,v2,"{""ml_score"": 0.4133526958330042, ""dl_score"": 0...",Cash November 25 Models,ed426cbb-2990-4c9d-9e5f-a9b49163a6e6,2026-03-18T15:21:48.293058,{},Trench 1,android,Dev_Test,2025-08-07


In [211]:

result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,187810,2025-02-01,2025-11-16
1,Dev_Train,102147,2024-10-01,2025-01-31


In [212]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=87577200-6cb1-48a7-b440-53624b12a7b2>

##### Trench 2

In [213]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='apps_score_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 2'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=baf074e0-73c7-4fb0-83a1-8cf1fcc29c25>

In [214]:
sq = """ 
select 
distinct 
loanmaster.customerId customer_id,
r.digitalLoanAccountId,
r.apps_score,
r.lr_score ml_score,
r.dl_score dl_score,
app_cnt_payday_ever,
app_cnt_finance_ever,
app_cnt_competitors_sil_ever,
app_cnt_competitors_ever,
app_cnt_finance_365d,
app_cnt_absence_tag_365d,
app_cnt_competitors_sil_365d,
app_cnt_finance_7d,
app_cnt_rated_for_3plus_ever,
app_cnt_payday_7d,
    case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
    when lower(loanmaster.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-01-31' then 'Dev_Train'
        when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from  risk_mart.applied_sil_new_applicants_loan_20241001_20251124_app_scored r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where apps_score is not null
and trench_category = 'Trench 2'
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


Job ID ef28982f-b811-4c44-b28b-76e134632057 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (11680, 18)


In [215]:
data.columns

Index(['customer_id', 'digitalLoanAccountId', 'apps_score', 'ml_score',
       'dl_score', 'app_cnt_payday_ever', 'app_cnt_finance_ever',
       'app_cnt_competitors_sil_ever', 'app_cnt_competitors_ever',
       'app_cnt_finance_365d', 'app_cnt_absence_tag_365d',
       'app_cnt_competitors_sil_365d', 'app_cnt_finance_7d',
       'app_cnt_rated_for_3plus_ever', 'app_cnt_payday_7d', 'osType',
       'application_date', 'Data_selection'],
      dtype='object')

In [216]:
data.head()

,customer_id,digitalLoanAccountId,apps_score,ml_score,dl_score,app_cnt_payday_ever,app_cnt_finance_ever,app_cnt_competitors_sil_ever,app_cnt_competitors_ever,app_cnt_finance_365d,app_cnt_absence_tag_365d,app_cnt_competitors_sil_365d,app_cnt_finance_7d,app_cnt_rated_for_3plus_ever,app_cnt_payday_7d,osType,application_date,Data_selection
0,1707173,a9cde955-55ed-4314-9c90-81aebb91ef05,0.506238,0.440279,0.560205,12,22.0,5,11,6.0,6.0,3.0,1.0,79.0,1.0,android,2025-02-19,Dev_Test
1,2927194,17126cf8-7534-4cf5-91e7-aa27d3951056,0.537502,0.445018,0.613171,16,25.0,4,10,16.0,24.0,0.0,0.0,54.0,0.0,android,2025-10-30,Dev_Test
2,2676745,6bebfeea-ba85-487e-b1ee-1f7c9743a230,0.713124,0.783601,0.655461,9,18.0,3,7,18.0,6.0,3.0,2.0,34.0,2.0,android,2025-01-11,Dev_Train
3,2443080,60b1caa0-0e92-4b62-982e-fdc4c2ed8047,0.497406,0.529783,0.470916,6,24.0,3,10,16.0,13.0,2.0,0.0,74.0,0.0,android,2025-10-11,Dev_Test
4,2227937,97d09b3f-231a-45a4-b4a0-80c835d592b8,0.435662,0.505808,0.378270,5,21.0,3,8,11.0,5.0,2.0,0.0,43.0,0.0,android,2025-04-20,Dev_Test


In [217]:
feature_column = ['ml_score', 'dl_score', 'app_cnt_payday_ever', 'app_cnt_finance_ever',
       'app_cnt_competitors_sil_ever', 'app_cnt_competitors_ever',
       'app_cnt_finance_365d', 'app_cnt_absence_tag_365d',
       'app_cnt_competitors_sil_365d', 'app_cnt_finance_7d',
       'app_cnt_rated_for_3plus_ever', 'app_cnt_payday_7d',]

dfd = transform_datav2(data, feature_column, a='apps_score', modelDisplayName='apps_score_model_sil', tc='Trench 2', subscription_name = 'Cash November 25 Models') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

the shape of the transformed dataframe is:	 (11680, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,1707173,a9cde955-55ed-4314-9c90-81aebb91ef05,de6de8c6-bf1b-45de-b2f7-3a844c35a483,0.506238,2026-03-18T15:32:20.045521,2026-03-18T15:32:20.045521,apps_score_model_sil,v2,"{""ml_score"": 0.4402788459980401, ""dl_score"": 0...",Cash November 25 Models,a6e9b6d7-5882-4fa8-8092-3a24f15c586f,2026-03-18T15:32:20.045521,{},Trench 2,android,Dev_Test,2025-02-19
1,2927194,17126cf8-7534-4cf5-91e7-aa27d3951056,4d0b03df-a34f-416f-a24b-95e9f1c671e3,0.537502,2026-03-18T15:32:20.045521,2026-03-18T15:32:20.045521,apps_score_model_sil,v2,"{""ml_score"": 0.44501757758608046, ""dl_score"": ...",Cash November 25 Models,45ac6c0d-63db-4051-865d-6a065f82d85f,2026-03-18T15:32:20.045521,{},Trench 2,android,Dev_Test,2025-10-30
2,2676745,6bebfeea-ba85-487e-b1ee-1f7c9743a230,6713094f-9846-4a21-bcc5-a97188554176,0.713124,2026-03-18T15:32:20.046523,2026-03-18T15:32:20.046523,apps_score_model_sil,v2,"{""ml_score"": 0.7836005901498201, ""dl_score"": 0...",Cash November 25 Models,84496f34-1271-44d8-bfec-8ce80322e71d,2026-03-18T15:32:20.046523,{},Trench 2,android,Dev_Train,2025-01-11
3,2443080,60b1caa0-0e92-4b62-982e-fdc4c2ed8047,85d107bb-0637-4b11-87cd-fd7f35acac17,0.497406,2026-03-18T15:32:20.046523,2026-03-18T15:32:20.046523,apps_score_model_sil,v2,"{""ml_score"": 0.529782718682881, ""dl_score"": 0....",Cash November 25 Models,e1eb0dc1-66e9-47aa-9dac-79f8d81d37a7,2026-03-18T15:32:20.046523,{},Trench 2,android,Dev_Test,2025-10-11
4,2227937,97d09b3f-231a-45a4-b4a0-80c835d592b8,e1eb8790-b56f-49d3-a5f4-2252377b387e,0.435662,2026-03-18T15:32:20.046523,2026-03-18T15:32:20.046523,apps_score_model_sil,v2,"{""ml_score"": 0.5058083924244491, ""dl_score"": 0...",Cash November 25 Models,3ee7dbe6-7980-47a5-8643-82a7b9c972eb,2026-03-18T15:32:20.046523,{},Trench 2,android,Dev_Test,2025-04-20


In [218]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,8445,2025-02-01,2025-11-16
1,Dev_Train,3235,2024-10-01,2025-01-31


In [219]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=e38db2d4-75c2-4b94-ab6e-0018cac16006>

##### Trench 3

In [220]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='apps_score_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 3'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=a7acffe1-8b46-4f61-a20b-664615b65b09>

In [221]:
sq = """ 
select 
r.customerId customer_id,
r.digitalLoanAccountId,
r.apps_score,
r.cb_score ml_score,
r.dl_score dl_score,
app_cnt_productivity_ever,
app_cnt_rated_for_3plus_ever, 
app_cnt_books_and_reference_ever,
app_cnt_tools_ever, 
app_median_time_bw_installed_mins_3d,
app_median_time_bw_installed_mins_30d,
app_cnt_communication_ever, 
app_cnt_finance_90d,
app_cnt_absence_tag_180d, 
app_cnt_shopping_ever,
app_cnt_social_ever, 
app_cnt_driver_90d, 
app_cnt_payday_365d,
app_cnt_driver_365d, 
app_cnt_music_and_audio_ever,
app_cnt_finance_180d, 
app_cnt_art_and_design_ever,
app_cnt_gaming_90d, 
app_avg_time_bw_installed_mins_30d,
app_cnt_education_ever,
case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
    when lower(loanmaster.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-01-31' then 'Dev_Train'
        when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from risk_mart.applied_sil_repeat_applicants_loan_20241001_20251124_app_scored r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where apps_score is not null
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID fe276517-8130-45bb-af0c-c49887f8be1b successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (19768, 28)


In [222]:
data.columns

Index(['customer_id', 'digitalLoanAccountId', 'apps_score', 'ml_score',
       'dl_score', 'app_cnt_productivity_ever', 'app_cnt_rated_for_3plus_ever',
       'app_cnt_books_and_reference_ever', 'app_cnt_tools_ever',
       'app_median_time_bw_installed_mins_3d',
       'app_median_time_bw_installed_mins_30d', 'app_cnt_communication_ever',
       'app_cnt_finance_90d', 'app_cnt_absence_tag_180d',
       'app_cnt_shopping_ever', 'app_cnt_social_ever', 'app_cnt_driver_90d',
       'app_cnt_payday_365d', 'app_cnt_driver_365d',
       'app_cnt_music_and_audio_ever', 'app_cnt_finance_180d',
       'app_cnt_art_and_design_ever', 'app_cnt_gaming_90d',
       'app_avg_time_bw_installed_mins_30d', 'app_cnt_education_ever',
       'osType', 'application_date', 'Data_selection'],
      dtype='object')

In [223]:
feature_column = ['ml_score', 'dl_score', 'app_cnt_productivity_ever', 'app_cnt_rated_for_3plus_ever',
       'app_cnt_books_and_reference_ever', 'app_cnt_tools_ever',
       'app_median_time_bw_installed_mins_3d',
       'app_median_time_bw_installed_mins_30d', 'app_cnt_communication_ever',
       'app_cnt_finance_90d', 'app_cnt_absence_tag_180d',
       'app_cnt_shopping_ever', 'app_cnt_social_ever', 'app_cnt_driver_90d',
       'app_cnt_payday_365d', 'app_cnt_driver_365d',
       'app_cnt_music_and_audio_ever', 'app_cnt_finance_180d',
       'app_cnt_art_and_design_ever', 'app_cnt_gaming_90d',
       'app_avg_time_bw_installed_mins_30d', 'app_cnt_education_ever',]

dfd = transform_datav2(data, feature_column, a='apps_score', modelDisplayName='apps_score_model_sil', tc='Trench 3', subscription_name = 'Cash November 25 Models') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

the shape of the transformed dataframe is:	 (19768, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3031736,1baffa6c-8aef-4717-9935-4eb80d56cf0c,5f2102fb-a5da-4b53-a00f-e714a9884b75,0.354866,2026-03-18T15:35:02.966103,2026-03-18T15:35:02.966103,apps_score_model_sil,v2,"{""ml_score"": 0.36518293809898444, ""dl_score"": ...",Cash November 25 Models,9bd033c4-0d92-4332-8ed2-70d684b73be4,2026-03-18T15:35:02.966103,{},Trench 3,android,Dev_Test,2025-06-29
1,2544817,50851b60-9af9-491a-a97d-ae25c49035f6,731caaf5-d762-44cd-9181-b3fe54eaa44a,0.362255,2026-03-18T15:35:02.966103,2026-03-18T15:35:02.966103,apps_score_model_sil,v2,"{""ml_score"": 0.5128092071919207, ""dl_score"": 0...",Cash November 25 Models,f5c28da2-b365-4dc8-8699-6e6a3c4aaa32,2026-03-18T15:35:02.966103,{},Trench 3,android,Dev_Train,2024-12-07
2,2511009,3f0a018e-e421-40f6-af26-029f04105a81,9d7faf2b-4f69-455c-899e-c439f517f278,0.441610,2026-03-18T15:35:02.966103,2026-03-18T15:35:02.966103,apps_score_model_sil,v2,"{""ml_score"": 0.49828466189616877, ""dl_score"": ...",Cash November 25 Models,3fd5532d-6043-47d0-ba84-c32bfd3818ff,2026-03-18T15:35:02.966103,{},Trench 3,android,Dev_Train,2025-01-23
3,3043128,02dc2ed9-a1c2-4389-9717-c1269d6ad4e3,b7ac33d9-c291-428c-82f6-a6a6e3cbc6da,0.252438,2026-03-18T15:35:02.967099,2026-03-18T15:35:02.967099,apps_score_model_sil,v2,"{""ml_score"": 0.3987318371382879, ""dl_score"": 0...",Cash November 25 Models,b8f0f06e-9d9b-4a48-9988-f0e48a28224b,2026-03-18T15:35:02.967099,{},Trench 3,android,Dev_Test,2025-10-03
4,3009495,dc3a437f-b33d-401d-b021-3aebead9256c,a89444ef-9202-4fb5-9a3b-63d9989aa272,0.314420,2026-03-18T15:35:02.967099,2026-03-18T15:35:02.967099,apps_score_model_sil,v2,"{""ml_score"": 0.42422612584765734, ""dl_score"": ...",Cash November 25 Models,feaf00ed-f2d6-4756-8378-fca9105d06eb,2026-03-18T15:35:02.967099,{},Trench 3,android,Dev_Test,2025-04-12


In [224]:
dfd['customerId'] = pd.to_numeric(dfd['customerId'], errors='coerce')

In [225]:
dfd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19768 entries, 0 to 19767
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   customerId            19768 non-null  int64  
 1   digitalLoanAccountId  19768 non-null  object 
 2   crifApplicationId     19768 non-null  object 
 3   prediction            19768 non-null  float64
 4   start_time            19768 non-null  object 
 5   end_time              19768 non-null  object 
 6   modelDisplayName      19768 non-null  object 
 7   modelVersionId        19768 non-null  object 
 8   calcFeature           19768 non-null  object 
 9   subscription_name     19768 non-null  object 
 10  message_id            19768 non-null  object 
 11  publish_time          19768 non-null  object 
 12  attributes            19768 non-null  object 
 13  trenchCategory        19768 non-null  object 
 14  deviceOs              19768 non-null  object 
 15  Data_selection     

In [226]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,14581,2025-02-01,2025-11-16
1,Dev_Train,5187,2024-10-01,2025-01-31


In [227]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=f33f4ea5-cd48-421c-a400-a78d08ce6aac>

##### Beta SIL Demo Score

##### Trench 1

In [228]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='beta_demo_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 1'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=989150cb-0377-4a96-ad75-316c10fb97f8>

In [229]:
sq = """select 
distinct 
loanmaster.customerId customer_id,
r.digitalLoanAccountId,
r.s_demo_score,
ln_vas_opted_flag,
ln_doc_type_rolled,
ln_industry_new_cat_bin,
ln_marital_status,
ln_age,
ln_education_level,
ln_cnt_dependents,
ln_ref2_type,
ln_loan_level_user_type,
ln_ref1_type,
ln_name_email_match_score,
ln_telconame,
ln_city_cat,
ln_brand_bin,
ln_apply_Is_Weekend,
r.ln_os_type osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-01-31' then 'Dev_Train'
        when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from worktable_data_analysis.sil_beta_demo_all_applied_backscored_20240801_20251015 r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where s_demo_score is not null
and trench_category = 'Trench 1'
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 5836386c-87d3-44a8-9e53-7800491a905f successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (345905, 21)


In [230]:
data.columns

Index(['customer_id', 'digitalLoanAccountId', 's_demo_score',
       'ln_vas_opted_flag', 'ln_doc_type_rolled', 'ln_industry_new_cat_bin',
       'ln_marital_status', 'ln_age', 'ln_education_level',
       'ln_cnt_dependents', 'ln_ref2_type', 'ln_loan_level_user_type',
       'ln_ref1_type', 'ln_name_email_match_score', 'ln_telconame',
       'ln_city_cat', 'ln_brand_bin', 'ln_apply_Is_Weekend', 'osType',
       'application_date', 'Data_selection'],
      dtype='object')

In [231]:
feature_column = [ 'ln_vas_opted_flag', 'ln_doc_type_rolled', 'ln_industry_new_cat_bin',
       'ln_marital_status', 'ln_age', 'ln_education_level',
       'ln_cnt_dependents', 'ln_ref2_type', 'ln_loan_level_user_type',
       'ln_ref1_type', 'ln_name_email_match_score', 'ln_telconame',
       'ln_city_cat', 'ln_brand_bin', 'ln_apply_Is_Weekend', 'osType']

dfd = transform_datav2(data, feature_column, a='s_demo_score', modelDisplayName='beta_demo_model_sil', tc='Trench 1', subscription_name = 'Cash November 25 Models') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

the shape of the transformed dataframe is:	 (345905, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3421024,8c1bd0f0-4c29-44cc-87cd-cf9b1efe6318,b0808e9b-6e5d-4934-9650-de5223c14071,0.265796,2026-03-18T15:41:45.332885,2026-03-18T15:41:45.332885,beta_demo_model_sil,v2,"{""ln_vas_opted_flag"": ""0"", ""ln_doc_type_rolled...",Cash November 25 Models,d05e76e0-8407-4264-a9a9-a91298c20fe0,2026-03-18T15:41:45.332885,{},Trench 1,Android,Dev_Test,2025-05-05
1,3519014,b8bd79bc-c2e7-4ba6-82bf-a107f50f960a,995da911-c6b6-4bb1-8e0d-eee67cdf50d4,0.382915,2026-03-18T15:41:45.334403,2026-03-18T15:41:45.334403,beta_demo_model_sil,v2,"{""ln_vas_opted_flag"": ""0"", ""ln_doc_type_rolled...",Cash November 25 Models,0d242533-a5d0-486f-8aa9-6a6c02d43001,2026-03-18T15:41:45.334403,{},Trench 1,Android,Dev_Test,2025-06-25
2,3429313,2aa4f96d-ea64-4deb-a814-9319c6253b7d,f5892b6e-3e4b-425b-9642-1d78fa95bf3e,0.420961,2026-03-18T15:41:45.334403,2026-03-18T15:41:45.334403,beta_demo_model_sil,v2,"{""ln_vas_opted_flag"": ""0"", ""ln_doc_type_rolled...",Cash November 25 Models,deed2106-fedf-4b55-94f0-cd4b45358047,2026-03-18T15:41:45.334403,{},Trench 1,Android,Dev_Test,2025-05-10
3,3543638,ee83fe52-fd9e-484e-a85e-9d9dff37a667,51a5231a-6bf5-4cd1-98d7-d4584f113dad,0.350725,2026-03-18T15:41:45.335422,2026-03-18T15:41:45.335422,beta_demo_model_sil,v2,"{""ln_vas_opted_flag"": ""0"", ""ln_doc_type_rolled...",Cash November 25 Models,22489ca6-7b56-466b-90a2-6bd44b0d5dd5,2026-03-18T15:41:45.335422,{},Trench 1,iOS,Dev_Test,2025-07-07
4,3518725,d9e942b4-bc9c-42c8-8f57-8962f5c21f4c,324d75d7-7ce7-41a7-ab89-18eb9d4c593d,0.264026,2026-03-18T15:41:45.335422,2026-03-18T15:41:45.335422,beta_demo_model_sil,v2,"{""ln_vas_opted_flag"": ""0"", ""ln_doc_type_rolled...",Cash November 25 Models,2bf1cdc0-0094-4232-8428-810d00baa110,2026-03-18T15:41:45.335422,{},Trench 1,Android,Dev_Test,2025-06-25


In [232]:
dfd['customerId'] = pd.to_numeric(dfd['customerId'], errors='coerce')

In [233]:
dfd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 345905 entries, 0 to 345904
Data columns (total 17 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   customerId            345905 non-null  int64  
 1   digitalLoanAccountId  345905 non-null  object 
 2   crifApplicationId     345905 non-null  object 
 3   prediction            345905 non-null  float64
 4   start_time            345905 non-null  object 
 5   end_time              345905 non-null  object 
 6   modelDisplayName      345905 non-null  object 
 7   modelVersionId        345905 non-null  object 
 8   calcFeature           345905 non-null  object 
 9   subscription_name     345905 non-null  object 
 10  message_id            345905 non-null  object 
 11  publish_time          345905 non-null  object 
 12  attributes            345905 non-null  object 
 13  trenchCategory        345905 non-null  object 
 14  deviceOs              345905 non-null  object 
 15  

In [234]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,185960,2025-02-01,2025-11-15
1,Dev_Train,159945,2024-08-01,2025-01-31


In [235]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=27bb11e9-09d3-4976-8b45-f9057fd69162>

##### Trench 2

In [236]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='beta_demo_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 2'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=b326f618-6202-4832-b577-611e97e2a23f>

In [237]:
sq = """select 
distinct 
loanmaster.customerId customer_id,
r.digitalLoanAccountId,
r.s_demo_score,
ln_vas_opted_flag,
ln_doc_type_rolled,
ln_industry_new_cat_bin,
ln_marital_status,
ln_age,
ln_education_level,
ln_cnt_dependents,
ln_ref2_type,
ln_loan_level_user_type,
ln_ref1_type,
ln_name_email_match_score,
ln_telconame,
ln_city_cat,
ln_brand_bin,
ln_apply_Is_Weekend,
r.ln_os_type osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-01-31' then 'Dev_Train'
        when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from worktable_data_analysis.sil_beta_demo_all_applied_backscored_20240801_20251015 r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where s_demo_score is not null
and trench_category = 'Trench 2'
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID d81fbd97-b27a-48a3-abec-270ad79b64dd successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (13192, 21)


In [238]:
data.columns

Index(['customer_id', 'digitalLoanAccountId', 's_demo_score',
       'ln_vas_opted_flag', 'ln_doc_type_rolled', 'ln_industry_new_cat_bin',
       'ln_marital_status', 'ln_age', 'ln_education_level',
       'ln_cnt_dependents', 'ln_ref2_type', 'ln_loan_level_user_type',
       'ln_ref1_type', 'ln_name_email_match_score', 'ln_telconame',
       'ln_city_cat', 'ln_brand_bin', 'ln_apply_Is_Weekend', 'osType',
       'application_date', 'Data_selection'],
      dtype='object')

In [239]:
feature_column = [ 'ln_vas_opted_flag', 'ln_doc_type_rolled', 'ln_industry_new_cat_bin',
       'ln_marital_status', 'ln_age', 'ln_education_level',
       'ln_cnt_dependents', 'ln_ref2_type', 'ln_loan_level_user_type',
       'ln_ref1_type', 'ln_name_email_match_score', 'ln_telconame',
       'ln_city_cat', 'ln_brand_bin', 'ln_apply_Is_Weekend', 'osType']

dfd = transform_datav2(data, feature_column, a='s_demo_score', modelDisplayName='beta_demo_model_sil', tc='Trench 2', subscription_name = 'Cash November 25 Models') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

the shape of the transformed dataframe is:	 (13192, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,1715371,36ecec46-108b-4a48-95bf-8ea2b0403bb8,d7be801b-fa48-4969-90d0-637e3386e5dc,0.413285,2026-03-18T16:03:33.946146,2026-03-18T16:03:33.946146,beta_demo_model_sil,v2,"{""ln_vas_opted_flag"": ""1"", ""ln_doc_type_rolled...",Cash November 25 Models,055e2488-6473-4292-84fe-8e07c861fda8,2026-03-18T16:03:33.946146,{},Trench 2,Android,Dev_Test,2025-03-27
1,3339603,3bca4adc-8f77-4861-986c-2298a75fd9f6,987bff86-4231-49ab-b648-211cbaac85cc,0.479304,2026-03-18T16:03:33.946146,2026-03-18T16:03:33.946146,beta_demo_model_sil,v2,"{""ln_vas_opted_flag"": ""0"", ""ln_doc_type_rolled...",Cash November 25 Models,5959249f-77bf-4e2c-9c86-c1c0ea00b3a9,2026-03-18T16:03:33.946146,{},Trench 2,Android,Dev_Test,2025-06-27
2,3154933,6e64b310-a63a-41fc-80f1-90ae6d5e517a,b6c6d1e5-7a28-4f1b-a6fc-3454a26f681d,0.486822,2026-03-18T16:03:33.946146,2026-03-18T16:03:33.946146,beta_demo_model_sil,v2,"{""ln_vas_opted_flag"": ""1"", ""ln_doc_type_rolled...",Cash November 25 Models,8b6de378-9bd7-4717-aab8-f3acfac5d6db,2026-03-18T16:03:33.946146,{},Trench 2,Android,Dev_Test,2025-07-17
3,3276135,fdd2a0d0-a449-4c06-ae6e-382e2d6e77aa,25a637f6-5dd9-44ae-93b5-237abe2bb9b9,0.698747,2026-03-18T16:03:33.947158,2026-03-18T16:03:33.947158,beta_demo_model_sil,v2,"{""ln_vas_opted_flag"": ""1"", ""ln_doc_type_rolled...",Cash November 25 Models,b0f16cb9-155a-4cc4-bd63-6d81a618128e,2026-03-18T16:03:33.947158,{},Trench 2,iOS,Dev_Test,2025-03-28
4,2333760,7a7f574c-76ae-4c08-83d3-9046f17253ac,775948f0-c086-416d-abcc-1a70fef9d149,0.383582,2026-03-18T16:03:33.947158,2026-03-18T16:03:33.947158,beta_demo_model_sil,v2,"{""ln_vas_opted_flag"": ""1"", ""ln_doc_type_rolled...",Cash November 25 Models,56c1b409-3623-4d4f-9c65-387823a257ca,2026-03-18T16:03:33.947158,{},Trench 2,Android,Dev_Test,2025-06-04


In [240]:
dfd['customerId'] = pd.to_numeric(dfd['customerId'], errors='coerce')

In [241]:
dfd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13192 entries, 0 to 13191
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   customerId            13192 non-null  int64  
 1   digitalLoanAccountId  13192 non-null  object 
 2   crifApplicationId     13192 non-null  object 
 3   prediction            13192 non-null  float64
 4   start_time            13192 non-null  object 
 5   end_time              13192 non-null  object 
 6   modelDisplayName      13192 non-null  object 
 7   modelVersionId        13192 non-null  object 
 8   calcFeature           13192 non-null  object 
 9   subscription_name     13192 non-null  object 
 10  message_id            13192 non-null  object 
 11  publish_time          13192 non-null  object 
 12  attributes            13192 non-null  object 
 13  trenchCategory        13192 non-null  object 
 14  deviceOs              13192 non-null  object 
 15  Data_selection     

In [242]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,8408,2025-02-01,2025-10-15
1,Dev_Train,4784,2024-08-01,2025-01-31


In [243]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=1ced3598-b252-4686-9ea2-598a271ff748>

##### Trench 3

In [244]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='beta_demo_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 3'
;"""

client.query(sq)

QueryJob<project=prj-prod-dataplatform, location=asia-southeast1, id=206cf727-0de1-4c8d-b5a1-28f0235e2f48>

In [245]:
sq = """select 
distinct 
loanmaster.customerId customer_id,
r.digitalLoanAccountId,
r.s_demo_score,
ln_vas_opted_flag,
ln_doc_type_rolled,
ln_industry_new_cat_bin,
ln_marital_status,
ln_age,
ln_education_level,
ln_cnt_dependents,
ln_ref2_type,
ln_loan_level_user_type,
ln_ref1_type,
ln_name_email_match_score,
ln_telconame,
ln_city_cat,
ln_brand_bin,
ln_apply_Is_Weekend,
r.ln_os_type osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-01-31' then 'Dev_Train'
        when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from worktable_data_analysis.sil_beta_demo_all_applied_backscored_20240801_20251015 r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where s_demo_score is not null
and trench_category = 'Trench 3'
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-11-17'
 ;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 4bdfb886-6b68-46a2-a8bd-6bdde2e8a09a successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (13866, 21)


In [246]:
data.columns

Index(['customer_id', 'digitalLoanAccountId', 's_demo_score',
       'ln_vas_opted_flag', 'ln_doc_type_rolled', 'ln_industry_new_cat_bin',
       'ln_marital_status', 'ln_age', 'ln_education_level',
       'ln_cnt_dependents', 'ln_ref2_type', 'ln_loan_level_user_type',
       'ln_ref1_type', 'ln_name_email_match_score', 'ln_telconame',
       'ln_city_cat', 'ln_brand_bin', 'ln_apply_Is_Weekend', 'osType',
       'application_date', 'Data_selection'],
      dtype='object')

In [247]:
feature_column = [ 'ln_vas_opted_flag', 'ln_doc_type_rolled', 'ln_industry_new_cat_bin',
       'ln_marital_status', 'ln_age', 'ln_education_level',
       'ln_cnt_dependents', 'ln_ref2_type', 'ln_loan_level_user_type',
       'ln_ref1_type', 'ln_name_email_match_score', 'ln_telconame',
       'ln_city_cat', 'ln_brand_bin', 'ln_apply_Is_Weekend', 'osType']

dfd = transform_datav2(data, feature_column, a='s_demo_score', modelDisplayName='beta_demo_model_sil', tc='Trench 3', subscription_name = 'Cash November 25 Models') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3055653,77507432-21b9-4f72-aade-115001a56b30,de5699b2-8e96-481a-9eeb-e5a950c7a60a,0.361895,2026-03-18T16:04:58.687178,2026-03-18T16:04:58.687178,beta_demo_model_sil,v2,"{""ln_vas_opted_flag"": ""1"", ""ln_doc_type_rolled...",Cash November 25 Models,95222195-a159-4c0d-8d0c-708f0de3a8c5,2026-03-18T16:04:58.687178,{},Trench 3,Android,Dev_Test,2025-03-08
1,3351398,4617665e-3fc7-4059-b8c0-eb62556653bb,8c1acb9f-65b1-4d19-83c1-0304fe3d92f1,0.099106,2026-03-18T16:04:58.687178,2026-03-18T16:04:58.687178,beta_demo_model_sil,v2,"{""ln_vas_opted_flag"": ""0"", ""ln_doc_type_rolled...",Cash November 25 Models,27d7c229-74ec-446a-a1d1-469bed95617e,2026-03-18T16:04:58.687178,{},Trench 3,Android,Dev_Test,2025-10-12
2,3021398,723b49a9-a41b-44b5-a3c9-c7d694174f18,a24f10e5-9c89-4809-9bf3-485a511681d9,0.063125,2026-03-18T16:04:58.687178,2026-03-18T16:04:58.687178,beta_demo_model_sil,v2,"{""ln_vas_opted_flag"": ""0"", ""ln_doc_type_rolled...",Cash November 25 Models,063537e9-c76c-4d76-aba1-0b3308f756c5,2026-03-18T16:04:58.687178,{},Trench 3,Android,Dev_Test,2025-07-10
3,2438022,a60365db-f0ce-4992-bff4-f959928986ac,059e2d28-2536-4b0e-b593-06d8e8c5dd10,0.082685,2026-03-18T16:04:58.687178,2026-03-18T16:04:58.687178,beta_demo_model_sil,v2,"{""ln_vas_opted_flag"": ""0"", ""ln_doc_type_rolled...",Cash November 25 Models,b7368525-ceff-4a53-b655-8f71267a0e11,2026-03-18T16:04:58.687178,{},Trench 3,Android,Dev_Test,2025-03-14
4,2892937,7e876b67-0696-4333-a781-254ad1e0a007,89f4f45d-6484-498d-aafb-0eec9ff99dd3,0.121096,2026-03-18T16:04:58.687178,2026-03-18T16:04:58.687178,beta_demo_model_sil,v2,"{""ln_vas_opted_flag"": ""0"", ""ln_doc_type_rolled...",Cash November 25 Models,818e6713-6594-40c9-82e1-dfa2747c3b60,2026-03-18T16:04:58.687178,{},Trench 3,Android,Dev_Test,2025-04-09


In [248]:
dfd['customerId'] = pd.to_numeric(dfd['customerId'], errors='coerce')

In [249]:
dfd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13866 entries, 0 to 13865
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   customerId            13866 non-null  int64  
 1   digitalLoanAccountId  13866 non-null  object 
 2   crifApplicationId     13866 non-null  object 
 3   prediction            13866 non-null  float64
 4   start_time            13866 non-null  object 
 5   end_time              13866 non-null  object 
 6   modelDisplayName      13866 non-null  object 
 7   modelVersionId        13866 non-null  object 
 8   calcFeature           13866 non-null  object 
 9   subscription_name     13866 non-null  object 
 10  message_id            13866 non-null  object 
 11  publish_time          13866 non-null  object 
 12  attributes            13866 non-null  object 
 13  trenchCategory        13866 non-null  object 
 14  deviceOs              13866 non-null  object 
 15  Data_selection     

In [250]:

result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,8346,2025-02-01,2025-10-15
1,Dev_Train,5520,2024-08-01,2025-01-31


In [251]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=bd96a3e3-e736-44a0-9621-cb4d2b0d15df>

# apps_score_cash

## Trench 1

risk_mart.quick_applied_aug2024_nov2025_app_scored_extended only had cb_score and dl_score and was lacking other features used for cb_score, Oleh gave another table select * from  risk_mart.quick_applied_aug2024_nov2025_app_scored_extended_v2;

In [252]:
sq = """ 
WITH
  base AS (
    SELECT
      r.customerId,
      r.digitalLoanAccountId,
      r.application_submission_date,
      CASE
        WHEN r.trenchCategory LIKE 'T1-A' THEN 'Trench 1'
        WHEN r.trenchCategory LIKE 'T2-A' THEN 'Trench 2'
        WHEN r.trenchCategory LIKE 'T3-A' THEN 'Trench 3'
        END trenchCategory,
        r.apps_score,
        r.cb_score ml_score,
        r.dl_score,
        r.app_cnt_absence_tag_90d,	
        r.app_cnt_rated_for_18plus_ever,	
        r.app_last_payday_install_to_apply_days,	
        r.app_cnt_lifestyle_ever,
        r.app_median_time_bw_installed_mins_ever,
        r.app_cnt_payday_ever,
        r.app_cnt_food_and_drink_ever,
        r.app_cnt_gaming_180d,
        r.app_vel_payday_30_over_365,
        r.app_first_app_cat,
        lower(r.deviceOs) osType,
      date(
        IF(
          lmt.new_loan_type = 'Flex-up',
          lmt.startApplyDateTime,
          lmt.termsAndConditionsSubmitDateTime))
        application_date,
      CASE
        WHEN dataset LIKE 'Dev_Train' THEN 'Dev_Train'
        WHEN dataset LIKE 'Dev_Test' THEN 'Dev_Test'
        WHEN dataset LIKE 'OOS' THEN 'Dev_Test'
        END Data_selection
    FROM risk_mart.quick_applied_aug2024_nov2025_app_scored_extended_v2 r
    LEFT JOIN `risk_credit_mis.loan_master_table` lmt
      ON lmt.digitalLoanAccountId = r.digitalLoanAccountId
  )
SELECT
  customerId customer_id,
  digitalLoanAccountId,
  application_submission_date,
  trenchCategory,
  apps_score,
  ml_score,
  dl_score,
  app_cnt_absence_tag_90d,	
  app_cnt_rated_for_18plus_ever,	
  app_last_payday_install_to_apply_days,	
  app_cnt_lifestyle_ever,
  app_median_time_bw_installed_mins_ever,
  app_cnt_payday_ever,
  app_cnt_food_and_drink_ever,
  app_cnt_gaming_180d,
  app_vel_payday_30_over_365,
  app_first_app_cat,
  osType,
  application_date,
  Data_selection
FROM base
WHERE
  trenchCategory = 'Trench 1'
  AND apps_score IS NOT NULL;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 6575c683-c7d6-455c-8432-b6fc427b7f92 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (310159, 20)


In [253]:
data.columns

Index(['customer_id', 'digitalLoanAccountId', 'application_submission_date',
       'trenchCategory', 'apps_score', 'ml_score', 'dl_score',
       'app_cnt_absence_tag_90d', 'app_cnt_rated_for_18plus_ever',
       'app_last_payday_install_to_apply_days', 'app_cnt_lifestyle_ever',
       'app_median_time_bw_installed_mins_ever', 'app_cnt_payday_ever',
       'app_cnt_food_and_drink_ever', 'app_cnt_gaming_180d',
       'app_vel_payday_30_over_365', 'app_first_app_cat', 'osType',
       'application_date', 'Data_selection'],
      dtype='object')

In [254]:
feature_column = ['apps_score', 'ml_score', 'dl_score',
       'app_cnt_absence_tag_90d', 'app_cnt_rated_for_18plus_ever',
       'app_last_payday_install_to_apply_days', 'app_cnt_lifestyle_ever',
       'app_median_time_bw_installed_mins_ever', 'app_cnt_payday_ever',
       'app_cnt_food_and_drink_ever', 'app_cnt_gaming_180d',
       'app_vel_payday_30_over_365', 'app_first_app_cat',]

dfd = transform_data_v1_1(data, feature_column, a='apps_score', modelDisplayName='apps_score_cash', tc='Trench 1', subscription_name = 'Cash December APPs Models Upgrade') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.info()

the shape of the transformed dataframe is:	 (310159, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 310159 entries, 0 to 310158
Data columns (total 17 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   customerId            310159 non-null  int64  
 1   digitalLoanAccountId  310159 non-null  object 
 2   crifApplicationId     310159 non-null  object 
 3   prediction            310159 non-null  float64
 4   start_time            310159 non-null  object 
 5   end_time              310159 non-null  object 
 6   modelDisplayName      310159 non-null  object 
 7   modelVersionId        310159 non-null  object 
 8   calcFeature           310159 non-null  object 
 9   subscription_name     310159 non-null  object 
 10  message_id            310159 non-null  object 
 11  publish_time          310159 non-null  object 
 12  attributes            310159 non-null  object 
 13  trenchCategory        310159 non-null  object 


In [255]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,90626,2025-07-01,2026-03-07
1,Dev_Train,219533,2024-08-13,2025-06-30


In [256]:
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3305024,8b5f9ae3-508c-4cbd-a379-68e331a38a21,d790ed47-4311-42ab-96ba-764093416806,0.583970,2026-03-18T17:27:14.941908,2026-03-18T17:27:14.941908,apps_score_cash,v1.1,"{""apps_score"": 0.583969557251809, ""ml_score"": ...",Cash December APPs Models Upgrade,8f7f18f3-3342-41b1-85a1-af797b82d95d,2026-03-18T17:27:14.941908,{},Trench 1,android,Dev_Train,2025-03-06
1,3058192,6418dbbc-c7ce-4c6f-b9e8-7bfeea554c0a,cf9653fc-b768-4d19-8650-2f68e5416cff,0.543028,2026-03-18T17:27:14.942915,2026-03-18T17:27:14.942915,apps_score_cash,v1.1,"{""apps_score"": 0.5430281136914239, ""ml_score"":...",Cash December APPs Models Upgrade,e06d61af-f415-4ca3-ac9a-57abb05f757f,2026-03-18T17:27:14.942915,{},Trench 1,android,Dev_Train,2024-11-27
2,3627484,f736004c-0358-4b77-b69f-ad8da6ebd8e5,987a9b6c-e684-406a-81e1-583b8723bc78,0.564354,2026-03-18T17:27:14.943932,2026-03-18T17:27:14.943932,apps_score_cash,v1.1,"{""apps_score"": 0.5643544423571051, ""ml_score"":...",Cash December APPs Models Upgrade,c82b5133-5be4-4a4c-9d47-cbd1c799166d,2026-03-18T17:27:14.943932,{},Trench 1,android,Dev_Test,2025-08-17
3,3791383,070a48cd-62fb-49e9-bbc2-f76b43b2cdf4,ce3b880c-1ad7-4329-8e68-a69263085a70,0.545836,2026-03-18T17:27:14.943932,2026-03-18T17:27:14.943932,apps_score_cash,v1.1,"{""apps_score"": 0.545835865772015, ""ml_score"": ...",Cash December APPs Models Upgrade,aee479b5-2a57-4aa4-a5c6-fee26f4ed93b,2026-03-18T17:27:14.943932,{},Trench 1,android,Dev_Test,2025-11-04
4,3369778,75703543-fa16-4ca5-b0dc-f62ca7ce493a,83791068-e198-4470-8d8c-97189e3ddb28,0.523562,2026-03-18T17:27:14.944932,2026-03-18T17:27:14.944932,apps_score_cash,v1.1,"{""apps_score"": 0.5235617085681417, ""ml_score"":...",Cash December APPs Models Upgrade,eca46219-6523-4191-9389-8f0fd9675f7a,2026-03-18T17:27:14.944932,{},Trench 1,android,Dev_Train,2025-04-09


In [257]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_application"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=5b5496f1-fbd0-4e17-b739-150fbf269061>

## Trench 2

In [258]:
sq = """ 
WITH
  base AS (
    SELECT
      r.customerId,
      r.digitalLoanAccountId,
      r.application_submission_date,
      CASE
        WHEN r.trenchCategory LIKE 'T1-A' THEN 'Trench 1'
        WHEN r.trenchCategory LIKE 'T2-A' THEN 'Trench 2'
        WHEN r.trenchCategory LIKE 'T3-A' THEN 'Trench 3'
        END trenchCategory,
        r.apps_score,
        r.cb_score ml_score,
        r.dl_score,
        r.app_cnt_absence_tag_90d,	
        r.app_cnt_rated_for_18plus_ever,	
        r.app_last_payday_install_to_apply_days,	
        r.app_cnt_lifestyle_ever,
        r.app_median_time_bw_installed_mins_ever,
        r.app_cnt_payday_ever,
        r.app_cnt_food_and_drink_ever,
        r.app_cnt_gaming_180d,
        r.app_vel_payday_30_over_365,
        r.app_first_app_cat,
    lower(r.deviceOs)  osType,
      date(
        IF(
          lmt.new_loan_type = 'Flex-up',
          lmt.startApplyDateTime,
          lmt.termsAndConditionsSubmitDateTime))
        application_date,
      CASE
        WHEN dataset LIKE 'Dev_Train' THEN 'Dev_Train'
        WHEN dataset LIKE 'Dev_Test' THEN 'Dev_Test'
        WHEN dataset LIKE 'OOS' THEN 'Dev_Test'
        END Data_selection
    FROM risk_mart.quick_applied_aug2024_nov2025_app_scored_extended_v2 r
    LEFT JOIN `risk_credit_mis.loan_master_table` lmt
      ON lmt.digitalLoanAccountId = r.digitalLoanAccountId
  )
SELECT
  customerId customer_id,
  digitalLoanAccountId,
  application_submission_date,
  trenchCategory,
  apps_score,
  ml_score,
  dl_score,
  app_cnt_absence_tag_90d,	
  app_cnt_rated_for_18plus_ever,	
  app_last_payday_install_to_apply_days,	
  app_cnt_lifestyle_ever,
  app_median_time_bw_installed_mins_ever,
  app_cnt_payday_ever,
  app_cnt_food_and_drink_ever,
  app_cnt_gaming_180d,
  app_vel_payday_30_over_365,
  app_first_app_cat,
  osType,
  application_date,
  Data_selection
FROM base
WHERE
  trenchCategory = 'Trench 2'
  AND apps_score IS NOT NULL;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 87414a2b-a1e6-49ee-be81-ec7545f65f48 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (120001, 20)


In [259]:
feature_column = ['apps_score', 'ml_score', 'dl_score',
       'app_cnt_absence_tag_90d', 'app_cnt_rated_for_18plus_ever',
       'app_last_payday_install_to_apply_days', 'app_cnt_lifestyle_ever',
       'app_median_time_bw_installed_mins_ever', 'app_cnt_payday_ever',
       'app_cnt_food_and_drink_ever', 'app_cnt_gaming_180d',
       'app_vel_payday_30_over_365', 'app_first_app_cat',]

dfd = transform_data_v1_1(data, feature_column, a='apps_score', modelDisplayName='apps_score_cash', tc='Trench 2', subscription_name = 'Cash December APPs Models Upgrade') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.info()

the shape of the transformed dataframe is:	 (120001, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120001 entries, 0 to 120000
Data columns (total 17 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   customerId            120001 non-null  int64  
 1   digitalLoanAccountId  120001 non-null  object 
 2   crifApplicationId     120001 non-null  object 
 3   prediction            120001 non-null  float64
 4   start_time            120001 non-null  object 
 5   end_time              120001 non-null  object 
 6   modelDisplayName      120001 non-null  object 
 7   modelVersionId        120001 non-null  object 
 8   calcFeature           120001 non-null  object 
 9   subscription_name     120001 non-null  object 
 10  message_id            120001 non-null  object 
 11  publish_time          120001 non-null  object 
 12  attributes            120001 non-null  object 
 13  trenchCategory        120001 non-null  object 


In [260]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,47650,2025-07-01,2025-12-03
1,Dev_Train,72351,2024-08-13,2025-06-30


In [261]:
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,2914517,dbc1ab14-f1fa-43b0-9405-864ecff7f099,e3bdda4f-fbff-49d9-aa44-17c66a880f3a,0.511849,2026-03-18T17:31:47.855960,2026-03-18T17:31:47.855960,apps_score_cash,v1.1,"{""apps_score"": 0.5118485957513261, ""ml_score"":...",Cash December APPs Models Upgrade,05242c4b-1965-4012-813f-694000a0f27f,2026-03-18T17:31:47.855960,{},Trench 2,android,Dev_Test,2025-07-24
1,2945279,630e3549-f3f4-4fa0-b7c0-4d9f11f4b0a7,91acdc82-e347-4df5-a6f5-29a30dc84ce5,0.488383,2026-03-18T17:31:47.855960,2026-03-18T17:31:47.855960,apps_score_cash,v1.1,"{""apps_score"": 0.4883828121888534, ""ml_score"":...",Cash December APPs Models Upgrade,85e3f556-4aee-4195-9320-3b228c62ec10,2026-03-18T17:31:47.855960,{},Trench 2,android,Dev_Test,2025-11-15
2,1740555,724fad59-4fcd-4374-9dc4-cf14de7bcea7,7fa4efab-1851-43f5-be73-0ba00787738a,0.522166,2026-03-18T17:31:47.856512,2026-03-18T17:31:47.856512,apps_score_cash,v1.1,"{""apps_score"": 0.5221657056438087, ""ml_score"":...",Cash December APPs Models Upgrade,68802daf-d6ce-47a2-9493-bcc16c7b2bfb,2026-03-18T17:31:47.856512,{},Trench 2,android,Dev_Train,2024-12-29
3,2639541,eb819725-a96c-4474-b967-79e86fe442e0,094af963-8f72-408f-acb1-38a74b8e8d51,0.522118,2026-03-18T17:31:47.856512,2026-03-18T17:31:47.856512,apps_score_cash,v1.1,"{""apps_score"": 0.5221179661612932, ""ml_score"":...",Cash December APPs Models Upgrade,994996e6-6e12-4e85-baf4-047abecdb7c8,2026-03-18T17:31:47.856512,{},Trench 2,android,Dev_Train,2024-10-06
4,3080979,6ed43790-22dc-4719-bfad-fe8c5ece5d77,42ac084b-5bde-4cec-bd74-cf60dde8df0b,0.547454,2026-03-18T17:31:47.856512,2026-03-18T17:31:47.856512,apps_score_cash,v1.1,"{""apps_score"": 0.5474538342977491, ""ml_score"":...",Cash December APPs Models Upgrade,fb3fbaf7-e28b-4af3-b20f-6443e854e14c,2026-03-18T17:31:47.856512,{},Trench 2,android,Dev_Test,2025-07-07


In [262]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=432e10ea-b0e8-4fd2-be7d-9dd4bd4acd08>

# Beta_stack_Model_cash

## Trench 1

In [263]:
sq = """ 
WITH
  base AS (
    SELECT
      r.customer_id,
      r.digitalLoanAccountId,
      r.ln_appln_submit_datetime,
      r.trench_category,
      r.demo_score,
      r.apps_score,
      r.credo_score,
      r.stack_score,
      lower(r.ln_os_type) osType,
      date(
        IF(
          lmt.new_loan_type = 'Flex-up',
          lmt.startApplyDateTime,
          lmt.termsAndConditionsSubmitDateTime))
        application_date,
      -- CASE
      --   WHEN dev_split LIKE 'Dev_Train' THEN 'Dev_Train'
      --   WHEN dev_split LIKE 'Dev_Test' THEN 'Dev_Test'
      --   ELSE dev_split
      --   END Data_selection
      case when lower(r.ln_os_type) like '%android%' and date(ln_appln_submit_datetime) >= '2024-10-01' and date(ln_appln_submit_datetime) < '2025-06-01' then 'Dev_Train'
           when lower(r.ln_os_type) like '%android%' and date(ln_appln_submit_datetime) >= '2025-06-01' and date(ln_appln_submit_datetime) <= '2025-11-30' then 'Dev_Test'
           when lower(r.ln_os_type) like '%ios%' and date(ln_appln_submit_datetime) >= '2024-10-01' and date(ln_appln_submit_datetime) <= '2025-02-28' then 'Dev_Train'
           when lower(r.ln_os_type) like '%ios%' and date(ln_appln_submit_datetime) > '2025-02-28' and date(ln_appln_submit_datetime) <= '2025-11-30' then 'Dev_Test' 
           end Data_selection
      from 
      `worktable_data_analysis.cash_beta_trench1_applied_loans_backscored_20241001_20251130`       r
    LEFT JOIN `risk_credit_mis.loan_master_table` lmt
      ON lmt.digitalLoanAccountId = r.digitalLoanAccountId
    where stack_score is not null
      )
      ,
b1 as 
(SELECT 
customer_id,
digitalLoanAccountId,
ln_appln_submit_datetime,
trench_category,
demo_score,
apps_score,
credo_score,
stack_score Beta_cash_stack_score,
osType,
application_date,
Data_selection,
FROM base
WHERE Data_selection IS NOT NULL
)
select * from b1 

"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 2c5ba4e6-9397-417d-829c-20f0df8b3ab6 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (382727, 11)


In [264]:
feature_column = ['demo_score', 'apps_score', 'credo_score', 'Beta_cash_stack_score']

dfd = transform_data_v1_1(data, feature_column, a='Beta_cash_stack_score', modelDisplayName='Beta-Cash-Stack-Model', tc='Trench 1', subscription_name = 'Cash December APPs Models Upgrade') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.info()

the shape of the transformed dataframe is:	 (382727, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 382727 entries, 0 to 382726
Data columns (total 17 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   customerId            382727 non-null  int64  
 1   digitalLoanAccountId  382727 non-null  object 
 2   crifApplicationId     382727 non-null  object 
 3   prediction            382727 non-null  float64
 4   start_time            382727 non-null  object 
 5   end_time              382727 non-null  object 
 6   modelDisplayName      382727 non-null  object 
 7   modelVersionId        382727 non-null  object 
 8   calcFeature           382727 non-null  object 
 9   subscription_name     382727 non-null  object 
 10  message_id            382727 non-null  object 
 11  publish_time          382727 non-null  object 
 12  attributes            382727 non-null  object 
 13  trenchCategory        382727 non-null  object 


In [265]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,185773,2025-03-01,2026-03-07
1,Dev_Train,196954,2024-10-01,2025-05-31


In [266]:
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,2970538,9a83fd11-e37f-48d3-86b6-a73e88a565f2,85cbc9f1-e70c-4ecb-9b38-21ad3a006c82,0.796923,2026-03-18T17:33:02.742032,2026-03-18T17:33:02.742032,Beta-Cash-Stack-Model,v1.1,"{""demo_score"": 0.623322555957233, ""apps_score""...",Cash December APPs Models Upgrade,c74b09e7-3ee2-467d-a1c4-daf271c4565b,2026-03-18T17:33:02.742032,{},Trench 1,android,Dev_Train,2024-10-25
1,3787942,577e6c42-db90-4026-95f1-cdda49949148,03c5b9de-b384-4749-b6c4-b5b38cfd49dc,0.459200,2026-03-18T17:33:02.743033,2026-03-18T17:33:02.743033,Beta-Cash-Stack-Model,v1.1,"{""demo_score"": 0.47043535820530136, ""apps_scor...",Cash December APPs Models Upgrade,36d141d9-d56b-46ff-8612-51d5a9be4360,2026-03-18T17:33:02.743033,{},Trench 1,android,Dev_Test,2025-11-03
2,3709575,c3f4aac4-8cb0-4667-9742-3fdddfd9062c,7c9633c0-bea7-4f73-8f17-2d4b01e31aa3,0.380464,2026-03-18T17:33:02.743033,2026-03-18T17:33:02.743033,Beta-Cash-Stack-Model,v1.1,"{""demo_score"": 0.46948615393285986, ""apps_scor...",Cash December APPs Models Upgrade,8797fd33-c915-4056-b95e-806dfb21dc90,2026-03-18T17:33:02.743033,{},Trench 1,android,Dev_Test,2025-10-13
3,3602314,d4a8eabf-74ac-4dcd-b07b-a412526dc42a,c58f792b-a51f-4477-8449-a7d28ab3837f,0.244064,2026-03-18T17:33:02.743033,2026-03-18T17:33:02.743033,Beta-Cash-Stack-Model,v1.1,"{""demo_score"": 0.3399234733750699, ""apps_score...",Cash December APPs Models Upgrade,1d238866-39c7-4116-b738-a1fb1c621fed,2026-03-18T17:33:02.743033,{},Trench 1,android,Dev_Test,2025-08-06
4,3816947,284e2a13-e43e-4d45-b3db-46f44c467901,9283432e-dd54-4b8e-a9f6-b66b72676d9f,0.533431,2026-03-18T17:33:02.743033,2026-03-18T17:33:02.743033,Beta-Cash-Stack-Model,v1.1,"{""demo_score"": 0.4066405101111439, ""apps_score...",Cash December APPs Models Upgrade,933eee30-e90c-4c0f-97b2-593bdc051d48,2026-03-18T17:33:02.743033,{},Trench 1,android,Dev_Test,2025-11-17


In [267]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=e49fad8e-6c9c-4915-8f92-b6d77849771f>

## Trench 2

In [ ]:
sq = """ 
WITH
  base AS (
    SELECT
      r.customer_id,
      r.digitalLoanAccountId,
      r.ln_appln_submit_datetime,
      r.trench_category,
      r.demo_score,
      r.apps_score,
      r.credo_score,
      r.stack_score,
      lower(r.ln_os_type) osType,
      date(
        IF(
          lmt.new_loan_type = 'Flex-up',
          lmt.startApplyDateTime,
          lmt.termsAndConditionsSubmitDateTime))
        application_date,
      -- CASE
      --   WHEN dev_split LIKE 'Dev_Train' THEN 'Dev_Train'
      --   WHEN dev_split LIKE 'Dev_Test' THEN 'Dev_Test'
      --   ELSE dev_split
      --   END Data_selection
      case when lower(r.ln_os_type) like '%android%' and date(ln_appln_submit_datetime) >= '2024-10-01' and date(ln_appln_submit_datetime) < '2025-06-01' then 'Dev_Train'
           when lower(r.ln_os_type) like '%android%' and date(ln_appln_submit_datetime) >= '2025-06-01' and date(ln_appln_submit_datetime) <= '2025-11-30' then 'Dev_Test'
           when lower(r.ln_os_type) like '%ios%' and date(ln_appln_submit_datetime) >= '2024-10-01' and date(ln_appln_submit_datetime) <= '2025-02-28' then 'Dev_Train'
           when lower(r.ln_os_type) like '%ios%' and date(ln_appln_submit_datetime) > '2025-02-28' and date(ln_appln_submit_datetime) <= '2025-11-30' then 'Dev_Test' 
           end Data_selection
      from 
      `worktable_data_analysis.cash_beta_trench2_applied_loans_backscored_20241001_20251130` r
    LEFT JOIN `risk_credit_mis.loan_master_table` lmt
      ON lmt.digitalLoanAccountId = r.digitalLoanAccountId
    where stack_score is not null
      )
      ,
b1 as 
(SELECT 
customer_id,
digitalLoanAccountId,
ln_appln_submit_datetime,
trench_category,
demo_score,
apps_score,
credo_score,
stack_score Beta_cash_stack_score,
osType,
application_date,
Data_selection,
FROM base
WHERE Data_selection IS NOT NULL
)
select * from b1 
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID c6ca0a4a-0ae0-4307-8edc-40db0544ba8e successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (164665, 11)


In [269]:
data.head()

,customer_id,digitalLoanAccountId,ln_appln_submit_datetime,trench_category,demo_score,apps_score,credo_score,Beta_cash_stack_score,osType,application_date,Data_selection
0,3294821,2b98a3d4-2951-4f0f-b115-52d60eba678b,2025-11-19 21:55:43+00:00,Trench 2,0.625605,0.505841,0.2723,0.577633,android,2025-11-19,Dev_Test
1,3021761,7ac89711-83a0-4062-850e-5d3d231ab938,2025-11-17 08:14:10+00:00,Trench 2,0.530444,0.494303,0.1310,0.292838,android,2025-11-17,Dev_Test
2,1797271,894c6f20-93be-469a-af28-4365afb3a482,2025-11-16 08:43:51+00:00,Trench 2,0.411650,0.469796,0.0589,0.254444,android,2025-11-16,Dev_Test
3,1541934,b45c7e57-d1f1-4c8d-ac49-e39c7a1db0ac,2025-11-19 01:52:32+00:00,Trench 2,0.614920,0.484233,0.1831,0.415921,android,2025-11-19,Dev_Test
4,3529941,f980369e-eec0-4a9f-9388-7a9243c97107,2025-11-16 01:52:41+00:00,Trench 2,0.312794,0.491421,0.2534,0.323676,android,2025-11-16,Dev_Test


In [270]:
feature_column = ['demo_score', 'apps_score', 'credo_score', 'Beta_cash_stack_score', 'stack_score']

dfd = transform_data_v1_1(data, feature_column, a='Beta_cash_stack_score', modelDisplayName='Beta-Cash-Stack-Model', tc='Trench 2', subscription_name = 'Cash December APPs Models Upgrade') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.info()

the shape of the transformed dataframe is:	 (164665, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 164665 entries, 0 to 164664
Data columns (total 17 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   customerId            164665 non-null  int64  
 1   digitalLoanAccountId  164665 non-null  object 
 2   crifApplicationId     164665 non-null  object 
 3   prediction            164665 non-null  float64
 4   start_time            164665 non-null  object 
 5   end_time              164665 non-null  object 
 6   modelDisplayName      164665 non-null  object 
 7   modelVersionId        164665 non-null  object 
 8   calcFeature           164665 non-null  object 
 9   subscription_name     164665 non-null  object 
 10  message_id            164665 non-null  object 
 11  publish_time          164665 non-null  object 
 12  attributes            164665 non-null  object 
 13  trenchCategory        164665 non-null  object 


In [271]:
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3294821,2b98a3d4-2951-4f0f-b115-52d60eba678b,a92aed64-0daf-4e8f-afc7-1fc5672da021,0.577633,2026-03-18T17:38:22.125715,2026-03-18T17:38:22.125715,Beta-Cash-Stack-Model,v1.1,"{""demo_score"": 0.6256050929353151, ""apps_score...",Cash December APPs Models Upgrade,7bd3b234-ac11-4d70-b808-24a0bb9a8b00,2026-03-18T17:38:22.125715,{},Trench 2,android,Dev_Test,2025-11-19
1,3021761,7ac89711-83a0-4062-850e-5d3d231ab938,f95cf878-2410-43f3-b7e8-4fb3b5af68e8,0.292838,2026-03-18T17:38:22.125715,2026-03-18T17:38:22.125715,Beta-Cash-Stack-Model,v1.1,"{""demo_score"": 0.5304440256692508, ""apps_score...",Cash December APPs Models Upgrade,91c55c06-bef0-43a7-b1b7-479d0df2e0db,2026-03-18T17:38:22.125715,{},Trench 2,android,Dev_Test,2025-11-17
2,1797271,894c6f20-93be-469a-af28-4365afb3a482,fb7db651-7d10-4865-b369-a74de401db15,0.254444,2026-03-18T17:38:22.125715,2026-03-18T17:38:22.125715,Beta-Cash-Stack-Model,v1.1,"{""demo_score"": 0.4116497849882912, ""apps_score...",Cash December APPs Models Upgrade,985cccb4-2f53-4415-b3a5-f59a87072ded,2026-03-18T17:38:22.125715,{},Trench 2,android,Dev_Test,2025-11-16
3,1541934,b45c7e57-d1f1-4c8d-ac49-e39c7a1db0ac,fed05cd4-bcf4-4dd3-8a81-b3fe7e6a6579,0.415921,2026-03-18T17:38:22.125715,2026-03-18T17:38:22.125715,Beta-Cash-Stack-Model,v1.1,"{""demo_score"": 0.6149202525161175, ""apps_score...",Cash December APPs Models Upgrade,4828d6f8-ec1f-4619-b1e9-f3e32adcb0d2,2026-03-18T17:38:22.125715,{},Trench 2,android,Dev_Test,2025-11-19
4,3529941,f980369e-eec0-4a9f-9388-7a9243c97107,3d1bf47b-c511-41aa-99ff-7f8d31bbe97c,0.323676,2026-03-18T17:38:22.125715,2026-03-18T17:38:22.125715,Beta-Cash-Stack-Model,v1.1,"{""demo_score"": 0.31279373300614005, ""apps_scor...",Cash December APPs Models Upgrade,cfd8c88a-f5f9-4345-a315-be573f9766a3,2026-03-18T17:38:22.125715,{},Trench 2,android,Dev_Test,2025-11-16


In [272]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,97192,2025-03-01,2025-12-03
1,Dev_Train,67473,2024-10-01,2025-05-31


In [273]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=5172f94e-ccc4-45d9-b2df-435bf2c6fa65>

# Alpha Cash Stack Model

## Trench 1

In [274]:
sq = """ 
WITH
  base AS (
    SELECT
      r.customer_id,
      r.digitalLoanAccountId,
      r.ln_appln_submit_datetime,
      r.trench_category,
      r.demo_score,
      r.apps_score,
      r.credo_score,
      r.stack_score,
      r.cic_score,
      lower(r.ln_os_type)  osType,
      date(
        IF(
          lmt.new_loan_type = 'Flex-up',
          lmt.startApplyDateTime,
          lmt.termsAndConditionsSubmitDateTime))
        application_date,
       case when lower(r.ln_os_type) like '%android%' and date(ln_appln_submit_datetime) >= '2024-10-01' and date(ln_appln_submit_datetime) < '2025-06-01' then 'Dev_Train'
           when lower(r.ln_os_type) like '%android%' and date(ln_appln_submit_datetime) >= '2025-06-01' and date(ln_appln_submit_datetime) <= '2025-11-30' then 'Dev_Test'
           when lower(r.ln_os_type) like '%ios%' and date(ln_appln_submit_datetime) >= '2024-10-01' and date(ln_appln_submit_datetime) <= '2025-02-28' then 'Dev_Train'
           when lower(r.ln_os_type) like '%ios%' and date(ln_appln_submit_datetime) > '2025-02-28' and date(ln_appln_submit_datetime) <= '2025-11-30' then 'Dev_Test' 
           end Data_selection
      -- CASE
      --   WHEN dev_split LIKE 'Dev_Train' THEN 'Dev_Train'
      --   WHEN dev_split LIKE 'Dev_Test' THEN 'Dev_Test'
      --   ELSE dev_split
      --   END Data_selection
    FROM
      `worktable_data_analysis.cash_alpha_trench1_applied_loans_backscored_20241001_20251130`
        r
    LEFT JOIN `risk_credit_mis.loan_master_table` lmt
      ON lmt.digitalLoanAccountId = r.digitalLoanAccountId
    WHERE stack_score is not null
  ),
  b1 as 
(SELECT 
customer_id,
digitalLoanAccountId,
ln_appln_submit_datetime,
trench_category,
demo_score,
apps_score,
credo_score,
stack_score,
cic_score,
osType,
application_date,
Data_selection,
FROM base
WHERE Data_selection IS NOT NULL
)
select * from b1;

"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


Job ID 15d695df-0b65-4217-9864-e3a920226adb successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (82621, 12)


In [275]:
feature_column = ['demo_score', 'apps_score', 'credo_score', 'cic_score', 'stack_score']

dfd = transform_data_v1_1(data, feature_column, a='stack_score', modelDisplayName='Alpha-Cash-Stack-Model', tc='Trench 1', subscription_name = 'Cash December APPs Models Upgrade') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.info()

the shape of the transformed dataframe is:	 (82621, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82621 entries, 0 to 82620
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   customerId            82621 non-null  int64  
 1   digitalLoanAccountId  82621 non-null  object 
 2   crifApplicationId     82621 non-null  object 
 3   prediction            82621 non-null  float64
 4   start_time            82621 non-null  object 
 5   end_time              82621 non-null  object 
 6   modelDisplayName      82621 non-null  object 
 7   modelVersionId        82621 non-null  object 
 8   calcFeature           82621 non-null  object 
 9   subscription_name     82621 non-null  object 
 10  message_id            82621 non-null  object 
 11  publish_time          82621 non-null  object 
 12  attributes            82621 non-null  object 
 13  trenchCategory        82621 non-null  object 
 14  deviceOs      

In [276]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,43339,2025-03-01,2025-11-30
1,Dev_Train,39282,2024-10-01,2025-05-31


In [277]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_application"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=52a38aa6-6f31-40ae-89cb-625070c9de1f>

## Trench 2

In [278]:
sq = """ 
WITH
  base AS (
    SELECT
      r.customer_id,
      r.digitalLoanAccountId,
      r.ln_appln_submit_datetime,
      r.trench_category,
      r.demo_score,
      r.apps_score,
      r.credo_score,
      r.stack_score,
      r.cic_score,
      lower(r.ln_os_type)  osType,
      date(
        IF(
          lmt.new_loan_type = 'Flex-up',
          lmt.startApplyDateTime,
          lmt.termsAndConditionsSubmitDateTime))
        application_date,
       case when lower(r.ln_os_type) like '%android%' and date(ln_appln_submit_datetime) >= '2024-10-01' and date(ln_appln_submit_datetime) < '2025-06-01' then 'Dev_Train'
           when lower(r.ln_os_type) like '%android%' and date(ln_appln_submit_datetime) >= '2025-06-01' and date(ln_appln_submit_datetime) <= '2025-11-30' then 'Dev_Test'
           when lower(r.ln_os_type) like '%ios%' and date(ln_appln_submit_datetime) >= '2024-10-01' and date(ln_appln_submit_datetime) <= '2025-02-28' then 'Dev_Train'
           when lower(r.ln_os_type) like '%ios%' and date(ln_appln_submit_datetime) > '2025-02-28' and date(ln_appln_submit_datetime) <= '2025-11-30' then 'Dev_Test' 
           end Data_selection
      -- CASE
      --   WHEN dev_split LIKE 'Dev_Train' THEN 'Dev_Train'
      --   WHEN dev_split LIKE 'Dev_Test' THEN 'Dev_Test'
      --   ELSE dev_split
      --   END Data_selection
    FROM
      `worktable_data_analysis.cash_alpha_trench2_applied_loans_backscored_20241001_20251130`
        r
    LEFT JOIN `risk_credit_mis.loan_master_table` lmt
      ON lmt.digitalLoanAccountId = r.digitalLoanAccountId
    WHERE stack_score is not null
  ),
  b1 as 
(SELECT 
customer_id,
digitalLoanAccountId,
ln_appln_submit_datetime,
trench_category,
demo_score,
apps_score,
credo_score,
stack_score,
cic_score,
osType,
application_date,
Data_selection,
FROM base
WHERE Data_selection IS NOT NULL
)
select * from b1;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 8089be6a-4fb7-4b5d-95d0-cc223d0b44f0 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (57433, 12)


In [279]:
feature_column = ['demo_score', 'apps_score', 'credo_score', 'cic_score', 'stack_score']

dfd = transform_data_v1_1(data, feature_column, a='stack_score', modelDisplayName='Alpha-Cash-Stack-Model', tc='Trench 2', subscription_name = 'Cash December APPs Models Upgrade') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.info()

the shape of the transformed dataframe is:	 (57433, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57433 entries, 0 to 57432
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   customerId            57433 non-null  int64  
 1   digitalLoanAccountId  57433 non-null  object 
 2   crifApplicationId     57433 non-null  object 
 3   prediction            57433 non-null  float64
 4   start_time            57433 non-null  object 
 5   end_time              57433 non-null  object 
 6   modelDisplayName      57433 non-null  object 
 7   modelVersionId        57433 non-null  object 
 8   calcFeature           57433 non-null  object 
 9   subscription_name     57433 non-null  object 
 10  message_id            57433 non-null  object 
 11  publish_time          57433 non-null  object 
 12  attributes            57433 non-null  object 
 13  trenchCategory        57433 non-null  object 
 14  deviceOs      

In [280]:
dfd.sample(5)

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
41696,2787297,bfa8e0c0-e245-4bd9-a581-eab2252416c2,6b6dd298-137e-435e-9269-e3f3b249bb4d,0.540426,2026-03-18T17:41:13.522405,2026-03-18T17:41:13.522405,Alpha-Cash-Stack-Model,v1.1,"{""demo_score"": 0.5621140713407864, ""credo_scor...",Cash December APPs Models Upgrade,b4cba127-c1ef-4c76-9c66-d5964a12b96e,2026-03-18T17:41:13.522405,{},Trench 2,ios,Dev_Test,2025-04-29
45369,3610124,02d046b1-bdd7-44b8-b537-ddc8940e5fe5,c569afcb-2125-4baa-9cca-1f5e1ce446df,0.252074,2026-03-18T17:41:13.905349,2026-03-18T17:41:13.905349,Alpha-Cash-Stack-Model,v1.1,"{""demo_score"": 0.356470648299489, ""credo_score...",Cash December APPs Models Upgrade,ecc38e00-33a2-42eb-96df-523b6aabc62c,2026-03-18T17:41:13.905349,{},Trench 2,ios,Dev_Test,2025-10-10
31583,2980761,1c03310a-6e98-4fb4-b4e4-7ec3b101e293,472b7d47-f3d0-44ba-a604-92e417fe527b,0.857218,2026-03-18T17:41:12.444791,2026-03-18T17:41:12.444791,Alpha-Cash-Stack-Model,v1.1,"{""demo_score"": 0.6215561487308605, ""apps_score...",Cash December APPs Models Upgrade,3ac67857-0259-4acb-94be-f9ffe1a7a0a0,2026-03-18T17:41:12.444791,{},Trench 2,android,Dev_Test,2025-11-03
32248,3446975,990fec12-7e70-4eab-bce7-f6e7e057681b,5a06b047-4106-4cb8-b299-7f235cb45502,0.288650,2026-03-18T17:41:12.521210,2026-03-18T17:41:12.521210,Alpha-Cash-Stack-Model,v1.1,"{""demo_score"": 0.38728158595692636, ""apps_scor...",Cash December APPs Models Upgrade,03ff3cf6-3c0f-4612-b6f1-4f3f9ea54754,2026-03-18T17:41:12.521210,{},Trench 2,android,Dev_Test,2025-08-20
7764,2185175,7ede8e11-66d7-41df-a778-54a4ee2d22af,7184c35d-6a9a-492d-8650-25740a597b1d,0.332492,2026-03-18T17:41:09.870092,2026-03-18T17:41:09.870092,Alpha-Cash-Stack-Model,v1.1,"{""demo_score"": 0.4476147748121498, ""apps_score...",Cash December APPs Models Upgrade,c3b1899f-f88a-4dd8-9a44-b65e4b61dcb1,2026-03-18T17:41:09.870092,{},Trench 2,android,Dev_Test,2025-08-02


In [281]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,34175,2025-03-01,2025-11-30
1,Dev_Train,23258,2024-10-01,2025-05-31


In [282]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_application"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=b3c08d1e-eb38-4671-83ab-cd0897e0821e>

# End